# Trabajo Práctico Especial — Fundamentos de la Ciencia de Datos
## CKD-NHANES 2021-2023: enfermedad renal crónica por estadios

**Año:** 2026 · **Grupo:** 39

**Integrantes:** Simón Campos Madarieta · Axel Ferreyra · Joaquín Fromm

La notebook sigue la misma organización que el informe: **1.** Introducción · **2.** Análisis exploratorio y limpieza de los datos · **3.** Hipótesis y resolución · **4.** Conclusiones · Referencias.

Para ejecutarla, el archivo `CKD_NHANES_2021_2023.csv` tiene que estar en la misma carpeta que la notebook (instrucciones en `README.md`).

---
## 1. Introducción

### 1.1 Origen de los datos

Los datos provienen del **National Health and Nutrition Examination Survey (NHANES)**, ciclo agosto 2021 – agosto 2023 (ciclo "L"), una encuesta de salud y nutrición de los Estados Unidos conducida por el National Center for Health Statistics (NCHS) de los CDC [1]. La encuesta combina una **entrevista** en el hogar con un **examen físico y de laboratorio** en un centro móvil.

El archivo `CKD_NHANES_2021_2023.csv` es un **conjunto derivado**: une variables de varios archivos de NHANES (demografía, medidas corporales, presión arterial, bioquímica en sangre, orina y cuestionarios de diabetes y tabaquismo) y agrega tres variables calculadas de función renal (`egfr`, `ckd_stage` y `ckd_present`). Su nombre coincide con un dataset publicado en Kaggle (*CKD-NHANES 2021-2023: Staged Kidney Disease*) [2], que declara 36 variables contra las 29 de este archivo; no se pudo verificar si son la misma versión.

Evidencia del origen, verificada en la sección 2 contra la documentación oficial de NHANES (archivo de demografía `DEMO_L`) [3]:

| Dato | Documentación de NHANES | Este archivo |
|---|---|---|
| Participantes del ciclo | 11.933 | 11.933 filas (sección 2.2) |
| Edad codificada como 80 ("80 años o más") | 525 | 525 (sección 2.6.1) |
| Índice de pobreza codificado como 5 ("5 o más") | 2.149 | 2.149 (sección 2.6.4) |
| Índice de pobreza faltante | 2.041 | 2.041 (sección 2.6.4) |
| `egfr` calculada con CKD-EPI 2021 sin raza [4] | declarado en la metadata [5] | se reproduce exactamente (sección 2.11.1) |

Particularidades del diseño de NHANES que condicionan todo el análisis:

- **Es una muestra compleja con pesos.** NHANES no es una muestra aleatoria simple: los archivos originales traen pesos muestrales para estimar valores de la población de EE.UU. **Este archivo no los incluye**, así que todo lo que se calcule acá describe a la **muestra**, no a la población.
- **Sobremuestreo por edad.** En este ciclo se dejó de sobremuestrear por raza/etnia e ingreso y se empezó a sobremuestrear por grupo de edad.
- **Baja tasa de respuesta.** Respondió la entrevista el 34,5 % de las personas seleccionadas y completó el examen el 25,6 % [6].
- **Hay personas entrevistadas pero no examinadas.** Según la documentación [3], 3.073 participantes solo respondieron la entrevista: no tienen medidas corporales ni análisis.

### 1.2 Contexto clínico

La **enfermedad renal crónica** (CKD, *chronic kidney disease*) se define por alteraciones de la estructura o de la función de los riñones presentes durante **al menos tres meses**. Se evalúa con dos ejes:

- **Filtrado (eGFR).** No se mide directamente: se estima a partir de la creatinina en sangre, la edad y el sexo con la ecuación CKD-EPI 2021 [4], y se expresa en mL/min/1,73 m². Un eGFR menor indica menor capacidad de filtrado.
- **Daño (albuminuria).** Se mide con la relación albúmina/creatinina en orina (ACR), en mg/g. Un ACR de 30 mg/g o más indica albuminuria aumentada, es decir, pérdida anormal de proteína por la orina.

Regla que usa el archivo para asignar `ckd_stage`, según `metadata.docx` [5]:

| `ckd_stage` | Regla del archivo |
|---|---|
| No CKD | eGFR ≥ 90 y ACR < 30 |
| Stage 1 | eGFR ≥ 90 y ACR ≥ 30 |
| Stage 2 | 60 ≤ eGFR < 90 |
| Stage 3a, 3b, 4 y 5 | eGFR < 60 (cortes en 45, 30 y 15) |
| Unknown | eGFR ausente |

La regla asigna *Stage 2* solo por el filtrado, sin exigir un marcador de daño. La guía clínica KDIGO [7] considera enfermedad renal crónica a un eGFR de 60 a 89 solo si además hay daño renal (por ejemplo, ACR ≥ 30). Por esa diferencia, los análisis de enfermedad renal de este trabajo usan un criterio clínico propio, construido en la sección 2.15.5.

Dos limitaciones vienen del origen de los datos: hay **una sola medición por persona**, así que no se puede confirmar la persistencia de tres meses, y **una asociación observada no demuestra causalidad** (la propia metadata lo advierte).

### 1.3 Organización de la notebook

- **Sección 2 — Análisis exploratorio y limpieza.** Las secciones 2.1 a 2.14 describen el dataset **en crudo**, sin modificar ningún dato: la estructura y los tipos de cada variable (2.2 y 2.3), los faltantes y duplicados (2.4 y 2.5), cada bloque de variables (2.6 a 2.10), las variables derivadas y la clasificación renal (2.11), la forma y los valores atípicos (2.12) y las relaciones entre variables (2.13). Cada problema detectado se marca con un código **[P#]**, y todos se resumen en la sección 2.14. La sección 2.15 aplica la limpieza sobre una copia de trabajo y define las poblaciones de análisis.
- **Sección 3 — Hipótesis y resolución.** Las tres hipótesis propias (3.1 a 3.3) y las tres planteadas en el enunciado (3.4 a 3.6).
- **Sección 4 — Conclusiones.** Tabla resumen de las seis hipótesis y limitaciones generales del trabajo.

---
## 2. Análisis exploratorio y limpieza de los datos

Las secciones 2.1 a 2.14 trabajan sobre `raw_dataset`, el archivo tal como viene. Los cálculos auxiliares se guardan en variables aparte: **ninguna de esas celdas modifica el dataset crudo**. Las correcciones se aplican en la sección 2.15, sobre una copia.

Para cada variable se distinguen tres capas: el tipo de dato que le asigna pandas (`dtype`), lo que dice la metadata y su tipo estadístico real. Cuando las tres no coinciden, eso es un hallazgo.

### 2.1 Carga de los datos

In [ ]:
# importamos las librerías que vamos a usar en toda la notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from os import path, listdir

In [ ]:
# el archivo de datos está en la misma carpeta que la notebook
archivo_datos = "CKD_NHANES_2021_2023.csv"

# verificamos que el archivo exista antes de abrirlo
if path.exists(archivo_datos):
  print(f"El archivo {archivo_datos} está disponible")
else:
  print(f"No se encontró {archivo_datos}. Archivos en la carpeta: {listdir()}")

In [ ]:
# abrimos el archivo usando una función específica de pandas
raw_dataset = pd.read_csv(archivo_datos)
raw_dataset.head()

### 2.2 Estructura general

In [ ]:
# tamaño del dataset
print(f"Filas: {raw_dataset.shape[0]}")
print(f"Columnas: {raw_dataset.shape[1]}")

In [ ]:
# tipos de dato y cantidad de valores no nulos de cada columna
raw_dataset.info()

In [ ]:
# participant_id y age están completas: verificamos si el float64 se debe a faltantes o a decimales
print(f"Nulos de participant_id: {raw_dataset['participant_id'].isna().sum()}")
print(f"participant_id con decimales: {(raw_dataset['participant_id'] % 1 != 0).sum()}")
print(f"Nulos de age: {raw_dataset['age'].isna().sum()}")

**Observaciones**

- **11.933 filas × 29 columnas.** Coincide con los 11.933 participantes del ciclo según la documentación de NHANES. **Cada fila es una persona** (la unidad de observación).
- pandas asigna `float64` a 25 columnas, `int64` a `ckd_present` y `object` (texto) a `gender`, `ethnicity` y `ckd_stage`.
- **[P13] El `float64` no siempre viene de los faltantes.** En una columna de enteros con algún `NaN`, pandas usa `float64` porque `int64` no admite faltantes: es lo que pasa con los códigos de cuestionario (`education_level`, `diabetes_diagnosed`, etc.). Pero `participant_id` está completa y no tiene decimales, y aun así es `float64`. El motivo es el formato del archivo: en el CSV todos los números están escritos con `.0` (por ejemplo, `130378.0`), probablemente por la conversión desde el formato original de NHANES. La única columna escrita sin decimales es `ckd_present`, y por eso es la única `int64`.
- La cantidad de valores no nulos varía muchísimo entre columnas, de 1.081 en `insulin_use` a 11.933. Se estudia en la sección 2.4.

**Características generales del conjunto**

| Aspecto | Qué se encontró | Dónde se analiza |
|---|---|---|
| **Tipos** | El `dtype` no coincide con el tipo estadístico en 8 columnas (el identificador, seis códigos de cuestionario y `ckd_stage`) | 2.3 |
| **Granularidad** | Una fila por persona y una sola medición por persona. Algunas columnas son derivadas de otras (`bmi`, `albumin_creatinine_ratio`, `egfr`, `ckd_stage`, `ckd_present`) y la presión es un único valor de tres lecturas | 2.7, 2.11 |
| **Alcance** | Una muestra compleja de la población de EE.UU., de todas las edades, sin los pesos muestrales: describe a la muestra, no a la población. Cubre más de lo que necesita un análisis renal de adultos (bebés y chicos sin análisis de sangre), por lo que los análisis se restringen a poblaciones definidas | 1.1, 2.4.3, 2.11.4, 2.15.7 |
| **Temporalidad** | Estudio transversal: agosto 2021 a agosto 2023, una foto por persona. No hay seguimiento, así que no se puede confirmar que una alteración renal dure tres meses ni ordenar causas y efectos en el tiempo | 1.2 |
| **Confiabilidad** | Fuente secundaria (un archivo derivado de NHANES). Hay ceros disfrazados, códigos de "no sabe / se negó", valores censurados en los extremos, respuestas autodeclaradas e incoherentes, y una etiqueta (`ckd_present`) que no significa lo que su nombre dice | 2.4, 2.6, 2.9, 2.10, 2.11 |

### 2.3 Tipos de variables: `dtype`, metadata y tipo estadístico

In [ ]:
# para cada columna mostramos el tipo que asignó pandas y la cantidad de valores distintos
for columna in raw_dataset.columns:
  print(f"{columna}: {raw_dataset[columna].dtype} - {raw_dataset[columna].nunique()} valores distintos")

**Las tres capas de cada variable**

| Variable | `dtype` | Qué dice la metadata | Tipo estadístico real | Observación |
|---|---|---|---|---|
| `participant_id` | float64 | código único; no usarlo como predictor | identificador (sin magnitud ni orden) | no es una variable de análisis; está completa y sin decimales **[P13]** |
| `age` | float64 | edad; 80 agrupa a 80 o más; el 0 puede aparecer como un número positivo minúsculo | cuantitativa continua de razón, registrada en años cumplidos (enteros) y censurada en 80 | 81 valores distintos: 0 a 80 **[P7] [P8]** |
| `gender` | object | Male / Female | cualitativa binaria (nominal de dos categorías) | coincide |
| `ethnicity` | object | seis grupos étnicos | cualitativa nominal | coincide |
| `education_level` | float64 | códigos 1 a 5; 7 y 9 = sin respuesta | cualitativa ordinal (1 a 5) | 6 valores: 1 a 5 y el 9 **[P12] [P13]** |
| `poverty_income_ratio` | float64 | ingreso familiar / umbral de pobreza; "cero indica ingreso muy bajo", 5 es el tope agrupado | cuantitativa continua de razón, censurada en 5 | **[P9] [P10] [P11]** |
| `bmi` | float64 | kg/m², calculado con peso y altura | cuantitativa continua (derivada) | se verifica en 2.11.1 |
| `weight_kg` | float64 | peso en kg | cuantitativa continua | — |
| `height_cm` | float64 | altura **o longitud** en cm | cuantitativa continua | no hay datos de menores de 2 años **[P19]** |
| `bp_systolic`, `bp_diastolic` | float64 | presión en mmHg | cuantitativas continuas, registradas como enteros | una sola lectura de las tres que toma NHANES **[P20]** |
| `serum_creatinine`, `blood_urea_nitrogen`, `albumin_serum`, `phosphorus`, `bicarbonate`, `calcium`, `uric_acid` | float64 | análisis de sangre, con sus unidades | cuantitativas continuas | la urea y el bicarbonato solo tienen valores enteros (resolución del registro) |
| `urine_creatinine`, `urine_albumin` | float64 | análisis de orina | cuantitativas continuas | `urine_albumin` censurada por izquierda **[P26]** |
| `albumin_creatinine_ratio` | float64 | ACR en mg/g, calculado | cuantitativa continua (derivada) | se verifica en 2.11.1 |
| `diabetes_diagnosed` | float64 | 1 sí, 2 no, 3 borderline; 7/9 sin respuesta | cualitativa ordinal (no < borderline < sí) cuyo código **no respeta** el orden | 4 valores: 1, 2, 3 y 9 **[P12] [P13] [P14]** |
| `insulin_use` | float64 | 1 sí, 2 no; se pregunta a un subconjunto | cualitativa binaria | faltantes por diseño **[P15]** |
| `diabetes_pills` | float64 | 1 sí, 2 no; según respuestas previas | cualitativa binaria | 3 valores: 1, 2 y 9 **[P12] [P16] [P17]** |
| `ever_smoked` | float64 | 1 sí, 2 no (100 cigarrillos en la vida) | cualitativa binaria | 4 valores: 1, 2, 7 y 9 **[P12] [P15]** |
| `current_smoker` | float64 | 1 todos los días, 2 algunos días, 3 no fuma | cualitativa ordinal (frecuencia), con el código de mayor a menor frecuencia | faltantes por diseño **[P15]** |
| `egfr` | float64 | mL/min/1,73 m², calculada | cuantitativa continua (derivada) | **[P4] [P5]** |
| `ckd_stage` | object | categoría renal; incluye Unknown | cualitativa ordinal, más una categoría que representa un faltante | **[P2] [P3] [P6]** |
| `ckd_present` | int64 | 0 = No CKD; 1 = el resto, incluido Unknown | cualitativa binaria (derivada) | **[P1]** |

**Consecuencias**

- **El `dtype` no alcanza para clasificar.** De las 25 columnas `float64`, 7 no son cantidades: el identificador y los seis códigos de cuestionario. A la inversa, `ckd_stage`, que pandas lee como texto, es ordinal. La metadata lo indica: *"las columnas numéricas que representan categorías deben tratarse como variables categóricas"*. Esas variables se describen con frecuencias y proporciones, no con medias.
- **La edad, la presión, la urea y el bicarbonato** se registran en valores enteros, pero miden magnitudes continuas: se tratan como continuas.
- **Todas las cuantitativas tienen un cero verdadero** (escala de razón), por lo que el coeficiente de variación es aplicable a todas.

Para las cuantitativas, la medida de centro y de dispersión que se reporta se elige según la forma de su distribución (sección 2.12.1).

### 2.4 Valores faltantes

Se distinguen tres situaciones: **nulos explícitos** (`NaN`), **nulos disfrazados** (valores que representan una ausencia pero no son `NaN`) y **faltantes por diseño** (la pregunta o el examen no correspondía a esa persona).

#### 2.4.1 Nulos explícitos

In [ ]:
# contamos los nulos por columna y los expresamos como porcentaje del total de filas
nulos = raw_dataset.isna().sum()
porcentaje_nulos = (nulos / raw_dataset.shape[0] * 100).round(1)
tabla_nulos = pd.DataFrame({"nulos": nulos, "porcentaje": porcentaje_nulos})
tabla_nulos.sort_values("nulos", ascending=False)

**Observaciones**

- Solo 6 columnas están completas: `participant_id`, `age`, `gender`, `ethnicity`, `ckd_stage` y `ckd_present`.
- Los faltantes forman **bloques con conteos casi iguales**, uno por cada componente de la encuesta: sangre (alrededor de 5.600, el 47 %), orina (alrededor de 3.780), presión (4.416, las dos columnas juntas) y medidas corporales (entre 3.179 y 3.462).
- `serum_creatinine`, `blood_urea_nitrogen` y `egfr` tienen exactamente 5.607 nulos, el mismo número que la categoría `Unknown` de `ckd_stage`. Es lo esperable: sin creatinina no se puede calcular el eGFR (se verifica en 2.8).
- Estos porcentajes **no** se pueden leer como "datos perdidos": gran parte responde al diseño de la encuesta (sección 2.4.3).

#### 2.4.2 Nulos disfrazados

**Códigos 7 y 9.** Según la metadata, cuando existen, significan *"se negó a responder"* y *"no sabe"*.

In [ ]:
# los códigos 7 y 9 son faltantes guardados como números: los contamos en cada columna de códigos
columnas_codigos = ["education_level", "diabetes_diagnosed", "insulin_use", "diabetes_pills", "ever_smoked", "current_smoker"]
for columna in columnas_codigos:
  cantidad_7 = (raw_dataset[columna] == 7).sum()
  cantidad_9 = (raw_dataset[columna] == 9).sum()
  print(f"{columna}: código 7 = {cantidad_7} - código 9 = {cantidad_9}")

**Ceros disfrazados.** La metadata advierte que el cero de la edad *"puede aparecer como un número positivo extremadamente pequeño por la conversión del archivo"*. Revisamos todas las columnas numéricas por si el mismo problema aparece en otras.

In [ ]:
# buscamos valores positivos pero extremadamente pequeños en todas las columnas numéricas
for columna in raw_dataset.select_dtypes(include="number").columns:
  cantidad = ((raw_dataset[columna] > 0) & (raw_dataset[columna] < 0.001)).sum()
  if cantidad > 0:
    print(f"{columna}: {cantidad} valores minúsculos")

# mostramos el valor exacto que aparece en lugar del cero
print(f"Valor mínimo de age: {raw_dataset['age'].min()}")
print(f"Valor mínimo de poverty_income_ratio: {raw_dataset['poverty_income_ratio'].min()}")

In [ ]:
# el menor valor real del índice de pobreza, sin contar el valor minúsculo
pobreza = raw_dataset["poverty_income_ratio"]
print(f"Menor valor de poverty_income_ratio mayor a 0,001: {pobreza[pobreza > 0.001].min()}")

**Observaciones**

- **Códigos 7/9 [P12].** Aparecen en `education_level` (11 con código 9), `diabetes_diagnosed` (4 con 9), `diabetes_pills` (7 con 9) y `ever_smoked` (7 con código 7 y 7 con 9). `insulin_use` y `current_smoker` no los tienen. Son pocos casos, pero si se dejan como números, `describe()` los promedia como si fueran cantidades: el máximo de 9 en `education_level` no es un "nivel 9". Se convierten en faltantes en la sección 2.15.3.
- **Ceros disfrazados [P7] [P9].** El valor `5.397605346934028e-79` aparece en 189 edades y en 100 índices de pobreza, y en ninguna otra columna. Corresponde a la forma en que quedan los ceros al convertir los archivos originales de NHANES (formato SAS, `.XPT`). En la edad son **bebés de menos de un año**, es decir, edad 0: no son faltantes. En el índice de pobreza la metadata dice que *"cero indica ingreso muy bajo"*, pero solo advierte el número minúsculo para la edad; el mecanismo es el mismo, así que estos 100 valores son el 0 del índice. La evidencia: el menor valor real del índice es 0,01, así que `5.4e-79` no puede ser un valor medido.
- Son **errores de representación que no son atípicos**: quedan dentro del rango de cada variable y ningún criterio de atípicos los detecta (sección 2.12.2); se identifican con la metadata y el dominio. Se corrigen en la sección 2.15.2.
- **La categoría `Unknown` de `ckd_stage`** también es un faltante disfrazado: es un texto que significa "no hay eGFR". Además, `ckd_present` la codifica como 1, igual que a los enfermos **[P1] [P6]**. Se analiza en la sección 2.11 y se trata en la 2.15.5.

#### 2.4.3 Faltantes por diseño

NHANES no hace todas las preguntas ni todos los exámenes a todas las edades. Buscamos desde qué edad aparece cada dato.

In [ ]:
# para cada variable buscamos la edad mínima entre quienes tienen el dato
columnas_por_edad = ["weight_kg", "height_cm", "bmi", "urine_albumin", "bp_systolic", "serum_creatinine", "ever_smoked", "education_level"]
for columna in columnas_por_edad:
  edades_con_dato = raw_dataset[raw_dataset[columna].notna()]["age"]
  print(f"{columna}: edad mínima con dato = {edades_con_dato.min():.0f} años")

In [ ]:
# contamos las personas que no tienen ninguna medición del examen físico ni del laboratorio
columnas_examen = ["weight_kg", "height_cm", "bmi", "bp_systolic", "bp_diastolic",
                   "serum_creatinine", "blood_urea_nitrogen", "albumin_serum", "phosphorus",
                   "bicarbonate", "calcium", "uric_acid", "urine_creatinine", "urine_albumin",
                   "albumin_creatinine_ratio"]
sin_examen = raw_dataset[raw_dataset[columnas_examen].isna().all(axis=1)]
print(f"Personas sin ningún dato de examen ni de laboratorio: {sin_examen.shape[0]}")

**Observaciones**

| Bloque de datos | Hay datos desde | Lectura |
|---|---|---|
| Peso | 0 años | se pesa a todas las edades |
| Altura y BMI | 2 años | la metadata dice "altura **o longitud**", pero no hay ningún dato de menores de 2 años (a ellos NHANES les mide la longitud acostados, en otra variable que no está en el archivo) **[P19]** |
| Orina | 3 años | — |
| Presión arterial | 8 años | — |
| Sangre (y por lo tanto eGFR) | 12 años | a los menores de 12 no se les extrae sangre |
| Tabaquismo | 18 años | pregunta solo para adultos |
| Nivel educativo | 20 años | pregunta solo para mayores de 20 |

- **[P15] Faltantes por diseño.** En estas variables, un `NaN` significa "no correspondía preguntar o medir", no "no sabemos". En las preguntas encadenadas del cuestionario la pregunta **no aplica** (por ejemplo, el uso de insulina en quien no tiene diabetes): no hay un valor real que falte, por lo que no se imputa. En los exámenes que dependen de la edad (a los menores de 12 no se les extrae sangre) el valor existe pero no se midió, y la falta depende de una variable observada, la edad. Las preguntas encadenadas se analizan en la sección 2.10.
- **[P18] La metadata dice que *"cada fila corresponde a una persona examinada"*, pero 3.089 filas no tienen ningún dato de examen ni de laboratorio.** La documentación de NHANES [3] informa 3.073 personas solo entrevistadas; la diferencia de 16 serían personas que fueron al examen pero no tienen ninguna de estas mediciones.

### 2.5 Duplicados

In [ ]:
# filas completamente duplicadas
print(f"Filas duplicadas: {raw_dataset.duplicated().sum()}")

# identificadores repetidos
print(f"participant_id repetidos: {raw_dataset['participant_id'].duplicated().sum()}")

# verificamos si los identificadores son consecutivos
id_minimo = raw_dataset["participant_id"].min()
id_maximo = raw_dataset["participant_id"].max()
print(f"Identificadores de {id_minimo:.0f} a {id_maximo:.0f}: {id_maximo - id_minimo + 1:.0f} valores posibles para {raw_dataset.shape[0]} filas")

In [ ]:
# buscamos filas iguales en todo salvo el identificador
sin_identificador = raw_dataset.drop("participant_id", axis=1)
print(f"Filas iguales a otra sin contar el identificador: {sin_identificador.duplicated().sum()}")

# miramos todas las filas involucradas y cuántas combinaciones distintas de valores forman
repetidas_sin_id = raw_dataset[sin_identificador.duplicated(keep=False)]
print(f"Filas involucradas: {repetidas_sin_id.shape[0]}")
print(f"Combinaciones distintas que se repiten: {repetidas_sin_id.drop('participant_id', axis=1).drop_duplicates().shape[0]}")

# la edad de las filas involucradas
repetidas_sin_id["age"].describe()

**Observaciones**

- **No hay duplicados reales.** Los 11.933 identificadores son únicos y consecutivos, del 130378 al 142310.
- **[P24] Sin el identificador aparecen filas repetidas.** Hay que separar tres números que miden cosas distintas: **196 combinaciones distintas** de valores que se repiten, en **560 filas involucradas**, que dan **364 copias sobrantes** (560 − 196 = 364, lo que cuenta `duplicated()`). Son sobre todo chicos (mediana de 10 años) sin examen: comparten edad, sexo y etnia, y no tienen ningún otro dato, así que coincidir en tan pocas columnas es esperable. **No son la misma persona.** Si en la limpieza se buscaran duplicados sin mirar el identificador, se borrarían personas reales. Por eso `participant_id` se conserva como identificador, aunque no se use como variable de análisis.

### 2.6 Variables demográficas y socioeconómicas

#### 2.6.1 Edad (`age`)

In [ ]:
# medidas resumen de la edad
raw_dataset["age"].describe()

In [ ]:
# contamos las edades con el valor minúsculo (edad 0) y las edades en el tope de 80
print(f"Edades menores a 1: {(raw_dataset['age'] < 1).sum()}")
print(f"Edades iguales a 80 (80 años o más): {(raw_dataset['age'] == 80).sum()}")

In [ ]:
# graficamos la distribución de la edad con un intervalo por año
plt.figure(figsize=(10, 5))
plt.hist(raw_dataset["age"], bins=range(0, 82), edgecolor="black")
plt.xlabel("Edad (años)")
plt.ylabel("Cantidad de personas")
plt.title("Distribución de la edad")
plt.show()

**Observaciones**

- Media 38,3 años y mediana 37, con un rango de 0 a 80 que cubre toda la vida y un desvío de 25,6 años. El primer cuartil es 13: **un cuarto de la muestra tiene 13 años o menos**. Esto importa porque muchos análisis de función renal solo valen para adultos.
- La media y la mediana casi coinciden, pero la distribución no tiene forma de campana: el histograma es casi plano, con mesetas (último punto). Por eso se informan también los cuartiles.
- **[P7]** El mínimo que muestra `describe()` es `5.397605e-79`: son las 189 edades 0 de la sección 2.4.2. Ese valor hace que `describe()` muestre todas las medidas de la columna en notación científica (`3.831786e+01` es 38,32).
- **[P8]** La barra de 80 años es mucho más alta que sus vecinas (525 personas) porque agrupa a **todas** las personas de 80 años o más. No es un pico real: es un dato **censurado**, en el que solo sabemos que la edad es "80 o más". Según la documentación de NHANES [3], la edad promedio de ese grupo es 85.
- La forma de la distribución está condicionada por el diseño, que sobremuestrea por grupos de edad: no refleja la pirámide de edades de EE.UU. Se ven tres mesetas: de 0 a 19 años hay unas 200 personas por año; a los 20 la cantidad cae de golpe (de 160 a 71) y entre los 20 y los 59 promedia unas 110; desde fines de los 50 vuelve a subir y entre los 60 y los 79 promedia unas 150.

#### 2.6.2 Sexo (`gender`) y grupo étnico (`ethnicity`)

In [ ]:
# frecuencias absolutas y porcentuales de sexo y de grupo étnico
for columna in ["gender", "ethnicity"]:
  print(f"{columna}:")
  for categoria, cantidad in raw_dataset[columna].value_counts().items():
    print(f"  {categoria}: {cantidad} ({cantidad / raw_dataset.shape[0] * 100:.1f} %)")

In [ ]:
# graficamos la cantidad de personas por grupo étnico, de mayor a menor
conteo_etnia = raw_dataset["ethnicity"].value_counts()
plt.figure(figsize=(9, 5))
plt.bar(conteo_etnia.index, conteo_etnia.values, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cantidad de personas")
plt.title("Personas por grupo étnico")
plt.show()

**Observaciones**

- Las dos variables están completas y sus categorías son consistentes: no hay variantes de escritura ni valores fuera de la lista de la metadata.
- Hay 6.358 mujeres (53,3 %) y 5.575 hombres (46,7 %).
- Los blancos no hispanos son **más de la mitad de la muestra (52,1 %)**. Le siguen negros no hispanos (13,4 %), otros hispanos (11,5 %), mexicano-americanos (9,4 %), otro/multirracial (7,9 %) y asiáticos no hispanos (5,7 %).
- **[P25]** Sin pesos muestrales, estas proporciones describen a la muestra y no a la población de EE.UU.
- Para la hipótesis (a) (sección 3.4): como la mitad de la muestra es blanca no hispana, cualquier conteo absoluto de casos va a estar dominado por ese grupo **solo por su tamaño**. Comparar conteos no es lo mismo que comparar proporciones dentro de cada grupo.

#### 2.6.3 Nivel educativo (`education_level`)

In [ ]:
# frecuencias de cada código, incluidos los nulos
raw_dataset["education_level"].value_counts(dropna=False).sort_index()

In [ ]:
# los nulos, ¿corresponden a menores de 20 años, a quienes no se les hace la pregunta?
nulos_educacion = raw_dataset[raw_dataset["education_level"].isna()]
print(f"Nulos de education_level: {nulos_educacion.shape[0]}")
print(f"  menores de 20 años: {(nulos_educacion['age'] < 20).sum()}")
print(f"  de 20 años o más: {(nulos_educacion['age'] >= 20).sum()}")

In [ ]:
# moda y mediana de la variable ordinal, calculadas solo sobre los códigos válidos (1 a 5)
educacion_valida = raw_dataset[raw_dataset["education_level"] <= 5]["education_level"]
print(f"Personas con un código válido: {educacion_valida.shape[0]}")
print(f"Moda: {educacion_valida.value_counts().index[0]:.0f}")
print(f"Mediana: {educacion_valida.describe()['50%']:.0f}")

In [ ]:
# graficamos los niveles educativos con su significado según la metadata (sin el código 9 ni los nulos)
etiquetas_educacion = {1: "Menos de 9no grado", 2: "9no a 11vo grado", 3: "Secundario completo",
                       4: "Superior incompleto o terciario", 5: "Universitario o más"}
cantidades = [(raw_dataset["education_level"] == codigo).sum() for codigo in etiquetas_educacion]

plt.figure(figsize=(9, 5))
plt.bar(list(etiquetas_educacion.values()), cantidades, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cantidad de personas")
plt.title("Máximo nivel educativo alcanzado (personas de 20 años o más)")
plt.show()

**Observaciones**

- Es una variable **ordinal**: los códigos 1 a 5 tienen un orden (de menos a más educación), pero la distancia entre niveles no es medible. Por eso la media de 3,80 que daría `describe()` no tiene interpretación, y además está contaminada por el código 9.
- **[P12]** 11 personas tienen el código 9 ("no sabe").
- De los 4.139 nulos, 4.124 son menores de 20 años, a quienes no se les hace la pregunta (faltante por diseño, **[P15]**). Solo 15 adultos de 20 años o más no tienen el dato.
- Sobre las 7.783 personas con un código válido, la **moda es 5** (universitario o más, 2.625 personas) y la **mediana es 4** (superior incompleto o terciario): por lo menos la mitad de quienes respondieron llegó a ese nivel o más. La distribución se concentra en los niveles altos.

#### 2.6.4 Índice de pobreza (`poverty_income_ratio`)

Es el ingreso familiar dividido por el umbral de pobreza que corresponde a ese hogar: 1 significa un ingreso igual al umbral de pobreza, y 2 el doble.

In [ ]:
# medidas resumen del índice de pobreza
raw_dataset["poverty_income_ratio"].describe()

In [ ]:
# valores especiales: ceros disfrazados, tope de 5 y nulos
pobreza = raw_dataset["poverty_income_ratio"]
print(f"Valores minúsculos (cero): {((pobreza > 0) & (pobreza < 0.001)).sum()}")
print(f"Valores iguales a 5 (5 o más): {(pobreza == 5).sum()} ({(pobreza == 5).sum() / pobreza.notna().sum() * 100:.1f} % de los que tienen dato)")
print(f"Nulos: {pobreza.isna().sum()} ({pobreza.isna().sum() / raw_dataset.shape[0] * 100:.1f} % del total)")

In [ ]:
# si los nulos fueran completamente al azar (MCAR), su porcentaje sería parecido en todos los grupos
for etnia in raw_dataset["ethnicity"].value_counts().index:
  grupo = raw_dataset[raw_dataset["ethnicity"] == etnia]
  print(f"{etnia}: {grupo['poverty_income_ratio'].isna().sum() / grupo.shape[0] * 100:.1f} % sin índice de pobreza")

# lo mismo por grupo de edad
for edad_minima, edad_maxima in [(0, 18), (18, 60), (60, 81)]:
  grupo = raw_dataset[(raw_dataset["age"] >= edad_minima) & (raw_dataset["age"] < edad_maxima)]
  print(f"De {edad_minima} a {edad_maxima - 1} años: {grupo['poverty_income_ratio'].isna().sum() / grupo.shape[0] * 100:.1f} % sin índice de pobreza")

In [ ]:
# graficamos la distribución del índice de pobreza
plt.figure(figsize=(10, 5))
plt.hist(raw_dataset["poverty_income_ratio"].dropna(), bins=50, edgecolor="black")
plt.axvline(x=1, color="red", linestyle="--", label="umbral de pobreza (índice = 1)")
plt.xlabel("Ingreso familiar / umbral de pobreza")
plt.ylabel("Cantidad de personas")
plt.title("Distribución del índice de pobreza")
plt.legend()
plt.show()

**Observaciones**

- **[P10]** La barra de 5 es la más alta del histograma: 2.149 personas, el 21,7 % de quienes tienen el dato. Es un **tope**: todos los ingresos de 5 veces el umbral de pobreza o más quedan registrados como 5. La cola derecha real de la distribución está cortada, y cualquier media o desvío calculado sobre esta variable queda subestimado. **Por eso se resume con la mediana y los cuartiles**, que no dependen de cuánto vale exactamente el 21,7 % de arriba.
- **[P9]** Los 100 valores minúsculos son el 0 del índice, la otra punta de la escala.
- **[P11]** Hay 2.041 nulos (17,1 %). A diferencia de la educación, esta pregunta se hace a todas las edades, así que **no son faltantes por diseño**: son hogares que no informaron su ingreso. La falta **no es completamente al azar (MCAR)**: el porcentaje de nulos varía según el grupo étnico, de 14,2 % en blancos no hispanos a 23,5 % en asiáticos no hispanos, mientras que casi no varía con la edad (entre 16,9 % y 17,3 % en los tres grupos). Puede depender del grupo étnico (MAR) o del propio ingreso (MNAR, por ejemplo si los ingresos muy altos o muy bajos se informan menos); la segunda posibilidad no se puede verificar con los datos. En consecuencia, eliminar las filas sin ingreso cambiaría la composición étnica de lo que queda.
- La mediana es 2,50 y el primer cuartil 1,18: una cuarta parte de la muestra vive con un ingreso de hasta 1,18 veces el umbral de pobreza.

### 2.7 Medidas corporales y presión arterial

In [ ]:
# medidas resumen de las variables corporales y de presión
columnas_cuerpo = ["bmi", "weight_kg", "height_cm", "bp_systolic", "bp_diastolic"]
raw_dataset[columnas_cuerpo].describe()

In [ ]:
# histogramas de las cinco variables
raw_dataset[columnas_cuerpo].hist(figsize=(12, 8), bins=50, edgecolor="black")
plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
# separamos la altura de menores y de adultos para ver si se mezclan dos poblaciones
altura_menores = raw_dataset[raw_dataset["age"] < 18]["height_cm"].dropna()
altura_adultos = raw_dataset[raw_dataset["age"] >= 18]["height_cm"].dropna()

plt.figure(figsize=(10, 5))
plt.hist(altura_menores, bins=50, alpha=0.5, label=f"menores de 18 años (n = {altura_menores.shape[0]})")
plt.hist(altura_adultos, bins=50, alpha=0.5, label=f"18 años o más (n = {altura_adultos.shape[0]})")
plt.xlabel("Altura (cm)")
plt.ylabel("Cantidad de personas")
plt.title("Altura según grupo de edad")
plt.legend()
plt.show()

In [ ]:
# controles de coherencia de la presión arterial
sistolica = raw_dataset["bp_systolic"]
diastolica = raw_dataset["bp_diastolic"]

# los decimales se buscan solo entre los valores cargados: NaN % 1 da NaN, y NaN != 0 da True
print(f"Valores de sistólica con decimales: {(sistolica.dropna() % 1 != 0).sum()}")
print(f"Valores de diastólica con decimales: {(diastolica.dropna() % 1 != 0).sum()}")
print(f"Sistólica menor que diastólica: {(sistolica < diastolica).sum()}")
print(f"Personas con solo una de las dos presiones: {(sistolica.isna() != diastolica.isna()).sum()}")
print(f"Sistólica de 180 mmHg o más: {(sistolica >= 180).sum()}")
print(f"Diastólica menor a 40 mmHg: {(diastolica < 40).sum()}")

**Observaciones**

- **[P22] Dos poblaciones mezcladas.** El histograma de la altura tiene una cola izquierda larga que no es "ruido": son los chicos. Al separar por edad aparecen dos distribuciones distintas. El peso muestra lo mismo: va de 2,7 kg (bebés) a 248,2 kg. Cualquier medida resumen calculada sobre toda la muestra mezcla las dos.
- `bmi` tiene media 27,2 y mediana 26,4 (media mayor que mediana: asimetría hacia la derecha) y un máximo de 74,8, un valor extremo pero posible (obesidad severa). Se reconstruye exactamente desde el peso y la altura (sección 2.11.1). Incluye chicos desde los 2 años, en quienes el BMI no se interpreta con los cortes de adultos: es otra mezcla de poblaciones.
- **La presión arterial es coherente.** Todos los valores son enteros, la sistólica nunca es menor que la diastólica y las dos faltan siempre juntas. Hay 66 sistólicas de 180 mmHg o más y 10 diastólicas menores a 40: son clínicamente posibles (crisis hipertensiva, presión baja), no errores de carga.
- **[P20]** NHANES toma **tres lecturas** de presión por persona y no publica un promedio [8]. El archivo trae un solo valor por persona, entero, y la metadata no dice cuál es: puede ser la primera lectura o un promedio redondeado.
- La presión se mide desde los 8 años. En chicos los valores de referencia son distintos que en adultos: otra mezcla de poblaciones.

### 2.8 Análisis de sangre

In [ ]:
# medidas resumen de los análisis de sangre
columnas_sangre = ["serum_creatinine", "blood_urea_nitrogen", "albumin_serum", "phosphorus",
                   "bicarbonate", "calcium", "uric_acid"]
raw_dataset[columnas_sangre].describe()

In [ ]:
# asimetría y curtosis (pandas devuelve el exceso de curtosis, con corrección por tamaño de muestra)
pd.DataFrame({"asimetría": raw_dataset[columnas_sangre].skew(),
              "curtosis": raw_dataset[columnas_sangre].kurt()}).round(2)

In [ ]:
# histogramas de los análisis de sangre
raw_dataset[columnas_sangre].hist(figsize=(12, 9), bins=50, edgecolor="black")
plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
# los nulos de la creatinina, del eGFR y la categoría Unknown deberían coincidir
print(f"Nulos de serum_creatinine: {raw_dataset['serum_creatinine'].isna().sum()}")
print(f"Nulos de egfr: {raw_dataset['egfr'].isna().sum()}")
print(f"Filas con ckd_stage == Unknown: {(raw_dataset['ckd_stage'] == 'Unknown').sum()}")

# personas con algún análisis de sangre pero sin creatinina
sin_creatinina = raw_dataset[raw_dataset["serum_creatinine"].isna()]
print(f"Sin creatinina pero con albúmina en sangre: {sin_creatinina['albumin_serum'].notna().sum()}")

In [ ]:
# los valores más extremos: personas con eGFR menor a 15 (Stage 5)
raw_dataset[raw_dataset["egfr"] < 15][["age", "gender", "serum_creatinine", "blood_urea_nitrogen",
                                       "egfr", "albumin_creatinine_ratio", "ckd_stage"]]

**Observaciones**

- **No hay valores fisiológicamente imposibles.** Todos los rangos son compatibles con personas reales, incluso los extremos: calcio de 7,5 a 11,7 mg/dL, bicarbonato de 15 a 37 mmol/L y albúmina de 2,2 a 5,5 g/dL.
- **[P23] La creatinina es la variable más asimétrica del bloque** (asimetría de 15,6 y exceso de curtosis de 447). Su mediana es 0,83 mg/dL, pero el máximo es 15,17. La forma la explican unas pocas personas con falla renal: las 11 de la tabla de arriba tienen eGFR menor a 15 y creatininas de 3,6 a 15,2. **No son errores: son los casos que el trabajo estudia**, y eliminarlos por atípicos dejaría afuera a los enfermos más graves. La urea (`blood_urea_nitrogen`) muestra lo mismo, con menos intensidad. En estas dos variables el resumen adecuado es la **mediana con los cuartiles**: la mediana de la creatinina (0,83) casi no se mueve por esas 11 personas, la media sí.
- El calcio, el bicarbonato y la albúmina son casi simétricos.
- **Los conteos coinciden:** 5.607 nulos de creatinina, 5.607 de eGFR y 5.607 `Unknown`. Hay 40 personas con algunos análisis de sangre pero sin creatinina: tuvieron extracción, pero ese análisis puntual falta. Por eso los nulos del bloque de sangre varían un poco entre columnas.

### 2.9 Análisis de orina

In [ ]:
# medidas resumen de los análisis de orina
columnas_orina = ["urine_creatinine", "urine_albumin", "albumin_creatinine_ratio"]
raw_dataset[columnas_orina].describe()

In [ ]:
# histograma del ACR en su escala original
acr = raw_dataset["albumin_creatinine_ratio"].dropna()

plt.figure(figsize=(10, 4))
plt.hist(acr, bins=50, edgecolor="black")
plt.xlabel("ACR (mg/g)")
plt.ylabel("Cantidad de personas")
plt.title("ACR en escala original")
plt.show()

In [ ]:
# el mismo histograma usando el logaritmo en base 10 del ACR
plt.figure(figsize=(10, 4))
plt.hist(np.log10(acr), bins=50, edgecolor="black")
plt.axvline(x=np.log10(30), color="red", linestyle="--", label="ACR = 30 mg/g")
plt.xlabel("log10(ACR)")
plt.ylabel("Cantidad de personas")
plt.title("ACR en escala logarítmica")
plt.legend()
plt.show()

In [ ]:
# contamos las personas con albuminuria aumentada (ACR de 30 mg/g o más, el umbral de la metadata)
print(f"ACR menor a 30: {(acr < 30).sum()}")
print(f"ACR de 30 o más: {(acr >= 30).sum()}")

# personas con creatinina en orina pero sin albúmina en orina
sin_albumina = raw_dataset[raw_dataset["urine_creatinine"].notna() & raw_dataset["urine_albumin"].isna()]
print(f"Con creatinina en orina pero sin albúmina en orina: {sin_albumina.shape[0]}")

In [ ]:
# los valores más frecuentes de la albúmina en orina
raw_dataset["urine_albumin"].value_counts().head()

In [ ]:
# el ACR de quienes tienen el valor mínimo de albúmina en orina
albumina_minima = raw_dataset[raw_dataset["urine_albumin"] == raw_dataset["urine_albumin"].min()]
print(f"Personas con urine_albumin == {raw_dataset['urine_albumin'].min()}: {albumina_minima.shape[0]}")
print(f"  de ellas, con ACR menor a 30: {(albumina_minima['albumin_creatinine_ratio'] < 30).sum()}")

**Observaciones**

- **El ACR es la variable más asimétrica del dataset.** La mediana es 8,27 mg/g y la media 33,36, cuatro veces más grande, con un máximo de 14.708. En una variable que no puede ser negativa, el desvío (251) es **siete veces la media**: la dispersión viene de unos pocos valores muy altos (cola larga a la derecha). La albúmina en orina muestra lo mismo (desvío 176, media 36).
- En escala original el histograma es una sola barra contra el cero y no muestra nada. En escala logarítmica la distribución se vuelve casi simétrica y se ve dónde cae el umbral clínico de 30 mg/g. El logaritmo se usa **solo para graficar**: no modifica el dato ni interviene en ningún cálculo.
- En esta variable **la media no sirve como resumen**: se usan la mediana y los cuartiles. Los coeficientes de Pearson que la involucren quedan dominados por los extremos (sección 2.13.1).
- 1.009 personas (12,4 % de quienes tienen el dato) tienen ACR de 30 mg/g o más, el umbral de albuminuria aumentada que indica la metadata.
- **[P21]** Una persona tiene creatinina en orina pero no albúmina, así que no tiene ACR.
- **[P26] El valor 0,21 de `urine_albumin` no es una medición.** Aparece 488 veces, cuando el siguiente valor más frecuente aparece 13. Es también el mínimo de la variable. La documentación de NHANES (archivo `ALB_CR_L`) [9] informa exactamente **488 resultados por debajo del límite de detección** del laboratorio, y explica que en esos casos se carga un **valor de relleno**: el límite de detección dividido por la raíz de 2. Lo único que sabemos de esas 488 personas es que su albúmina es "menor que el límite". Es un dato **censurado por izquierda**, el caso opuesto a los topes de la edad y del ingreso.
  - Esto explica el pequeño grupo de valores bajos que se ve a la izquierda del histograma logarítmico: son ACR calculados con el valor de relleno.
  - Para clasificar no cambia nada: las 488 tienen ACR menor a 30, y con una albúmina por debajo del límite de detección no podría ser de otra forma. Pero sí afecta cualquier medida que use el valor numérico, como medias o correlaciones.

### 2.10 Antecedentes de diabetes y tabaquismo

In [ ]:
# frecuencias de cada código, incluidos los nulos
for columna in ["diabetes_diagnosed", "insulin_use", "diabetes_pills", "ever_smoked", "current_smoker"]:
  print(f"{columna}:")
  for codigo, cantidad in raw_dataset[columna].value_counts(dropna=False).sort_index().items():
    print(f"  {codigo}: {cantidad}")

In [ ]:
# proporción de "sí" entre quienes respondieron sí o no (sin códigos 7/9 ni nulos)
respondieron_tabaco = raw_dataset[raw_dataset["ever_smoked"].isin([1, 2])]
print(f"Fumó alguna vez: {(respondieron_tabaco['ever_smoked'] == 1).sum()} de {respondieron_tabaco.shape[0]} ({(respondieron_tabaco['ever_smoked'] == 1).sum() / respondieron_tabaco.shape[0] * 100:.1f} %)")

respondieron_diabetes = raw_dataset[raw_dataset["diabetes_diagnosed"].isin([1, 2, 3])]
print(f"Diabetes diagnosticada: {(respondieron_diabetes['diabetes_diagnosed'] == 1).sum()} de {respondieron_diabetes.shape[0]} ({(respondieron_diabetes['diabetes_diagnosed'] == 1).sum() / respondieron_diabetes.shape[0] * 100:.1f} %)")

In [ ]:
# ¿a quiénes se les hizo cada pregunta? miramos la respuesta a la pregunta anterior
print("Respuesta a diabetes_diagnosed de quienes tienen insulin_use:")
print(raw_dataset[raw_dataset["insulin_use"].notna()]["diabetes_diagnosed"].value_counts(dropna=False).sort_index())

print("\nRespuesta a diabetes_diagnosed de quienes tienen diabetes_pills:")
print(raw_dataset[raw_dataset["diabetes_pills"].notna()]["diabetes_diagnosed"].value_counts(dropna=False).sort_index())

print("\nRespuesta a ever_smoked de quienes tienen current_smoker:")
print(raw_dataset[raw_dataset["current_smoker"].notna()]["ever_smoked"].value_counts(dropna=False).sort_index())

In [ ]:
# personas que dicen no tener diabetes pero toman pastillas para la diabetes
no_diabeticos_con_pastillas = raw_dataset[(raw_dataset["diabetes_diagnosed"] == 2) & (raw_dataset["diabetes_pills"] == 1)]
print(f"Dicen no tener diabetes y toman pastillas para la diabetes: {no_diabeticos_con_pastillas.shape[0]}")

# los nulos de diabetes_diagnosed, ¿son bebés?
nulos_diabetes = raw_dataset[raw_dataset["diabetes_diagnosed"].isna()]
print(f"Nulos de diabetes_diagnosed: {nulos_diabetes.shape[0]} - con edad menor a 1: {(nulos_diabetes['age'] < 1).sum()}")

**Observaciones**

- **`diabetes_diagnosed`:** 1.081 sí, 10.371 no, 284 borderline, 4 con código 9 **[P12]** y 193 nulos, de los cuales 189 son bebés de menos de un año.
  - **[P14]** Los códigos son 1 = sí, 2 = no y 3 = borderline. La variable tiene un orden natural (no < borderline < sí) que **el código no respeta**: el borderline queda "por encima" del sí. Hay que tenerlo en cuenta antes de usar el código como número o de ordenarlo.
- **`insulin_use`:** solo la tienen las 1.081 personas con diabetes diagnosticada (344 sí, 737 no). Los 10.852 nulos son faltantes por diseño **[P15]**.
- **[P16] `diabetes_pills` se preguntó a un subconjunto definido por una variable que no está en el archivo:** a los 1.081 diabéticos, a 282 de los 284 borderline y a **918 de las 10.371 personas sin diabetes**. Una interpretación compatible con estos conteos es que esas 918 personas tengan un diagnóstico previo de **prediabetes**, una condición que el archivo no registra. En cualquier caso, entre las personas sin diabetes un nulo en `diabetes_pills` significa que la pregunta no se hizo, **no** que no tomen pastillas.
- **[P17]** 60 personas dicen no tener diabetes pero toman pastillas para bajar la glucosa. No se interpreta como un error: es consistente con el punto anterior, es decir, con personas con prediabetes, sin diagnóstico de diabetes, a las que se les indicó medicación oral. Por eso se conservan sin cambios (sección 2.15.6).
- **`ever_smoked`:** 3.243 sí, 4.878 no, 7 con código 7 y 7 con código 9 **[P12]**. Los 3.798 nulos son, en su gran mayoría, menores de 18 años **[P15]**. Como es binaria, se resume con una **proporción**: entre quienes respondieron sí o no, el 39,9 % fumó alguna vez (3.243 de 8.121).
- **`current_smoker`:** solo la tienen los 3.243 que fumaron alguna vez (952 todos los días, 238 algunos días y 2.053 que ya no fuman). Es **ordinal**, porque mide frecuencia, y su código va de mayor a menor frecuencia (1 = todos los días). Para medir "tabaquismo" hay que combinarla con `ever_smoked`: usada sola, deja afuera a todos los que nunca fumaron.
- Lo mismo vale para la diabetes: entre quienes respondieron 1, 2 o 3, el 9,2 % tiene diabetes diagnosticada (1.081 de 11.736). Es una proporción de la **muestra**, que incluye chicos y no usa pesos: no es la prevalencia de EE.UU. **[P25]**.

### 2.11 Variables derivadas y clasificación renal

#### 2.11.1 Verificación de `bmi`, `albumin_creatinine_ratio` y `egfr`

Las tres variables se calculan a partir de otras columnas del mismo archivo. Las recalculamos, sin modificar el dataset, para comprobar que son lo que la metadata dice que son.

In [ ]:
# recalculamos el BMI: peso (kg) / altura (m) al cuadrado
bmi_calculado = raw_dataset["weight_kg"] / (raw_dataset["height_cm"] / 100) ** 2
diferencia_bmi = (bmi_calculado - raw_dataset["bmi"]).abs()
print(f"BMI - filas comparadas: {diferencia_bmi.notna().sum()} - diferencia máxima: {diferencia_bmi.max():.4f}")

# recalculamos el ACR: albúmina (µg/mL) / creatinina (mg/dL) x 100 da mg/g
acr_calculado = raw_dataset["urine_albumin"] / raw_dataset["urine_creatinine"] * 100
diferencia_acr = (acr_calculado - raw_dataset["albumin_creatinine_ratio"]).abs()
print(f"ACR - filas comparadas: {diferencia_acr.notna().sum()} - diferencia máxima: {diferencia_acr.max():.4f}")

In [ ]:
# ecuación CKD-EPI 2021 sin raza, tal como la describe la metadata
def calcular_egfr(creatinina, edad, sexo):
  if pd.isna(creatinina):
    return float("nan")
  if sexo == "Female":
    kappa = 0.7
    alfa = -0.241
    factor_sexo = 1.012
  else:
    kappa = 0.9
    alfa = -0.302
    factor_sexo = 1
  cociente = creatinina / kappa
  return 142 * min(cociente, 1) ** alfa * max(cociente, 1) ** -1.200 * 0.9938 ** edad * factor_sexo

# aplicamos la función a cada fila y comparamos con la columna egfr del archivo
egfr_calculado = raw_dataset.apply(lambda fila: calcular_egfr(fila["serum_creatinine"], fila["age"], fila["gender"]), axis=1)
diferencia_egfr = (egfr_calculado - raw_dataset["egfr"]).abs()
print(f"eGFR - filas comparadas: {diferencia_egfr.notna().sum()} - diferencia máxima: {diferencia_egfr.max():.4f}")

In [ ]:
# personas de 80 años o más con eGFR: la ecuación usó 80 aunque la edad real sea mayor
print(f"Personas con edad 80 y eGFR calculada: {((raw_dataset['age'] == 80) & raw_dataset['egfr'].notna()).sum()}")

# cuánto cambia el eGFR por 5 años de diferencia (80 contra 85, la edad promedio del grupo)
print(f"Factor por 5 años de edad: 0.9938 ** 5 = {0.9938 ** 5:.4f}")

**Observaciones**

- **Las tres variables derivadas se reproducen exactamente.** Las diferencias máximas son 0,05 en el BMI, 0,005 en el ACR y 0,005 en el eGFR. Coinciden con la mitad de la última cifra que guarda el archivo: el BMI tiene un decimal y el ACR y el eGFR tienen dos, así que son errores de redondeo y no diferencias de cálculo. Esto confirma que el eGFR se calculó con **CKD-EPI 2021 sin ajuste por raza**, como declara la metadata.

Ejemplo con la primera fila (hombre de 43 años, creatinina 0,8 mg/dL):

```
creatinina / kappa = 0,8 / 0,9 = 0,8889      (menor que 1: el término max(...) vale 1)
min(...) ** alfa   = 0,8889 ** (-0,302) = 1,0362
0,9938 ** 43       = 0,7653
eGFR = 142 × 1,0362 × 1 × 0,7653 = 112,61    (el archivo dice 112.61)
```

- **[P5]** La ecuación usa la edad, y para las 298 personas de 80 años o más con eGFR usó **80** aunque su edad real sea mayor. Cada año de edad multiplica el eGFR por 0,9938. Para alguien de 85, la edad promedio del grupo, el factor de 5 años es 0,9694: **el eGFR del archivo le queda sobreestimado en un 3 % aproximadamente**.

#### 2.11.2 Reproducción de la regla de `ckd_stage`

Aplicamos la regla de la sección 5 de la metadata y la comparamos con la columna del archivo. Cuando la regla no se puede aplicar (eGFR de 90 o más, sin ACR), la función devuelve `"Sin ACR"` para que esos casos queden a la vista.

In [ ]:
# reproducimos la regla de la metadata para asignar el estadio
def asignar_estadio(egfr, acr):
  if pd.isna(egfr):
    return "Unknown"
  if egfr >= 90:
    if pd.isna(acr):
      return "Sin ACR"
    if acr >= 30:
      return "Stage 1 (Kidney Damage)"
    return "No CKD"
  if egfr >= 60:
    return "Stage 2 (Mildly Decreased)"
  if egfr >= 45:
    return "Stage 3a (Mild-Moderate)"
  if egfr >= 30:
    return "Stage 3b (Moderate-Severe)"
  if egfr >= 15:
    return "Stage 4 (Severely Decreased)"
  return "Stage 5 (Kidney Failure)"

estadio_calculado = raw_dataset.apply(lambda fila: asignar_estadio(fila["egfr"], fila["albumin_creatinine_ratio"]), axis=1)
no_coinciden = raw_dataset[estadio_calculado != raw_dataset["ckd_stage"]]
print(f"Filas donde la regla de la metadata no coincide con ckd_stage: {no_coinciden.shape[0]}")
print(f"Estadio calculado en esas filas: {estadio_calculado[estadio_calculado != raw_dataset['ckd_stage']].value_counts().to_dict()}")
print(f"Estadio del archivo en esas filas: {no_coinciden['ckd_stage'].value_counts().to_dict()}")

**Observaciones**

- La regla de la metadata reproduce `ckd_stage` en todas las filas **salvo 70**.
- **[P3]** Esas 70 personas tienen eGFR de 90 o más y **no tienen ACR**. La regla dice que *No CKD* exige ACR < 30, así que sin ACR no se las puede clasificar. El archivo igual las puso en *No CKD*: trató "sin dato de orina" como "orina normal". Es un supuesto que la metadata no declara.

#### 2.11.3 Distribución de `ckd_stage` y `ckd_present`

In [ ]:
# frecuencias de cada estadio (el orden alfabético coincide con el orden clínico)
conteo_estadios = raw_dataset["ckd_stage"].value_counts().sort_index()
conteo_estadios

In [ ]:
# graficamos los estadios en su orden clínico
plt.figure(figsize=(10, 5))
plt.bar(conteo_estadios.index, conteo_estadios.values, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cantidad de personas")
plt.title("Personas por estadio renal (ckd_stage)")
plt.show()

In [ ]:
# para cada estadio, qué valor toma ckd_present
for estadio in conteo_estadios.index:
  subconjunto = raw_dataset[raw_dataset["ckd_stage"] == estadio]
  print(f"{estadio}: {subconjunto['ckd_present'].value_counts().to_dict()}")

# total de ckd_present igual a 1
cantidad_ckd = (raw_dataset["ckd_present"] == 1).sum()
print(f"\nckd_present == 1: {cantidad_ckd} ({cantidad_ckd / raw_dataset.shape[0] * 100:.1f} % de la muestra)")

**Observaciones**

- `ckd_stage` es **ordinal**. El orden alfabético de sus etiquetas coincide con el clínico (No CKD, Stage 1, …, Stage 5), pero `Unknown` no es un nivel de esa escala: es la ausencia de dato **[P6]**.
- La categoría más grande es `Unknown` (5.607, el 47 %). Entre los que tienen estadio, la más frecuente es *No CKD* (3.592), seguida de *Stage 2* (1.875).
- **[P1] `ckd_present` vale 1 para el 69,9 % de la muestra** (8.341 personas). Pero 5.607 de esos "1" son `Unknown`, personas sin eGFR. Usar `ckd_present` tal como viene haría pasar por enfermas a todas las personas sin análisis de sangre, entre ellas 2.545 menores de 12 años. **No se usa en ningún análisis:** se elimina de la copia de trabajo (sección 2.15.5).

#### 2.11.4 ¿Quiénes son los `Unknown`?

In [ ]:
# separamos a las personas sin eGFR por grupo de edad
desconocidos = raw_dataset[raw_dataset["ckd_stage"] == "Unknown"]
print(f"Unknown: {desconocidos.shape[0]}")
print(f"  menores de 12 años (no se les extrae sangre): {(desconocidos['age'] < 12).sum()}")
print(f"  de 12 a 17 años: {((desconocidos['age'] >= 12) & (desconocidos['age'] < 18)).sum()}")
print(f"  de 18 años o más: {(desconocidos['age'] >= 18).sum()}")

# entre los adultos sin eGFR: ¿no fueron al examen o fueron pero no tienen análisis de sangre?
desconocidos_adultos = desconocidos[desconocidos["age"] >= 18]
adultos_sin_examen = desconocidos_adultos[desconocidos_adultos[columnas_examen].isna().all(axis=1)].shape[0]
print(f"  adultos sin ningún dato de examen ni de laboratorio: {adultos_sin_examen}")
print(f"  adultos con algún dato de examen pero sin creatinina: {desconocidos_adultos.shape[0] - adultos_sin_examen}")

In [ ]:
# porcentaje de Unknown dentro de cada grupo étnico, en toda la muestra y solo entre adultos
for etnia in raw_dataset["ethnicity"].value_counts().index:
  grupo = raw_dataset[raw_dataset["ethnicity"] == etnia]
  grupo_adultos = grupo[grupo["age"] >= 18]
  porcentaje_total = (grupo["ckd_stage"] == "Unknown").sum() / grupo.shape[0] * 100
  porcentaje_menores_12 = (grupo["age"] < 12).sum() / grupo.shape[0] * 100
  porcentaje_adultos = (grupo_adultos["ckd_stage"] == "Unknown").sum() / grupo_adultos.shape[0] * 100
  print(f"{etnia}: {porcentaje_total:.1f} % Unknown - {porcentaje_menores_12:.1f} % son menores de 12 - entre adultos: {porcentaje_adultos:.1f} % Unknown")

**Observaciones**

- De los 5.607 `Unknown`, **2.545 son menores de 12 años**, a quienes no se les extrae sangre, 567 tienen entre 12 y 17 y **2.495 son adultos**.
- De esos 2.495 adultos, **1.821 no tienen ningún dato de examen ni de laboratorio** (respondieron la entrevista pero no fueron al examen, **[P18]**) y **674 fueron al examen** (tienen peso, presión u orina) pero no tienen creatinina en sangre.
- **[P6] La falta de eGFR no es completamente al azar.** En los menores de 12 depende de la edad (no se les extrae sangre), una variable observada (MAR). En los 2.495 adultos sin eGFR no se puede determinar si la falta depende solo de variables observadas o también del propio estado renal (MNAR), por ejemplo si las personas más enfermas concurrieron menos al examen.
- El estado renal es la variable que se estudia, por lo que **no se imputa**. El tratamiento de estas filas y la limitación que implica se describen en la sección 2.15.5.
- **La proporción de `Unknown` cambia según el grupo étnico**: 42,0 % en blancos no hispanos contra 54,7 % en mexicano-americanos y 56,6 % en otro/multirracial. **Una parte se explica por la edad**: los grupos con más `Unknown` tienen más chicos (29 a 30 % de menores de 12, contra 16,7 % en blancos no hispanos). **Pero la diferencia sigue entre adultos**: 28,8 % de `Unknown` en blancos no hispanos contra 37,2 % en negros no hispanos. Esto importa para la hipótesis (a), que compara grupos étnicos.

#### 2.11.5 La regla de Stage 2 del archivo

In [ ]:
# en Stage 2 la regla del archivo no mira el ACR: vemos cuántos tienen albuminuria
estadio_2 = raw_dataset[raw_dataset["ckd_stage"] == "Stage 2 (Mildly Decreased)"]
print(f"Stage 2: {estadio_2.shape[0]}")
print(f"  con ACR menor a 30 (sin marcador de daño): {(estadio_2['albumin_creatinine_ratio'] < 30).sum()}")
print(f"  con ACR de 30 o más: {(estadio_2['albumin_creatinine_ratio'] >= 30).sum()}")
print(f"  sin ACR: {estadio_2['albumin_creatinine_ratio'].isna().sum()}")

In [ ]:
# comparamos, entre los adultos con eGFR, la proporción de CKD según la regla del archivo y según el criterio clínico (eGFR < 60 o ACR >= 30)
adultos_con_egfr = raw_dataset[(raw_dataset["age"] >= 18) & raw_dataset["egfr"].notna()]
ckd_archivo = (adultos_con_egfr["ckd_present"] == 1).sum()
print(f"Adultos con eGFR: {adultos_con_egfr.shape[0]}")
print(f"  CKD según el archivo (ckd_present == 1): {ckd_archivo} ({ckd_archivo / adultos_con_egfr.shape[0] * 100:.1f} %)")

# con el criterio clínico solo se puede clasificar a quien tiene eGFR < 60 o tiene ACR
clasificables = adultos_con_egfr[(adultos_con_egfr["egfr"] < 60) | adultos_con_egfr["albumin_creatinine_ratio"].notna()]
ckd_criterio = ((clasificables["egfr"] < 60) | (clasificables["albumin_creatinine_ratio"] >= 30)).sum()
print(f"  con eGFR >= 60 y sin ACR (no clasificables con el criterio clínico): {adultos_con_egfr.shape[0] - clasificables.shape[0]}")
print(f"  CKD según el criterio clínico: {ckd_criterio} de {clasificables.shape[0]} clasificables ({ckd_criterio / clasificables.shape[0] * 100:.1f} %)")

**Observaciones**

- **[P2]** De las 1.875 personas en *Stage 2*, **1.645 tienen ACR menor a 30**: un filtrado levemente disminuido sin ningún marcador de daño. La regla del archivo las cuenta como enfermedad renal crónica, mientras que la guía KDIGO [7] solo considera CKD a un eGFR de 60 a 89 si además hay daño. Este patrón es frecuente en personas mayores, porque el filtrado disminuye con la edad.
- Entre los adultos con eGFR, la regla del archivo marca como CKD al **46,8 %**; el criterio clínico (eGFR < 60 o ACR ≥ 30), al **17,4 %** de los clasificables. Como referencia, el CDC estima alrededor de un 14 % de CKD en adultos de EE.UU. con datos de NHANES [10], una cifra no directamente comparable porque usa los pesos muestrales y excluye el *Stage 5*.
- Los 83 adultos con eGFR de 60 o más y sin ACR no se pueden clasificar con el criterio clínico: suponer que su ACR es normal sería repetir el supuesto del **[P3]**.
- Por esta diferencia, los análisis de enfermedad renal usan el criterio clínico, que se construye en la sección 2.15.5 (columna `ckd_clinical`).

#### 2.11.6 eGFR en adolescentes

In [ ]:
# adolescentes de 12 a 17 años con eGFR calculada con una ecuación para adultos
adolescentes = raw_dataset[(raw_dataset["age"] >= 12) & (raw_dataset["age"] < 18) & raw_dataset["egfr"].notna()]
print(f"Adolescentes con eGFR: {adolescentes.shape[0]}")
adolescentes["ckd_stage"].value_counts()

**Observaciones**

- **[P4]** 668 adolescentes de 12 a 17 años tienen un eGFR calculado con CKD-EPI 2021, una ecuación desarrollada y validada en adultos; en pediatría se usan otras ecuaciones (por ejemplo, la de Schwartz). Además, 85 de esos adolescentes quedaron en *Stage 1* por tener ACR ≥ 30.
- Por eso los análisis de enfermedad renal se restringen a adultos (sección 2.15.5).

### 2.12 Forma, dispersión y valores atípicos

#### 2.12.1 Medidas resumen según la forma de cada variable

Para cada variable cuantitativa se calculan la media, la mediana, el desvío, el IQR (`Q3 − Q1`), el coeficiente de variación (`CV = desvío / media × 100`) y la asimetría. La medida de centro y de dispersión que se reporta depende de la forma: media y desvío para las distribuciones casi simétricas; mediana e IQR para las asimétricas, censuradas o con atípicos. El CV se usa para comparar la dispersión de variables con unidades distintas, y es aplicable porque todas tienen un cero verdadero (sección 2.3). La asimetría se calcula con `skew()`, que incluye una corrección por tamaño de muestra.

In [ ]:
# lista de las variables cuantitativas continuas
columnas_continuas = ["age", "poverty_income_ratio", "bmi", "weight_kg", "height_cm",
                      "bp_systolic", "bp_diastolic", "serum_creatinine", "blood_urea_nitrogen",
                      "albumin_serum", "phosphorus", "bicarbonate", "calcium", "uric_acid",
                      "urine_creatinine", "urine_albumin", "albumin_creatinine_ratio", "egfr"]

# para cada variable: centro, dispersión absoluta y relativa, y asimetría
filas_resumen = []
for columna in columnas_continuas:
  estadisticas = raw_dataset[columna].describe()
  filas_resumen.append({"variable": columna,
                        "n": int(estadisticas["count"]),
                        "media": round(estadisticas["mean"], 2),
                        "mediana": round(estadisticas["50%"], 2),
                        "desvío": round(estadisticas["std"], 2),
                        "IQR": round(estadisticas["75%"] - estadisticas["25%"], 2),
                        "CV (%)": round(estadisticas["std"] / estadisticas["mean"] * 100, 1),
                        "asimetría": round(raw_dataset[columna].skew(), 2)})

tabla_resumen = pd.DataFrame(filas_resumen)
tabla_resumen

**Observaciones**

- **Casi simétricas** (media ≈ mediana, asimetría chica): `egfr`, `albumin_serum`, `bicarbonate`, `calcium`, y con asimetría leve `phosphorus`, `uric_acid` y `bp_diastolic`. Para ellas, media y desvío son un resumen razonable.
- **Asimétricas a la derecha** (media mayor que mediana): `albumin_creatinine_ratio`, `urine_albumin`, `serum_creatinine`, `blood_urea_nitrogen`, `urine_creatinine`, `bmi` y `bp_systolic`. El ACR y la albúmina en orina tienen CV de 753 % y 494 %: desvío varias veces mayor que la media. Se resumen con mediana e IQR.
- **Censuradas** (`age` en 80, `poverty_income_ratio` en 5, `urine_albumin` por debajo del límite de detección): aunque la edad y el índice de pobreza tengan asimetría casi nula, el tope sesga la media y el desvío hacia abajo: poco en la edad (el 4,4 % está en 80) y bastante en el índice de pobreza (el 21,7 % está en 5). Mediana y cuartiles.
- **Mezcla de poblaciones** (`height_cm`, `weight_kg`): la altura tiene media **menor** que la mediana y asimetría negativa (−1,51) por la cola de los chicos; el peso tiene media y mediana parecidas, pero eso no la vuelve una sola población. Se resumen por grupo de edad (secciones 2.7 y 2.12.2).
- **Comparadas por CV**, las variables más homogéneas son el calcio (4 %) y la albúmina en sangre (9 %). Sus desvíos crudos (0,38 y 0,36) no son comparables con el de la creatinina en orina (90,7), porque están en escalas distintas.

#### 2.12.2 Valores atípicos

Se distingue entre **valor atípico** (alejado del resto de la distribución) y **error** (valor incorrecto según el dominio o la metadata): un atípico puede ser real, y un error puede no ser atípico (los ceros disfrazados de la sección 2.4.2 quedan dentro del rango). Para detectar atípicos se combinan tres criterios:

- **Dominio:** rangos fisiológicamente posibles, revisados en las secciones 2.7 a 2.9 (no se encontraron valores imposibles).
- **Tukey:** atípico si queda por debajo de `Q1 − 1,5 × IQR` o por encima de `Q3 + 1,5 × IQR`.
- **Puntaje z:** atípico si `|z| > 3`, con `z = (valor − media) / desvío`. Supone normalidad y es sensible a los propios extremos, que inflan el desvío.

Los cuartiles se toman de `describe()` (interpolación lineal). Como los límites dependen de la población sobre la que se calculan, cada variable se analiza con toda la muestra y solo con los adultos.

In [ ]:
# función auxiliar: límites de Tukey y cantidad de valores atípicos de una serie
def contar_atipicos(serie):
  estadisticas = serie.describe()
  iqr = estadisticas["75%"] - estadisticas["25%"]
  limite_inferior = estadisticas["25%"] - 1.5 * iqr
  limite_superior = estadisticas["75%"] + 1.5 * iqr
  cantidad = ((serie < limite_inferior) | (serie > limite_superior)).sum()
  return limite_inferior, limite_superior, cantidad

# función auxiliar: cantidad de valores con puntaje z mayor a 3 en valor absoluto
def contar_atipicos_z(serie):
  estadisticas = serie.describe()
  puntaje_z = (serie - estadisticas["mean"]) / estadisticas["std"]
  return (puntaje_z.abs() > 3).sum()

In [ ]:
# contamos atípicos en cada variable continua, con los dos criterios, en toda la muestra y solo en adultos
adultos = raw_dataset[raw_dataset["age"] >= 18]

filas_tabla = []
for columna in columnas_continuas:
  _, _, atipicos_muestra = contar_atipicos(raw_dataset[columna])
  _, _, atipicos_adultos = contar_atipicos(adultos[columna])
  filas_tabla.append({"variable": columna,
                      "Tukey (toda la muestra)": atipicos_muestra,
                      "% muestra": round(atipicos_muestra / raw_dataset[columna].notna().sum() * 100, 1),
                      "Tukey (solo adultos)": atipicos_adultos,
                      "% adultos": round(atipicos_adultos / adultos[columna].notna().sum() * 100, 1),
                      "|z| > 3 (toda la muestra)": contar_atipicos_z(raw_dataset[columna]),
                      "|z| > 3 (solo adultos)": contar_atipicos_z(adultos[columna])})

tabla_atipicos = pd.DataFrame(filas_tabla)
tabla_atipicos

In [ ]:
# la altura como ejemplo: los límites cambian según el grupo con el que se calculan
limite_inferior_muestra, limite_superior_muestra, _ = contar_atipicos(raw_dataset["height_cm"])
limite_inferior_adultos, limite_superior_adultos, _ = contar_atipicos(adultos["height_cm"])
print(f"Límites con toda la muestra: {limite_inferior_muestra:.2f} a {limite_superior_muestra:.2f} cm")
print(f"Límites con solo adultos: {limite_inferior_adultos:.2f} a {limite_superior_adultos:.2f} cm")

# adultos que quedan afuera de los límites calculados con toda la muestra
altura_adultos = adultos["height_cm"]
afuera = ((altura_adultos < limite_inferior_muestra) | (altura_adultos > limite_superior_muestra)).sum()
print(f"Adultos fuera de los límites de toda la muestra: {afuera}")

# boxplot de la altura por grupo de edad
grupo_edad = raw_dataset["age"].apply(lambda x: "menor de 18" if x < 18 else "18 o más")
plt.figure(figsize=(8, 5))
sns.boxplot(x=grupo_edad, y=raw_dataset["height_cm"])
plt.xlabel("Grupo de edad")
plt.ylabel("Altura (cm)")
plt.title("Altura según grupo de edad")
plt.show()

**Observaciones**

- **[P22] La altura es el ejemplo más claro de mezcla de poblaciones.** Con los cuartiles de toda la muestra hay 770 atípicos (9,1 %), casi todos chicos. Con los cuartiles de los adultos quedan 10 (0,2 %). Y si a los adultos se les aplican los límites de toda la muestra, queda afuera uno solo. Los tres números son correctos: responden preguntas distintas. **Un valor es atípico respecto de una población**, y hay que decir siempre cuál.
- El efecto también puede ir al revés: en el calcio, restringir a adultos **aumenta** los atípicos de Tukey (de 93 a 171), porque la distribución de los adultos es más angosta y sus límites quedan más cerca.
- **Los dos criterios no coinciden, y la diferencia se explica por la forma.** En las variables muy asimétricas, Tukey marca como atípica buena parte de la cola derecha (en el ACR, 1.003 valores, el 12 %), y el puntaje z marca muchos menos (46): el desvío está tan inflado por los extremos (251, siete veces la media) que hace falta un valor enorme para llegar a `|z| > 3`. Ninguno de los dos alcanza solo. Y en ninguna de las dos variables esos valores son "errores": es la cola derecha real de la distribución (sección 2.9). En la creatinina, los atípicos incluyen a las personas con falla renal **[P23]**.
- `age` y `poverty_income_ratio` no tienen atípicos con ningún criterio: son variables acotadas por diseño (de 0 a 80 y de 0 a 5).
- **Ningún atípico encontrado es un error de carga**, y los errores que sí se encontraron (ceros disfrazados, códigos 7/9) no son atípicos. El tratamiento de los atípicos se define en la sección 2.15.

### 2.13 Relaciones entre variables

La herramienta para estudiar cada par de variables depende de la combinación de tipos: correlación de Pearson y de Spearman, con diagramas de dispersión, entre dos cuantitativas (2.13.1); medidas resumen y boxplots por grupo entre una cuantitativa y una cualitativa (2.13.2); y proporciones dentro de cada grupo entre dos cualitativas (2.13.3). Los tests correspondientes se aplican en la sección 3.

#### 2.13.1 Cuantitativa con cuantitativa: correlaciones

In [ ]:
# matriz de correlación de Pearson entre las variables continuas
matriz_correlacion = raw_dataset[columnas_continuas].corr()

plt.figure(figsize=(15, 12))
sns.heatmap(matriz_correlacion, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación de Pearson entre variables continuas (datos crudos)")
plt.show()

In [ ]:
# la misma matriz con el coeficiente de Spearman, calculado sobre los rangos
matriz_spearman = raw_dataset[columnas_continuas].corr(method="spearman")

plt.figure(figsize=(15, 12))
sns.heatmap(matriz_spearman, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación de Spearman entre variables continuas (datos crudos)")
plt.show()

In [ ]:
# pares con correlación fuerte (|r| >= 0,7) con alguno de los dos coeficientes, recorriendo solo la mitad superior de la matriz
print("Pares con |r| >= 0,7:")
for i, fila in enumerate(columnas_continuas):
  for columna in columnas_continuas[i + 1:]:
    pearson = matriz_correlacion.loc[fila, columna]
    spearman = matriz_spearman.loc[fila, columna]
    if abs(pearson) >= 0.7 or abs(spearman) >= 0.7:
      print(f"  {fila} - {columna}: Pearson {pearson:.2f}, Spearman {spearman:.2f}")

# pares donde los dos coeficientes difieren en 0,3 o más
print("\nPares donde Pearson y Spearman difieren en 0,3 o más:")
for i, fila in enumerate(columnas_continuas):
  for columna in columnas_continuas[i + 1:]:
    pearson = matriz_correlacion.loc[fila, columna]
    spearman = matriz_spearman.loc[fila, columna]
    if abs(pearson - spearman) >= 0.3:
      print(f"  {fila} - {columna}: Pearson {pearson:.2f}, Spearman {spearman:.2f}")

In [ ]:
# graficamos los dos pares que mejor muestran la diferencia entre Pearson y Spearman
pares_creatinina = raw_dataset[["serum_creatinine", "egfr"]].dropna()
plt.figure(figsize=(6, 6))
plt.scatter(pares_creatinina["serum_creatinine"], pares_creatinina["egfr"], alpha=0.3)
plt.xlabel("Creatinina en sangre (mg/dL)")
plt.ylabel("eGFR (mL/min/1,73 m²)")
plt.title("Creatinina y eGFR")
plt.show()

pares_acr = raw_dataset[["serum_creatinine", "albumin_creatinine_ratio"]].dropna()
plt.figure(figsize=(6, 6))
plt.scatter(pares_acr["serum_creatinine"], pares_acr["albumin_creatinine_ratio"], alpha=0.3)
plt.xlabel("Creatinina en sangre (mg/dL)")
plt.ylabel("ACR (mg/g)")
plt.title("Creatinina en sangre y ACR")
plt.show()

**Observaciones**

- **Los pares con |r| ≥ 0,7 son, al menos en parte, relaciones por construcción, no hallazgos:**
  - `bmi` – `weight_kg` (Pearson 0,91, Spearman 0,92): el BMI se calcula con el peso.
  - `urine_albumin` – `albumin_creatinine_ratio` (0,83 y 0,61): el ACR se calcula con la albúmina.
  - `age` – `egfr` (−0,78 y −0,79): la ecuación del eGFR incluye la edad, y además el filtrado baja con los años.
  - `serum_creatinine` – `egfr` (−0,57 y −0,71): el eGFR se calcula con la creatinina.
  - `weight_kg` – `height_cm` (Pearson 0,73): está inflada por la mezcla de chicos y adultos **[P22]**.
- **Creatinina y eGFR:** el gráfico muestra una **curva**, no una recta, y unos pocos valores de falla renal muy a la derecha. Pearson (relación lineal, sensible a los extremos) da −0,57 y Spearman (relación monótona) da −0,71.
- **Creatinina en sangre y ACR: los dos coeficientes dicen cosas opuestas.** Pearson da 0,50 (asociación positiva moderada) y Spearman −0,11 (casi nada). El gráfico muestra por qué: casi todos los puntos se amontonan contra el origen, y unas pocas personas con falla renal tienen la creatinina y el ACR altísimos a la vez. **Esos pocos puntos fabrican el Pearson.** Lo mismo pasa entre la creatinina en sangre y la albúmina en orina (Pearson 0,41, Spearman 0,05). Con variables tan asimétricas, un coeficiente de Pearson no se interpreta sin el gráfico y sin el de Spearman.
- Al revés, `urine_creatinine` – `urine_albumin` tiene Pearson 0,08 y Spearman 0,59: los extremos de la albúmina en orina **esconden** una relación monótona que sí existe (una orina más concentrada tiene más de las dos cosas). Es justamente lo que corrige el ACR al dividir una por la otra.
- **Cada coeficiente se calcula sobre un subconjunto distinto de personas.** `corr()` descarta los nulos de a pares: la correlación peso–altura incluye chicos de 2 años, pero las de laboratorio solo personas de 12 años o más. Los coeficientes de la matriz **no son comparables entre sí**, porque no describen a la misma población.
- Todo esto se calculó sobre los datos crudos, con las mezclas de poblaciones y los valores extremos incluidos. Una correlación describe asociación, **no causa**. Las correlaciones relevantes para las hipótesis se recalculan en la sección 3, sobre las poblaciones de análisis.

#### 2.13.2 Cuantitativa con cualitativa: presión sistólica según la función renal

Esta exploración se hace sobre los datos crudos, antes de construir el criterio clínico de la sección 2.15.5. Por eso, en esta sección y en la siguiente se usa un indicador que no depende de esa construcción: **eGFR menor a 60**, que es enfermedad renal tanto con la regla del archivo (estadios 3a a 5) como con el criterio clínico. Se trabaja con los adultos que tienen eGFR, porque a los chicos se les aplica una ecuación de adultos **[P4]**. Las pruebas formales van en la sección 3.

In [ ]:
# adultos con eGFR, separados en dos grupos según si el eGFR es menor a 60
adultos_con_egfr = raw_dataset[(raw_dataset["age"] >= 18) & raw_dataset["egfr"].notna()]
grupo_renal = adultos_con_egfr["egfr"].apply(lambda x: "eGFR < 60" if x < 60 else "eGFR >= 60")
print(f"Adultos con eGFR: {adultos_con_egfr.shape[0]}")
print(f"  con eGFR < 60: {(grupo_renal == 'eGFR < 60').sum()}")

# medidas resumen de la presión sistólica en cada grupo
for grupo in ["eGFR >= 60", "eGFR < 60"]:
  presion_grupo = adultos_con_egfr[grupo_renal == grupo]["bp_systolic"]
  estadisticas = presion_grupo.describe()
  print(f"{grupo}: n = {estadisticas['count']:.0f}, Q1 = {estadisticas['25%']:.0f}, mediana = {estadisticas['50%']:.0f}, Q3 = {estadisticas['75%']:.0f} mmHg")

# boxplot de la presión sistólica por grupo
plt.figure(figsize=(7, 5))
sns.boxplot(x=grupo_renal, y=adultos_con_egfr["bp_systolic"], order=["eGFR >= 60", "eGFR < 60"])
plt.xlabel("Grupo de función renal")
plt.ylabel("Presión sistólica (mmHg)")
plt.title("Presión sistólica según el eGFR (adultos)")
plt.show()

In [ ]:
# ¿la edad está detrás de las dos cosas? miramos la presión y el porcentaje con eGFR < 60 por grupo de edad
for edad_minima, edad_maxima in [(18, 40), (40, 60), (60, 81)]:
  grupo = adultos_con_egfr[(adultos_con_egfr["age"] >= edad_minima) & (adultos_con_egfr["age"] < edad_maxima)]
  porcentaje_egfr_bajo = (grupo["egfr"] < 60).sum() / grupo.shape[0] * 100
  print(f"De {edad_minima} a {edad_maxima - 1} años: n = {grupo.shape[0]}, mediana de la sistólica = {grupo['bp_systolic'].describe()['50%']:.0f} mmHg, eGFR < 60 = {porcentaje_egfr_bajo:.1f} %")

**Observaciones**

- Hay 5.658 adultos con eGFR, y 477 tienen eGFR menor a 60. Los que tienen presión registrada son 5.027 y 457 en cada grupo.
- **El grupo con eGFR menor a 60 tiene la presión sistólica más alta:** mediana 126 mmHg contra 119, y la caja (el 50 % central) corrida hacia arriba (114 a 143 contra 110 a 131). Es una diferencia de **asociación**, que motiva la hipótesis (b).
- **Pero la edad está detrás de las dos variables.** Al pasar de los 18–39 a los 60 años o más, la mediana de la sistólica sube de 112 a 127 mmHg y el porcentaje con eGFR menor a 60 sube de 0,1 % a 16,9 %. Parte de la diferencia entre los dos grupos puede ser simplemente que el grupo con eGFR bajo es más viejo. Es un **confusor**: cualquier análisis de la hipótesis (b) tiene que tener en cuenta la edad.
- Los grupos se comparan con medianas y boxplots porque la presión sistólica es asimétrica a la derecha (sección 2.12.1) y los grupos tienen tamaños muy distintos (5.027 contra 457).

#### 2.13.3 Cualitativa con cualitativa: grupo étnico y función renal

In [ ]:
# para cada grupo étnico: cuántos adultos tienen eGFR < 60, qué parte de todos los casos son, y qué proporción del grupo representan
casos_totales = (adultos_con_egfr["egfr"] < 60).sum()
proporciones_etnia = []
for etnia in adultos_con_egfr["ethnicity"].value_counts().index:
  grupo = adultos_con_egfr[adultos_con_egfr["ethnicity"] == etnia]
  casos = (grupo["egfr"] < 60).sum()
  proporciones_etnia.append({"grupo étnico": etnia,
                             "adultos con eGFR": grupo.shape[0],
                             "casos (eGFR < 60)": casos,
                             "% de todos los casos": round(casos / casos_totales * 100, 1),
                             "% dentro del grupo": round(casos / grupo.shape[0] * 100, 1),
                             "edad mediana": grupo["age"].describe()["50%"],
                             "% de 60 años o más": round((grupo["age"] >= 60).sum() / grupo.shape[0] * 100, 1)})

tabla_etnia = pd.DataFrame(proporciones_etnia)
tabla_etnia

In [ ]:
# graficamos la proporción dentro de cada grupo, ordenada de mayor a menor
tabla_etnia_ordenada = tabla_etnia.sort_values("% dentro del grupo", ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(tabla_etnia_ordenada["grupo étnico"], tabla_etnia_ordenada["% dentro del grupo"], color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("% de adultos con eGFR < 60")
plt.title("Proporción de adultos con eGFR < 60 dentro de cada grupo étnico")
plt.show()

**Observaciones**

- **Conteo y proporción responden preguntas distintas.** Los blancos no hispanos son el 66,0 % de todos los casos con eGFR menor a 60 (315 de 477), pero dentro de su grupo la proporción es 9,4 %, **menor** que la de los negros no hispanos (12,5 %). La hipótesis (a) habla de "la mayoría de los casos": es una pregunta sobre el conteo, y el conteo depende de cuántas personas de cada grupo hay en la muestra (los blancos no hispanos son 3.350 de los 5.658 adultos con eGFR).
- **La edad vuelve a aparecer como confusor.** Los blancos no hispanos tienen la edad mediana más alta (60 años, y el 51 % tiene 60 o más), contra 44 años en los mexicano-americanos (24 % de 60 o más). Como el eGFR baja con la edad, un grupo más viejo va a tener más casos aunque su riesgo a igual edad sea el mismo.
- La muestra no tiene pesos **[P25]** y la falta de eGFR varía por grupo étnico **[P6]**: estas proporciones describen a los adultos con eGFR de la muestra, no a la población de EE.UU.

### 2.14 Resumen de problemas detectados en los datos crudos

| ID | Variable(s) | Problema | Evidencia | Sección |
|---|---|---|---|---|
| **Variables derivadas y clasificación renal** | | | | |
| P1 | `ckd_present` | Los `Unknown` (sin eGFR) están codificados como 1, igual que los enfermos | 8.341 "1" (69,9 %), de los cuales 5.607 son `Unknown` | 2.11.3 |
| P2 | `ckd_stage` | Stage 2 no exige marcador de daño: cuenta como CKD lo que KDIGO no considera CKD | 1.645 de 1.875 en Stage 2 tienen ACR < 30; adultos: 46,8 % contra 17,4 % | 2.11.5 |
| P3 | `ckd_stage` | Personas sin ACR clasificadas como *No CKD*, cuando la regla exige ACR < 30 | 70 con eGFR ≥ 90 y ACR faltante | 2.11.2 |
| P4 | `egfr` | Ecuación de adultos aplicada a adolescentes | 668 de 12 a 17 años; 85 en Stage 1 | 2.11.6 |
| P5 | `egfr`, `age` | La ecuación usó la edad censurada en 80 | 298 personas: eGFR sobreestimado ~3 % a los 85 años | 2.11.1 |
| P6 | `ckd_stage` | `Unknown` es un faltante disfrazado de categoría, y no falta al azar | 5.607: 2.545 menores de 12, 567 de 12 a 17, 2.495 adultos (1.821 sin examen y 674 con examen pero sin sangre); varía por etnia | 2.11.4 |
| **Codificación** | | | | |
| P7 | `age` | El cero aparece como `5.397605e-79` (son bebés de menos de un año) | 189 valores | 2.4.2 |
| P8 | `age` | Censura en 80 ("80 o más") | 525 personas | 2.6.1 |
| P9 | `poverty_income_ratio` | El cero aparece como `5.397605e-79`; la metadata solo lo advierte para la edad | 100 valores | 2.4.2 |
| P10 | `poverty_income_ratio` | Censura en 5 ("5 o más") | 2.149 personas (21,7 % de quienes tienen dato) | 2.6.4 |
| P11 | `poverty_income_ratio` | Ingreso no informado (no es faltante por diseño) y no al azar | 2.041 nulos (17,1 %); de 14,2 % a 23,5 % según el grupo étnico | 2.6.4 |
| P12 | `education_level`, `diabetes_diagnosed`, `diabetes_pills`, `ever_smoked` | Códigos 7/9 ("se negó" / "no sabe") guardados como números | 11 + 4 + 7 + 14 casos | 2.4.2 |
| P13 | `participant_id` y columnas de códigos | Categóricas e identificador guardados como `float64` (el CSV escribe todos los números con `.0`); `ckd_stage` es texto pero ordinal | 7 de las 25 columnas `float64` no son cantidades | 2.2, 2.3 |
| P14 | `diabetes_diagnosed` | El código (1 sí, 2 no, 3 borderline) no respeta el orden natural de la variable | — | 2.10 |
| P26 | `urine_albumin` (y el ACR que se calcula con ella) | Valores por debajo del límite de detección reemplazados por un valor de relleno (0,21): censura por izquierda | 488 valores iguales a 0,21; la documentación de NHANES informa 488 | 2.9 |
| **Faltantes y cuestionario** | | | | |
| P15 | `insulin_use`, `current_smoker`, `education_level`, tabaquismo, laboratorio, presión, altura | Faltantes por diseño (pregunta o examen que no correspondía): no se imputan | `insulin_use`: 10.852 nulos; `current_smoker`: 8.690; educación: 4.124 menores de 20; etc. | 2.4.3, 2.10 |
| P16 | `diabetes_pills` | Se preguntó a un subconjunto definido por una variable que no está en el archivo (interpretación: diagnóstico previo de prediabetes) | 918 no diabéticos tienen la pregunta | 2.10 |
| P17 | `diabetes_diagnosed`, `diabetes_pills` | Respuestas aparentemente incoherentes (interpretación: prediabetes con medicación oral) | 60 dicen no tener diabetes y toman pastillas | 2.10 |
| **Metadata contra datos** | | | | |
| P18 | todo el dataset | La metadata dice "cada fila es una persona examinada" | 3.089 filas sin ningún dato de examen ni laboratorio | 2.4.3 |
| P19 | `height_cm` | La metadata dice "altura o longitud", pero no hay datos de menores de 2 años | 0 de 368 menores de 2 años | 2.4.3 |
| P20 | `bp_systolic`, `bp_diastolic` | Un solo valor de las tres lecturas de NHANES, sin aclarar cuál | — | 2.7 |
| P21 | `albumin_creatinine_ratio` | Creatinina en orina sin albúmina, así que no hay ACR | 1 persona | 2.9 |
| **Distribución y población** | | | | |
| P22 | altura, peso, BMI, presión y correlaciones | Mezcla de chicos y adultos | altura: 770 atípicos de Tukey en toda la muestra contra 10 entre adultos | 2.7, 2.12 |
| P23 | `serum_creatinine`, `albumin_creatinine_ratio`, `urine_albumin` | Valores extremos clínicamente reales: no son errores, pero dominan la media, el desvío y Pearson | creatinina hasta 15,17; ACR hasta 14.708 (CV 753 %); creatinina–ACR: Pearson 0,50, Spearman −0,11 | 2.8, 2.9, 2.12, 2.13.1 |
| P24 | todo el dataset | Filas iguales salvo el identificador (personas distintas) | 196 combinaciones en 560 filas: 364 copias sobrantes | 2.5 |
| P25 | todo el dataset | Sin pesos muestrales: se describe la muestra, no la población | — | 1.1, 2.6.2, 2.13.3 |

**Hasta esta sección no se modificó ningún dato.** El tratamiento de cada problema se aplica y se justifica en la sección 2.15.

### 2.15 Limpieza

La limpieza se aplica sobre una copia del dataset crudo, `preprocessed_dataset`: `raw_dataset` queda sin cambios y el archivo CSV no se modifica. Criterios generales:

- **No se elimina ninguna fila.** Las restricciones que necesita cada análisis (por ejemplo, solo adultos) se aplican como selecciones con nombre, definidas una sola vez en la sección 2.15.7.
- **No se imputan faltantes.** Los faltantes por diseño corresponden a preguntas o exámenes que no correspondían **[P15]**; los que no son por diseño, como el ingreso no informado, no son completamente al azar **[P11]**; y el estado renal es la variable que se estudia, por lo que no se completa con valores estimados. Cada análisis usa los casos completos de las variables que necesita e informa cuántos quedan.
- **No se eliminan valores atípicos.** No se encontraron valores imposibles (sección 2.12.2), y los extremos de las variables renales son los casos que el trabajo estudia **[P23]**. La sensibilidad a los extremos se maneja en cada análisis, con medidas y tests adecuados a la forma de cada variable.
- Cada acción indica el problema que resuelve **[P#]** y se verifica con conteos antes y después.

#### 2.15.1 Copia de trabajo

In [ ]:
# copiamos el dataset crudo: todas las correcciones se hacen sobre esta copia
preprocessed_dataset = raw_dataset.copy()
print(f"Filas: {preprocessed_dataset.shape[0]} - Columnas: {preprocessed_dataset.shape[1]}")

#### 2.15.2 Ceros disfrazados [P7] [P9]

El valor `5.397605e-79` de `age` y de `poverty_income_ratio` representa un cero (sección 2.4.2). Se reemplaza por **0**, que es el valor real, y no por un faltante:

- en la edad corresponde a bebés de menos de un año;
- en el índice de pobreza, la metadata indica que el cero representa un ingreso muy bajo. Dejarlo como faltante excluiría de los análisis socioeconómicos a los hogares de menores ingresos (76 adultos) y desplazaría la muestra hacia ingresos más altos.

In [ ]:
# reemplazamos el valor minúsculo por 0 y verificamos que la cantidad de ceros coincida con la de valores minúsculos
for columna in ["age", "poverty_income_ratio"]:
  minusculos_antes = ((preprocessed_dataset[columna] > 0) & (preprocessed_dataset[columna] < 0.001)).sum()
  preprocessed_dataset[columna] = preprocessed_dataset[columna].apply(lambda x: 0 if x < 0.001 else x)
  ceros_despues = (preprocessed_dataset[columna] == 0).sum()
  print(f"{columna}: valores minúsculos antes = {minusculos_antes} - ceros después = {ceros_despues} - nulos = {preprocessed_dataset[columna].isna().sum()}")

# adultos con índice de pobreza igual a 0
adultos_pobreza_cero = ((preprocessed_dataset["age"] >= 18) & (preprocessed_dataset["poverty_income_ratio"] == 0)).sum()
print(f"Adultos con índice de pobreza igual a 0: {adultos_pobreza_cero}")

**Verificación.** Los ceros nuevos coinciden con los valores minúsculos (189 en la edad y 100 en el índice de pobreza), y los 2.041 nulos del índice no cambian.

#### 2.15.3 Recodificación de las variables de cuestionario [P12] [P13] [P14]

Las variables de cuestionario se recodifican con un diccionario explícito por variable. Cada diccionario incluye **solo los códigos válidos**: los códigos 7 (*se negó a responder*) y 9 (*no sabe*) no figuran, por lo que pasan a faltantes **[P12]**. Los códigos nuevos son numéricos y tienen significado: las binarias quedan en 0/1 y las ordinales con un código que respeta su orden.

| Variable | Tipo | Código original | Código nuevo | Significado del código nuevo |
|---|---|---|---|---|
| `education_level` | ordinal | 1 a 5 | 1 a 5 (sin cambios) | 1 menos de 9no grado · 2 9no a 11vo grado · 3 secundario completo · 4 superior incompleto o terciario · 5 universitario o más |
| `diabetes_diagnosed` → `diabetes_status` | ordinal | 2 / 3 / 1 | 0 / 1 / 2 | 0 no · 1 borderline · 2 sí |
| `insulin_use` | binaria | 1 / 2 | 1 / 0 | 1 usa insulina · 0 no |
| `diabetes_pills` | binaria | 1 / 2 | 1 / 0 | 1 toma pastillas para bajar la glucosa · 0 no |
| `ever_smoked` | binaria | 1 / 2 | 1 / 0 | 1 fumó al menos 100 cigarrillos en su vida · 0 no |
| las cinco anteriores | — | 7 / 9 | faltante | se negó a responder / no sabe |

- `diabetes_diagnosed` se renombra como `diabetes_status` porque el significado de sus códigos cambia: en el archivo el 1 es "sí" y el 3 "borderline", un código que no respeta el orden natural de la variable **[P14]**.
- `current_smoker` conserva sus códigos (1 todos los días, 2 algunos días, 3 actualmente no fuma): no tiene códigos 7/9 y se usa para construir `smoking_status` (sección 2.15.4).

In [ ]:
# diccionarios con los códigos válidos de cada variable: los códigos 7 y 9 no figuran y quedan como faltantes
diccionarios_codigos = {"education_level": {1: 1, 2: 2, 3: 3, 4: 4, 5: 5},
                        "diabetes_diagnosed": {2: 0, 3: 1, 1: 2},
                        "insulin_use": {1: 1, 2: 0},
                        "diabetes_pills": {1: 1, 2: 0},
                        "ever_smoked": {1: 1, 2: 0}}

# aplicamos cada diccionario y verificamos que los nulos nuevos sean exactamente los códigos 7 y 9
for columna, diccionario in diccionarios_codigos.items():
  nulos_antes = preprocessed_dataset[columna].isna().sum()
  codigos_sin_respuesta = preprocessed_dataset[columna].isin([7, 9]).sum()
  preprocessed_dataset[columna] = preprocessed_dataset[columna].map(diccionario)
  nulos_despues = preprocessed_dataset[columna].isna().sum()
  print(f"{columna}: nulos antes = {nulos_antes} + códigos 7/9 = {codigos_sin_respuesta} -> nulos después = {nulos_despues}")

# renombramos diabetes_diagnosed porque sus códigos ya no significan lo mismo que en el archivo
preprocessed_dataset = preprocessed_dataset.rename(columns={"diabetes_diagnosed": "diabetes_status"})

In [ ]:
# comparamos las frecuencias de cada código antes (archivo crudo) y después de recodificar
pares_columnas = [("education_level", "education_level"), ("diabetes_diagnosed", "diabetes_status"),
                  ("insulin_use", "insulin_use"), ("diabetes_pills", "diabetes_pills"), ("ever_smoked", "ever_smoked")]
for columna_original, columna_nueva in pares_columnas:
  print(f"{columna_original} (crudo): {raw_dataset[columna_original].value_counts().sort_index().to_dict()}")
  print(f"{columna_nueva} (recodificada): {preprocessed_dataset[columna_nueva].value_counts().sort_index().to_dict()}")

**Verificación.** En cada variable, los nulos nuevos son exactamente los códigos 7 y 9 (36 en total: 11 + 4 + 7 + 14), y la frecuencia de cada código válido se conserva con su código nuevo (por ejemplo, los 1.081 "sí" de diabetes pasan de 1 a 2).

#### 2.15.4 Variable de tabaquismo [P15]

`ever_smoked` (fumó al menos 100 cigarrillos en su vida) y `current_smoker` (frecuencia actual) se combinan en una sola variable ordinal, `smoking_status`. `current_smoker` solo existe para quienes fumaron alguna vez, así que usada sola deja afuera a quienes nunca fumaron; y `ever_smoked` sola no distingue a quienes dejaron de fumar.

| `smoking_status` | Condición | Significado |
|---|---|---|
| 0 | `ever_smoked` = 0 | nunca fumó |
| 1 | `ever_smoked` = 1 y `current_smoker` = 3 | ex fumador |
| 2 | `ever_smoked` = 1 y `current_smoker` = 1 o 2 | fumador actual (todos los días o algunos días) |
| faltante | resto | menor de 18 años (pregunta no realizada) o sin respuesta válida |

In [ ]:
# combinamos ever_smoked y current_smoker en una variable ordinal de tres niveles
def clasificar_tabaquismo(fumo_alguna_vez, frecuencia_actual):
  if fumo_alguna_vez == 0:
    return 0
  if fumo_alguna_vez == 1 and frecuencia_actual == 3:
    return 1
  if fumo_alguna_vez == 1 and (frecuencia_actual == 1 or frecuencia_actual == 2):
    return 2
  return float("nan")

preprocessed_dataset["smoking_status"] = preprocessed_dataset.apply(lambda fila: clasificar_tabaquismo(fila["ever_smoked"], fila["current_smoker"]), axis=1)

# verificamos cada nivel contra las columnas de origen
print(f"0 nunca fumó: {(preprocessed_dataset['smoking_status'] == 0).sum()} - ever_smoked == 0: {(preprocessed_dataset['ever_smoked'] == 0).sum()}")
print(f"1 ex fumador: {(preprocessed_dataset['smoking_status'] == 1).sum()} - current_smoker == 3: {(preprocessed_dataset['current_smoker'] == 3).sum()}")
print(f"2 fumador actual: {(preprocessed_dataset['smoking_status'] == 2).sum()} - current_smoker 1 o 2: {preprocessed_dataset['current_smoker'].isin([1, 2]).sum()}")
print(f"Sin dato: {preprocessed_dataset['smoking_status'].isna().sum()} - menores de 18: {(preprocessed_dataset['smoking_status'].isna() & (preprocessed_dataset['age'] < 18)).sum()} - adultos: {(preprocessed_dataset['smoking_status'].isna() & (preprocessed_dataset['age'] >= 18)).sum()}")

**Verificación.** Cada nivel coincide con su columna de origen: 4.878 personas nunca fumaron, 2.053 son ex fumadores y 1.190 fuman actualmente. Las 3.812 personas sin dato son 3.780 menores de 18 años, a quienes no se les hace la pregunta, y 32 adultos sin respuesta válida.

#### 2.15.5 Variables de función renal [P1] [P2] [P3] [P4] [P6]

**`ckd_stage`.** Conserva la clasificación del archivo, que se usa solo para describir la gravedad. La categoría `Unknown` no es un estadio sino la ausencia de eGFR **[P6]**: se reemplaza por un faltante. Las 70 personas sin ACR que el archivo clasifica como *No CKD* **[P3]** se dejan como están en esta columna; con el criterio clínico de esta sección quedan sin clasificar.

In [ ]:
# Unknown representa la ausencia de eGFR: lo reemplazamos por un faltante
print(f"ckd_stage == Unknown antes: {(preprocessed_dataset['ckd_stage'] == 'Unknown').sum()} - nulos antes: {preprocessed_dataset['ckd_stage'].isna().sum()}")
preprocessed_dataset["ckd_stage"] = preprocessed_dataset["ckd_stage"].replace("Unknown", pd.NA)
print(f"ckd_stage nulos después: {preprocessed_dataset['ckd_stage'].isna().sum()} - nulos de egfr: {preprocessed_dataset['egfr'].isna().sum()}")

**`ckd_present`.** Codifica como 1 tanto a los estadios de enfermedad como a `Unknown` **[P1]**, y hereda la regla de *Stage 2* **[P2]**. No se usa en ningún análisis, así que se elimina de la copia de trabajo (el archivo crudo la conserva).

In [ ]:
# eliminamos ckd_present de la copia de trabajo
preprocessed_dataset = preprocessed_dataset.drop("ckd_present", axis=1)
print(f"Columnas: {preprocessed_dataset.shape[1]}")

**Criterio clínico de enfermedad renal crónica (`ckd_clinical`).** Se define CKD como **eGFR < 60 mL/min/1,73 m² o ACR ≥ 30 mg/g**: los dos ejes que describe la metadata, sin la regla de *Stage 2* del archivo, que cuenta como enfermedad un filtrado levemente disminuido sin marcador de daño **[P2]** (sección 2.11.5). La columna vale 1 (con CKD), 0 (sin CKD) o queda como faltante cuando la persona no se puede clasificar:

| Situación | `ckd_clinical` | Motivo |
|---|---|---|
| menor de 18 años | faltante | el eGFR se calcula con una ecuación para adultos **[P4]** |
| adulto sin eGFR | faltante | sin el filtrado no se puede confirmar ni descartar la enfermedad |
| adulto con eGFR < 60 | 1 | cumple el criterio por el filtrado, tenga o no ACR |
| adulto con eGFR ≥ 60 y ACR ≥ 30 | 1 | cumple el criterio por el daño |
| adulto con eGFR ≥ 60 y ACR < 30 | 0 | no cumple ninguno de los dos ejes |
| adulto con eGFR ≥ 60 y sin ACR | faltante | no se puede suponer que el ACR es normal (es el supuesto que el archivo aplica en **[P3]**) |

Para clasificar a una persona se exige el eGFR: los adultos sin eGFR y con ACR ≥ 30 tampoco se clasifican, de modo que el mismo requisito vale para todos.

In [ ]:
# criterio clínico: CKD si eGFR < 60 o ACR >= 30; sin clasificar si falta el dato necesario
def clasificar_ckd(edad, egfr, acr):
  if edad < 18 or pd.isna(egfr):
    return float("nan")
  if egfr < 60:
    return 1
  if pd.isna(acr):
    return float("nan")
  if acr >= 30:
    return 1
  return 0

preprocessed_dataset["ckd_clinical"] = preprocessed_dataset.apply(lambda fila: clasificar_ckd(fila["age"], fila["egfr"], fila["albumin_creatinine_ratio"]), axis=1)
print(f"Con CKD (1): {(preprocessed_dataset['ckd_clinical'] == 1).sum()}")
print(f"Sin CKD (0): {(preprocessed_dataset['ckd_clinical'] == 0).sum()}")
print(f"Sin clasificar: {preprocessed_dataset['ckd_clinical'].isna().sum()}")

In [ ]:
# desglose de las personas sin clasificar según el motivo
sin_clasificar = preprocessed_dataset[preprocessed_dataset["ckd_clinical"].isna()]
adultos_sin_egfr = sin_clasificar[(sin_clasificar["age"] >= 18) & sin_clasificar["egfr"].isna()]
print(f"Sin clasificar: {sin_clasificar.shape[0]}")
print(f"  menores de 18 años: {(sin_clasificar['age'] < 18).sum()}")
print(f"  adultos con eGFR >= 60 y sin ACR: {((sin_clasificar['age'] >= 18) & sin_clasificar['egfr'].notna()).sum()}")
print(f"  adultos sin eGFR: {adultos_sin_egfr.shape[0]}")

# los adultos sin eGFR: quiénes no fueron al examen y quiénes fueron pero no tienen creatinina en sangre
cantidad_sin_examen = adultos_sin_egfr[columnas_examen].isna().all(axis=1).sum()
cantidad_con_examen = adultos_sin_egfr.shape[0] - cantidad_sin_examen
cantidad_con_acr = adultos_sin_egfr["albumin_creatinine_ratio"].notna().sum()
print(f"    sin ningún dato de examen ni de laboratorio: {cantidad_sin_examen}")
print(f"    con algún dato de examen pero sin creatinina: {cantidad_con_examen}")
print(f"      con ACR >= 30: {(adultos_sin_egfr['albumin_creatinine_ratio'] >= 30).sum()}")
print(f"      con ACR < 30: {(adultos_sin_egfr['albumin_creatinine_ratio'] < 30).sum()}")
print(f"      sin ACR: {cantidad_con_examen - cantidad_con_acr}")

In [ ]:
# comparamos a los adultos con y sin eGFR en edad y en proporción de diabetes diagnosticada
adultos_con_egfr_limpio = preprocessed_dataset[(preprocessed_dataset["age"] >= 18) & preprocessed_dataset["egfr"].notna()]
for nombre, grupo in [("con eGFR", adultos_con_egfr_limpio), ("sin eGFR", adultos_sin_egfr)]:
  diabetes_valida = grupo["diabetes_status"].dropna()
  porcentaje_diabetes = (diabetes_valida == 2).sum() / diabetes_valida.shape[0] * 100
  print(f"Adultos {nombre}: n = {grupo.shape[0]} - edad media = {grupo['age'].describe()['mean']:.1f} años - diabetes diagnosticada = {porcentaje_diabetes:.1f} %")

**Verificación.** De los 5.658 adultos con eGFR, 5.575 quedan clasificados: **972 con CKD (17,4 %)** y 4.603 sin CKD. Las 6.358 personas sin clasificar son 3.780 menores de 18 años, 83 adultos con eGFR ≥ 60 y sin ACR, y 2.495 adultos sin eGFR.

**Limitación: los adultos sin eGFR.** Los análisis de enfermedad renal se hacen sobre los 5.575 adultos clasificados, y quedan fuera **2.495 adultos sin eGFR** (el 30,6 % de los 8.153 adultos): 1.821 no concurrieron al examen y 674 lo hicieron pero no tienen creatinina en sangre (de ellos, 112 tienen ACR ≥ 30, 508 tienen ACR < 30 y 54 no tienen ACR).

- **Por qué no se clasifican:** sin el eGFR no se conoce el filtrado, así que no se puede confirmar ni descartar la enfermedad. Un ACR normal no alcanza para considerar sana a una persona.
- **Por qué no se imputan:** el eGFR define la variable que se estudia, y estimarlo equivaldría a asignar diagnósticos; con un 30 % de faltantes, los valores estimados tendrían un peso muy grande en los resultados. Tampoco se los agrupa en una categoría "sin dato", que no aportaría información a ningún test.
- **Qué implica excluirlos:** los resultados no se sesgan solo si la falta de eGFR no depende del estado renal, y eso no se puede comprobar con los datos. Si las personas con peor salud renal hubieran concurrido menos al examen, la proporción de CKD estaría subestimada. Como indicio a favor, los adultos con y sin eGFR tienen una edad media (52,7 y 51,0 años) y una proporción de diabetes diagnosticada (13,5 % y 12,4 %) similares. En contra, la falta de eGFR varía según el grupo étnico (28,8 % en blancos no hispanos contra 37,2 % en negros no hispanos, sección 2.11.4), lo que se tiene en cuenta en la hipótesis (a).

#### 2.15.6 Problemas que se conservan sin cambios

- **[P5] eGFR calculado con la edad censurada.** Las 298 personas de 80 años o más con eGFR se conservan: el valor no se puede recalcular sin la edad real. La sobreestimación (alrededor del 3 % a los 85 años) se declara como limitación.
- **[P8] [P10] Censuras de la edad (80) y del índice de pobreza (5).** Se conservan: excluir esos casos dejaría afuera a las 525 personas de 80 años o más y a los 2.149 hogares de mayores ingresos, y reemplazarlos inventaría valores. Por la censura, estas variables se resumen y comparan con medianas, cuartiles o tests basados en rangos, y no con la media.
- **[P26] Albúmina en orina de 0,21** (valor de relleno por debajo del límite de detección). Se conserva: no cambia la clasificación, porque las 488 personas tienen ACR < 30, y pasarla a faltante dejaría sin clasificar a 394 adultos con eGFR ≥ 60 que en ese eje no tienen daño.
- **[P16] [P17] Las 60 personas sin diabetes diagnosticada que toman pastillas para bajar la glucosa.** Se conservan sin cambios. `diabetes_status` registra un diagnóstico declarado, y la medicación sola no prueba ese diagnóstico: la interpretación más consistente es que se trate de personas con prediabetes con medicación oral (sección 2.10). Reclasificarlas como diabéticas supondría un diagnóstico que no declararon.
- **[P15] Faltantes por diseño de `insulin_use` y `diabetes_pills`.** No se imputan: estas variables solo se analizan dentro del grupo al que se le hizo la pregunta.
- **[P22] Mezcla de chicos y adultos.** Se resuelve con las poblaciones de análisis (sección 2.15.7), no modificando datos.
- **[P23] Valores extremos de las variables renales.** Se conservan (criterios generales de esta sección).
- **[P24] Filas iguales salvo el identificador.** Son personas distintas: no se eliminan.

#### 2.15.7 Poblaciones de análisis

Dos selecciones se usan en más de un análisis y se definen una sola vez. Cada hipótesis agrega sus propios filtros (por ejemplo, los casos completos de las variables que usa) e informa el tamaño de la muestra resultante.

| Selección | Definición | Uso |
|---|---|---|
| `poblacion_adultos` | personas de 18 años o más | análisis que no usan la clasificación renal; evita mezclar chicos y adultos **[P22]** |
| `poblacion_ckd` | adultos con `ckd_clinical` clasificada | todos los análisis de enfermedad renal |

In [ ]:
# poblaciones de análisis comunes a las hipótesis
poblacion_adultos = preprocessed_dataset[preprocessed_dataset["age"] >= 18].copy()
poblacion_ckd = preprocessed_dataset[preprocessed_dataset["ckd_clinical"].notna()].copy()
casos_ckd = (poblacion_ckd["ckd_clinical"] == 1).sum()
print(f"poblacion_adultos: {poblacion_adultos.shape[0]} personas")
print(f"poblacion_ckd: {poblacion_ckd.shape[0]} adultos clasificados (edad mínima: {poblacion_ckd['age'].min():.0f} años)")
print(f"  con CKD: {casos_ckd} ({casos_ckd / poblacion_ckd.shape[0] * 100:.1f} %)")

#### 2.15.8 Resultado de la limpieza

| Problema | Acción | Afectados |
|---|---|---|
| [P7] | `age`: valor minúsculo → 0 | 189 valores |
| [P9] | `poverty_income_ratio`: valor minúsculo → 0 | 100 valores |
| [P12] | códigos 7 y 9 → faltante | 36 valores (11 + 4 + 7 + 14) |
| [P13] [P14] | recodificación de cinco variables de cuestionario; `diabetes_diagnosed` → `diabetes_status` | 5 columnas |
| [P15] | variable nueva `smoking_status` (3 niveles) | 8.121 personas con dato |
| [P6] | `ckd_stage`: `Unknown` → faltante | 5.607 valores |
| [P1] | se elimina `ckd_present` | 1 columna |
| [P2] [P3] [P4] | variable nueva `ckd_clinical` (criterio clínico, solo adultos) | 5.575 adultos clasificados, 972 con CKD |
| resto | sin cambios, con su justificación (sección 2.15.6) | — |

La copia de trabajo conserva las 11.933 filas y tiene 30 columnas: las 29 del archivo, menos `ckd_present`, más `smoking_status` y `ckd_clinical`.

In [ ]:
# estructura final de la copia de trabajo
preprocessed_dataset.info()

In [ ]:
# guardamos la versión 1 del dataset limpio como punto de control
preprocessed_dataset.to_pickle("preprocessed_dataset_v1.pkl")
print(f"preprocessed_dataset_v1.pkl: {preprocessed_dataset.shape[0]} filas y {preprocessed_dataset.shape[1]} columnas")

---
## 3. Hipótesis y resolución

Criterios comunes a todas las hipótesis:

- **Nivel de significancia:** α = 0,05. En las salidas, los p-valores menores a 0,001 aparecen redondeados a 0,000.
- **Población:** las hipótesis sobre enfermedad renal usan `poblacion_ckd`, los 5.575 adultos clasificados con el criterio clínico (sección 2.15.7), y la variable `ckd_clinical`. Cuando una hipótesis necesita además otra variable, informa cuántas personas quedan con ese dato.
- **CKF y CKD.** Las hipótesis (a) y (b) del enunciado mencionan *CKF* y la (c), *CKD*. Se interpretan como el mismo concepto, enfermedad renal crónica (CKD). La lectura alternativa, CKF como falla renal (*Stage 5*), dejaría solo 11 casos, demasiado pocos para cualquier análisis.
- **Comparación de una variable cuantitativa entre dos grupos** (secciones 3.2 y 3.5). Se sigue siempre el mismo procedimiento:
  1. **Gráfico:** boxplot por grupo.
  2. **Normalidad de cada grupo:** test de Shapiro-Wilk (H₀: la variable sigue una distribución normal) y QQ plot. Con grupos de cientos o miles de personas, Shapiro-Wilk rechaza ante desvíos mínimos de la normalidad; a la vez, con más de 30 observaciones por grupo el test t es robusto a la falta de normalidad. Por eso la normalidad se informa, pero no decide sola el test.
  3. **Igualdad de varianzas:** test de Levene (H₀: las varianzas de los dos grupos son iguales). La robustez del test t no cubre la diferencia de varianzas: si Levene rechaza, se usa el test t de Welch, que no supone varianzas iguales.
  4. **Test principal:** test t para muestras independientes, con la corrección de Welch si corresponde. Compara las medias, cuya diferencia se expresa en las unidades de la variable.
  5. **Control de robustez:** un test no paramétrico, que no supone normalidad: Mann-Whitney U si las varianzas pueden considerarse iguales, o Kruskal-Wallis en caso contrario. Si el test principal y el de control coinciden, la conclusión no depende del supuesto de normalidad.
- **Modelos de regresión** (secciones 3.3 y 3.6). Se usan los casos completos de las variables de cada modelo y se informa cuántos adultos quedan. Las variables cualitativas se codifican como columnas 0/1. Los predictores cuantitativos se estandarizan (media 0 y desvío 1), porque tienen rangos muy distintos y así sus coeficientes se pueden comparar; cada uno se informa además en las unidades originales, dividiéndolo por el desvío del predictor. Las columnas 0/1 no se estandarizan: su coeficiente ya expresa la diferencia entre la categoría y la de referencia, y estandarizarlas haría depender esa lectura de la proporción de cada categoría. La estandarización no cambia los p-valores ni el ajuste del modelo. Cada coeficiente se informa con su intervalo de confianza del 95 %. La multicolinealidad se evalúa con las correlaciones entre los predictores y con el factor de inflación de la varianza (VIF), calculado con la constante del modelo: vale 1 cuando un predictor no se relaciona linealmente con los demás y crece a medida que esa relación aumenta.

### 3.1 Hipótesis propia univariada: prevalencia de enfermedad renal crónica

#### 3.1.1 Definición

**Motivación.** El CDC estima que alrededor del 14 % de los adultos de EE.UU. tiene enfermedad renal crónica [10]. Antes de estudiar con qué factores se asocia la enfermedad, interesa saber si la proporción de adultos con CKD de esta muestra, definida con el criterio clínico de la sección 2.15.5, es compatible con esa estimación.

**Enunciado.** La proporción de adultos con enfermedad renal crónica en la muestra difiere del 14 % estimado por el CDC para los adultos de EE.UU.

**Utilidad.**
- Cuantifica la frecuencia de la enfermedad en la población que usan los análisis de enfermedad renal.
- Contrasta el criterio clínico con una estimación externa (la regla del archivo marcaba como CKD al 46,8 % de los adultos con eGFR, sección 2.11.5).
- Indica si las proporciones de la muestra pueden leerse como proporciones de la población, lo que condiciona la interpretación de las hipótesis siguientes.

#### 3.1.2 Estrategia

- **Variable:** `ckd_clinical`, cualitativa binaria (1 con CKD, 0 sin CKD), en `poblacion_ckd`: los 5.575 adultos clasificados (sección 2.15.7).
- **Hipótesis:** sea p la proporción de adultos con CKD.
  - H₀: p = 0,14
  - H₁: p ≠ 0,14 (bilateral: el enunciado no fija una dirección)
  - Nivel de significancia: α = 0,05
- **Test: Z para una proporción.** Es el test de hipótesis sobre una proporción. Mide a cuántos errores estándar está la proporción observada del valor de referencia:

  `z = (p̂ − p₀) / √(p₀ · (1 − p₀) / n)`

  donde `p̂` es la proporción de la muestra, `p₀` = 0,14 el valor de referencia y `n` el tamaño de la muestra.
  - **Por qué Z y no t.** En una variable binaria, la varianza queda determinada por la propia proporción: vale `p · (1 − p)`. Si H₀ fuera cierta, el desvío sería `√(0,14 · 0,86)`, un valor conocido que no hace falta estimar con la muestra. Por eso el estadístico se compara con la distribución normal estándar, de ahí el nombre Z. El test t, en cambio, se usa cuando el desvío se estima con los datos (como en el test sobre una media de una variable cuantitativa), y por eso se compara con una t de Student. Es la misma construcción que el test Z para dos proporciones, aplicada a un solo grupo contra un valor fijo.
  - Un test t sobre la variable 0/1 llevaría a la misma conclusión con una muestra de este tamaño, pero supone una variable cuantitativa con distribución aproximadamente normal, que no es el caso de una variable binaria.
  - El p-valor bilateral es la probabilidad, si H₀ fuera cierta, de obtener un `|z|` al menos tan grande como el observado.
- **Supuestos:**
  - la variable es binaria;
  - las observaciones son independientes (una fila por persona; ver la discusión sobre el diseño muestral);
  - la aproximación normal es válida: `n · p₀` y `n · (1 − p₀)` son al menos 10.
- **Magnitud:** además del test, se calcula el intervalo de confianza del 95 % de la proporción, `p̂ ± 1,96 · √(p̂ · (1 − p̂) / n)`, para describir cuánto difiere y no solo si difiere.

#### 3.1.3 Resultados y discusión

In [ ]:
# proporción de adultos con CKD en la muestra, junto a la estimación del CDC usada como referencia
casos_ckd = (poblacion_ckd["ckd_clinical"] == 1).sum()
porcentaje_muestra = casos_ckd / poblacion_ckd.shape[0] * 100

plt.figure(figsize=(6, 5))
plt.bar(["Muestra (criterio clínico)", "Estimación del CDC"], [porcentaje_muestra, 14], color=["steelblue", "gray"])
plt.ylabel("% de adultos con CKD")
plt.title("Prevalencia de enfermedad renal crónica en adultos")
plt.show()

In [ ]:
# supuesto de la aproximación normal: n · p0 y n · (1 − p0) tienen que ser al menos 10
n = poblacion_ckd.shape[0]
p0 = 0.14
print(f"n = {n}")
print(f"n · p0 = {n * p0:.1f} - n · (1 − p0) = {n * (1 - p0):.1f}")

In [ ]:
# test Z para una proporción: diferencia entre la proporción observada y la de referencia, dividida por el error estándar bajo H0
p_muestral = casos_ckd / n
error_estandar_h0 = np.sqrt(p0 * (1 - p0) / n)
z_statistic = (p_muestral - p0) / error_estandar_h0
p_value = 2 * (1 - stats.norm.cdf(abs(z_statistic)))
print(f"Casos: {casos_ckd} de {n} - proporción muestral = {p_muestral:.4f}")
print(f"Error estándar bajo H0 = {error_estandar_h0:.4f}")
print(f"Test Z para una proporción: Estadístico={z_statistic:.3f}, p-valor={p_value:.3f}")

# intervalo de confianza del 95 % para la proporción
error_estandar_muestral = np.sqrt(p_muestral * (1 - p_muestral) / n)
limite_inferior = p_muestral - 1.96 * error_estandar_muestral
limite_superior = p_muestral + 1.96 * error_estandar_muestral
print(f"IC 95 %: [{limite_inferior * 100:.1f} %; {limite_superior * 100:.1f} %]")

# interpretación de los resultados
alpha = 0.05  # nivel de significancia
if p_value > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("La proporción de adultos con CKD en la muestra no difiere significativamente del 14 %.")
else:
  print("Se rechaza la hipótesis nula.")
  print("La proporción de adultos con CKD en la muestra difiere significativamente del 14 %.")

**Resultados**

- En la muestra hay 972 adultos con CKD sobre 5.575 clasificados: **17,4 %**, con un intervalo de confianza del 95 % de **16,4 % a 18,4 %**.
- Los supuestos se cumplen: `n · p₀` = 780,5 y `n · (1 − p₀)` = 4.794,5, muy por encima de 10.
- El estadístico es z = 7,39, con un p-valor menor a 0,001. El signo positivo de z indica que la proporción muestral es mayor que la de referencia.

**Conclusión: se rechaza H₀. La proporción de adultos con enfermedad renal crónica en la muestra (17,4 %) difiere significativamente del 14 % estimado por el CDC y es mayor: el intervalo de confianza del 95 % queda completo por encima de ese valor. La hipótesis se confirma.**

**Discusión**

- **Magnitud.** La diferencia es de unos 3,4 puntos porcentuales. Con 5.575 personas el test detecta diferencias mucho menores, así que el resultado se interpreta con el intervalo: la prevalencia muestral está entre 2,4 y 4,4 puntos por encima de la referencia.
- **La diferencia no implica que la prevalencia real de EE.UU. sea otra.** El resultado es coherente con la forma en que se construyó la muestra:
  - el archivo no tiene pesos muestrales **[P25]** y el ciclo sobremuestrea por grupo de edad (sección 1.1). La enfermedad renal es mucho más frecuente en edades avanzadas: el eGFR menor a 60 pasa de 0,1 % entre los 18 y 39 años a 16,9 % a partir de los 60 (sección 2.13.2). Una muestra con más personas mayores que la población tiene, por eso, una prevalencia mayor;
  - la estimación del CDC usa los pesos muestrales y excluye el *Stage 5* (sección 2.11.5), por lo que las dos cifras no se calculan igual.
- **Limitaciones.**
  - Quedan fuera los 2.495 adultos sin eGFR (sección 2.15.5). Si la falta de eGFR fuera más frecuente entre personas con peor salud renal, la prevalencia de la muestra estaría subestimada, y la diferencia con el 14 % sería todavía mayor.
  - El test supone observaciones independientes, como en un muestreo aleatorio simple. NHANES usa un diseño muestral complejo, que selecciona a los participantes por etapas (zonas y hogares; sección 1.1), así que las observaciones no son completamente independientes.
  - Con una sola medición por persona no se confirma la persistencia de tres meses que exige la definición de CKD (sección 1.2): algunos casos pueden ser alteraciones transitorias.
- **Consecuencia para el resto del trabajo.** Las proporciones de la muestra no se generalizan directamente a la población de EE.UU.: las hipótesis siguientes comparan grupos dentro de la muestra. Y como la edad explica buena parte de la prevalencia, se la tiene en cuenta como posible confusor.

### 3.2 Hipótesis propia bivariada: ácido úrico y enfermedad renal crónica

#### 3.2.1 Definición

**Motivación.** La metadata describe al ácido úrico como un producto del metabolismo de las purinas que *"se elimina en parte por los riñones"* [5]. Si el filtrado renal disminuye, es esperable que el ácido úrico se acumule en la sangre.

**Enunciado.** El nivel de ácido úrico en sangre difiere entre los adultos con y sin enfermedad renal crónica.

**Utilidad.** Complementa a las hipótesis del enunciado, centradas en factores de riesgo, con una variable metabólica vinculada a la función renal. El ácido úrico no interviene en la definición de CKD (el criterio usa el eGFR y el ACR), por lo que una asociación no está dada por construcción.

#### 3.2.2 Estrategia

- **Variables:** `uric_acid`, cuantitativa continua (mg/dL), y `ckd_clinical`, binaria, en `poblacion_ckd`. Se comparan dos grupos independientes: adultos con CKD y sin CKD.
- **Hipótesis:** sean μ₁ la media de ácido úrico de los adultos con CKD y μ₀ la de los adultos sin CKD.
  - H₀: μ₁ = μ₀
  - H₁: μ₁ ≠ μ₀ (bilateral)
  - Nivel de significancia: α = 0,05
- **Procedimiento:** la comparación de dos grupos descrita al inicio de la sección 3 (boxplot, Shapiro-Wilk y QQ plot, Levene, test t con la corrección de Welch si corresponde, y test no paramétrico como control).
- **Posibles confusores:** el sexo y la edad, que se describen en la discusión.

#### 3.2.3 Resultados y discusión

In [ ]:
# ácido úrico de cada grupo, sin los nulos
acido_urico_ckd = poblacion_ckd[poblacion_ckd["ckd_clinical"] == 1]["uric_acid"].dropna()
acido_urico_sin_ckd = poblacion_ckd[poblacion_ckd["ckd_clinical"] == 0]["uric_acid"].dropna()
for nombre, grupo in [("Con CKD", acido_urico_ckd), ("Sin CKD", acido_urico_sin_ckd)]:
  estadisticas = grupo.describe()
  print(f"{nombre}: n = {estadisticas['count']:.0f} - media = {estadisticas['mean']:.2f} - mediana = {estadisticas['50%']:.2f} - desvío = {estadisticas['std']:.2f} mg/dL")

In [ ]:
# boxplot del ácido úrico según la enfermedad renal
sns.boxplot(x="ckd_clinical", y="uric_acid", data=poblacion_ckd, notch=True)
plt.title("Ácido úrico en sangre según la enfermedad renal crónica")
plt.xlabel("Enfermedad renal crónica")
plt.xticks(ticks=[0, 1], labels=["Sin CKD", "Con CKD"])
plt.ylabel("Ácido úrico (mg/dL)")
plt.show()

In [ ]:
# test de Shapiro-Wilk en cada grupo
stat, p = stats.shapiro(acido_urico_ckd)
print(f"Test de Shapiro-Wilk para adultos con CKD: Estadístico={stat:.3f}, p-valor={p:.3f}")

stat, p = stats.shapiro(acido_urico_sin_ckd)
print(f"Test de Shapiro-Wilk para adultos sin CKD: Estadístico={stat:.3f}, p-valor={p:.3f}")

In [ ]:
# QQ plot para adultos con CKD
stats.probplot(acido_urico_ckd, dist="norm", plot=plt)
plt.title("QQ Plot del ácido úrico en adultos con CKD")
plt.show()

# QQ plot para adultos sin CKD
stats.probplot(acido_urico_sin_ckd, dist="norm", plot=plt)
plt.title("QQ Plot del ácido úrico en adultos sin CKD")
plt.show()

In [ ]:
# test de Levene: H0 = las varianzas de los dos grupos son iguales
stat_levene, p_levene = stats.levene(acido_urico_ckd, acido_urico_sin_ckd)
print(f"Test de Levene para uric_acid: Estadístico={stat_levene:.3f}, p-valor={p_levene:.3f}")

alpha = 0.05  # nivel de significancia
if p_levene > alpha:
  varianzas_iguales = True
  print("No hay evidencia para rechazar la igualdad de varianzas: se usa el test t para muestras independientes.")
else:
  varianzas_iguales = False
  print("Se rechaza la igualdad de varianzas: se usa el test t de Welch.")

In [ ]:
# test principal: test t para muestras independientes (con la corrección de Welch si las varianzas difieren)
t_statistic, p_value = stats.ttest_ind(acido_urico_ckd, acido_urico_sin_ckd, equal_var=varianzas_iguales)
print(f"Test t: Estadístico={t_statistic:.3f}, p-valor={p_value:.3f}")
print(f"Diferencia de medias (con CKD - sin CKD): {acido_urico_ckd.describe()['mean'] - acido_urico_sin_ckd.describe()['mean']:.2f} mg/dL")

# interpretación de los resultados
if p_value > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("No hay una diferencia significativa en el ácido úrico entre adultos con y sin CKD.")
else:
  print("Se rechaza la hipótesis nula.")
  print("Existe una diferencia significativa en el ácido úrico entre adultos con y sin CKD.")

In [ ]:
# control de robustez: test no paramétrico (Mann-Whitney U si las varianzas son iguales, Kruskal-Wallis si no)
if varianzas_iguales:
  stat, p = stats.mannwhitneyu(acido_urico_ckd, acido_urico_sin_ckd)
  print(f"Test de Mann-Whitney U para uric_acid: Estadístico={stat:.3f}, p-valor={p:.3f}")
else:
  stat, p = stats.kruskal(acido_urico_ckd, acido_urico_sin_ckd)
  print(f"Test de Kruskal-Wallis para uric_acid: Estadístico={stat:.3f}, p-valor={p:.3f}")

if p > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
else:
  print("Se rechaza la hipótesis nula: coincide con el test principal.")

In [ ]:
# posibles confusores: ácido úrico y proporción con CKD según el sexo, y edad de cada grupo
for sexo in ["Female", "Male"]:
  grupo = poblacion_ckd[poblacion_ckd["gender"] == sexo]
  mediana_ckd = grupo[grupo["ckd_clinical"] == 1]["uric_acid"].describe()["50%"]
  mediana_sin_ckd = grupo[grupo["ckd_clinical"] == 0]["uric_acid"].describe()["50%"]
  porcentaje_ckd_sexo = (grupo["ckd_clinical"] == 1).sum() / grupo.shape[0] * 100
  print(f"{sexo}: mediana de ácido úrico = {grupo['uric_acid'].describe()['50%']:.1f} mg/dL (con CKD {mediana_ckd:.1f} - sin CKD {mediana_sin_ckd:.1f}) - con CKD = {porcentaje_ckd_sexo:.1f} %")

print(f"Edad mediana: con CKD = {poblacion_ckd[poblacion_ckd['ckd_clinical'] == 1]['age'].describe()['50%']:.0f} años - sin CKD = {poblacion_ckd[poblacion_ckd['ckd_clinical'] == 0]['age'].describe()['50%']:.0f} años")

**Resultados**

- **Descripción:** los 972 adultos con CKD tienen una media de 5,65 mg/dL y una mediana de 5,5; los 4.603 sin CKD, una media de 5,05 y una mediana de 5,0. En el boxplot, la caja del grupo con CKD está corrida hacia arriba.
- **Normalidad:** Shapiro-Wilk rechaza en los dos grupos (p < 0,001), aunque el estadístico es cercano a 1 (0,984 en ambos); los QQ plots muestran que el apartamiento se concentra en la cola derecha (valores altos). Con grupos de 972 y 4.603 personas, el test t es robusto a esta falta de normalidad.
- **Varianzas:** Levene rechaza la igualdad (estadístico 42,55, p < 0,001): el desvío es de 1,62 mg/dL en el grupo con CKD y de 1,35 en el grupo sin CKD. Por eso se usa el test t de Welch, con Kruskal-Wallis como control.
- **Test t de Welch:** t = 10,81, p < 0,001. La diferencia de medias es de 0,60 mg/dL, a favor del grupo con CKD.
- **Kruskal-Wallis:** estadístico 116,35, p < 0,001: coincide con el test principal.

**Conclusión: se rechaza H₀. El nivel de ácido úrico en sangre difiere significativamente entre los adultos con y sin enfermedad renal crónica, y es mayor en los adultos con CKD (media de 5,65 contra 5,05 mg/dL). La hipótesis se confirma.**

**Discusión**

- **Magnitud.** La diferencia es de 0,6 mg/dL en la media y de 0,5 en la mediana: moderada frente a la variabilidad dentro de cada grupo (desvíos de 1,35 y 1,62 mg/dL). Con muestras de este tamaño, la significancia se interpreta junto con esa magnitud.
- **Interpretación.** El resultado es coherente con que el ácido úrico se elimine en parte por los riñones [5] y se acumule cuando la función renal disminuye. Con una sola medición por persona no se puede establecer qué ocurrió primero, así que se trata de una asociación y no de una relación causal.
- **Sexo.** Los hombres tienen más ácido úrico que las mujeres (mediana de 5,7 contra 4,5 mg/dL) y algo más de CKD (18,5 % contra 16,6 %), por lo que parte de la diferencia podría deberse al sexo. Sin embargo, la diferencia se mantiene dentro de cada sexo: mediana de 5,1 contra 4,4 mg/dL en las mujeres y de 6,0 contra 5,7 en los hombres.
- **Edad.** Los adultos con CKD son mayores (mediana de 68 contra 52 años). El análisis conjunto de la CKD con la edad, el sexo y el BMI se hace en la sección 3.3.

### 3.3 Hipótesis propia multivariada: ácido úrico ajustado por edad, sexo y BMI

#### 3.3.1 Definición

**Motivación.** En la sección 3.2, los adultos con CKD tienen en promedio 0,60 mg/dL más de ácido úrico que los adultos sin CKD. Esa comparación no considera otras diferencias entre los grupos que también pueden vincularse con el ácido úrico: los hombres tienen más ácido úrico que las mujeres y una proporción algo mayor de CKD, y los adultos con CKD son mayores (sección 3.2). El BMI se incluye porque la metadata menciona el exceso de peso entre los factores relacionados con el deterioro renal [5]: si además se vinculara con el ácido úrico, podría explicar parte de la diferencia.

**Enunciado.** La diferencia de ácido úrico entre los adultos con y sin enfermedad renal crónica se mantiene al comparar adultos de igual edad, sexo y BMI.

**Utilidad.**
- Distingue si el vínculo de la sección 3.2 es propio de la enfermedad renal o refleja diferencias de edad, sexo o peso entre los grupos.
- Cuantifica la diferencia que queda al tener en cuenta esas variables y muestra en qué grupos aparece.

#### 3.3.2 Estrategia

- **Población:** los adultos de `poblacion_ckd` con datos de ácido úrico, edad, sexo y BMI (casos completos).
- **Técnica: regresión lineal múltiple**, ajustada por mínimos cuadrados.

| Variable | Rol | Tipo | Codificación en el modelo |
|---|---|---|---|
| `uric_acid` | respuesta | cuantitativa continua (mg/dL) | sin transformar |
| `ckd_clinical` | predictor | binaria | 0/1 (1 = con CKD) |
| `age` | predictor | cuantitativa continua (años) | estandarizada |
| `gender` | predictor | binaria | `hombre`: 1 hombre, 0 mujer |
| `bmi` | predictor | cuantitativa continua (kg/m²) | estandarizada |

- **Modelo:**

  `uric_acid = β₀ + β₁ · ckd_clinical + β₂ · age + β₃ · hombre + β₄ · bmi + ε`

  Cada coeficiente mide el cambio esperado en el ácido úrico por unidad de su predictor, manteniendo constantes los demás. Como `ckd_clinical` queda como variable 0/1, β₁ es directamente la diferencia de ácido úrico, en mg/dL, entre adultos con y sin CKD de igual edad, sexo y BMI.
- **Hipótesis sobre el coeficiente de la CKD:**
  - H₀: β₁ = 0 (a igualdad de edad, sexo y BMI, el ácido úrico no difiere según la CKD)
  - H₁: β₁ ≠ 0 (bilateral)
  - Nivel de significancia: α = 0,05
  - Se usa el test t del coeficiente. Se informan además el test F (significancia global del modelo) y el R² (proporción de la variabilidad del ácido úrico que explica el modelo).
- **Supuestos**, evaluados sobre los residuos (diferencia entre el valor observado y el ajustado por el modelo):
  - normalidad: QQ plot y tests de Omnibus y Jarque-Bera (H₀: los residuos siguen una distribución normal);
  - homocedasticidad: gráfico de valores ajustados contra residuos, que debería mostrar una banda de ancho constante alrededor de 0;
  - independencia: estadístico de Durbin-Watson, que debería estar cerca de 2;
  - linealidad: residuos contra cada predictor cuantitativo, sin patrones curvos;
  - ausencia de multicolinealidad: correlaciones entre los predictores y VIF.
- **Control de robustez:** con miles de observaciones, los tests de normalidad rechazan ante desvíos pequeños. Como control que no depende de ese supuesto, se compara el ácido úrico de los adultos con y sin CKD con el test de Kruskal-Wallis (el control de la sección 3.2) dentro de cada combinación de sexo y grupo de edad (18 a 39, 40 a 59 y 60 años o más, los grupos de la sección 3.4).

#### 3.3.3 Resultados y discusión

In [ ]:
# casos completos de las variables del modelo
datos_acido_urico = poblacion_ckd.dropna(subset=["uric_acid", "age", "gender", "bmi"]).copy()
print(f"Adultos con todas las variables del modelo: {datos_acido_urico.shape[0]} de {poblacion_ckd.shape[0]} (con CKD: {(datos_acido_urico['ckd_clinical'] == 1).sum()})")

# codificamos el sexo como variable binaria: 1 si es hombre, 0 si es mujer
datos_acido_urico["hombre"] = datos_acido_urico["gender"].apply(lambda x: 1 if x == "Male" else 0)
print(f"Verificación: {datos_acido_urico['hombre'].sum()} hombres codificados con 1 (Male en gender: {(datos_acido_urico['gender'] == 'Male').sum()})")

In [ ]:
# correlación entre la variable respuesta y los predictores
matriz_correlacion_acido_urico = datos_acido_urico[["uric_acid", "ckd_clinical", "age", "hombre", "bmi"]].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(matriz_correlacion_acido_urico, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre el ácido úrico y los predictores")
plt.show()

In [ ]:
# variable respuesta y predictores
y_acido_urico = datos_acido_urico["uric_acid"]
X_acido_urico = datos_acido_urico[["ckd_clinical", "age", "hombre", "bmi"]]

# estandarizamos los predictores cuantitativos (media 0 y desvío 1); las variables 0/1 quedan como están
columnas_cuantitativas_acido_urico = ["age", "bmi"]
scaler_acido_urico = StandardScaler()
X_acido_urico_scaled_df = X_acido_urico.copy()
X_acido_urico_scaled_df[columnas_cuantitativas_acido_urico] = scaler_acido_urico.fit_transform(X_acido_urico[columnas_cuantitativas_acido_urico])

# verificamos la estandarización: medias cercanas a 0 y desvíos cercanos a 1
print(X_acido_urico_scaled_df[columnas_cuantitativas_acido_urico].describe().loc[["mean", "std"]].round(3))

In [ ]:
# agregamos una constante como característica, para estimar la ordenada al origen, y ajustamos el modelo
X_acido_urico_scaled_df_ = sm.add_constant(X_acido_urico_scaled_df)
modelo_acido_urico = sm.OLS(y_acido_urico, X_acido_urico_scaled_df_).fit()
print(modelo_acido_urico.summary())

In [ ]:
# desvío de cada predictor estandarizado, para volver a las unidades originales
desvios_acido_urico = pd.Series(np.sqrt(scaler_acido_urico.var_), index=columnas_cuantitativas_acido_urico)
print(desvios_acido_urico.round(2))

# coeficientes con su intervalo de confianza del 95 %; en unidades originales, los de los predictores estandarizados
# se dividen por su desvío y los de las variables 0/1 ya expresan la diferencia entre las dos categorías
divisores_acido_urico = pd.Series(1.0, index=X_acido_urico.columns)
divisores_acido_urico[columnas_cuantitativas_acido_urico] = desvios_acido_urico
coeficientes_acido_urico = modelo_acido_urico.params[X_acido_urico.columns]
intervalos_acido_urico = modelo_acido_urico.conf_int().loc[X_acido_urico.columns]
tabla_coeficientes_acido_urico = pd.DataFrame({"coeficiente del modelo": coeficientes_acido_urico,
                                               "IC 95 % inferior": intervalos_acido_urico[0],
                                               "IC 95 % superior": intervalos_acido_urico[1],
                                               "coeficiente en unidades originales": coeficientes_acido_urico / divisores_acido_urico,
                                               "p-valor": modelo_acido_urico.pvalues[X_acido_urico.columns]})
tabla_coeficientes_acido_urico.round(4)

In [ ]:
# diferencia de ácido úrico asociada a la CKD, en mg/dL: sin ajustar (diferencia de medias) y ajustada por edad, sexo y BMI
media_con_ckd = y_acido_urico[datos_acido_urico["ckd_clinical"] == 1].describe()["mean"]
media_sin_ckd = y_acido_urico[datos_acido_urico["ckd_clinical"] == 0].describe()["mean"]
intervalo_ckd = modelo_acido_urico.conf_int().loc["ckd_clinical"]
print(f"Diferencia sin ajustar (diferencia de medias): {media_con_ckd - media_sin_ckd:.2f} mg/dL")
print(f"Diferencia ajustada por edad, sexo y BMI: {coeficientes_acido_urico['ckd_clinical']:.2f} mg/dL (IC 95 %: {intervalo_ckd.iloc[0]:.2f} a {intervalo_ckd.iloc[1]:.2f})")

# test t del coeficiente de la CKD: H0 = a igualdad de edad, sexo y BMI, el ácido úrico no difiere según la CKD
p_value_ckd = modelo_acido_urico.pvalues["ckd_clinical"]
print(f"Test t del coeficiente de ckd_clinical: Estadístico={modelo_acido_urico.tvalues['ckd_clinical']:.3f}, p-valor={p_value_ckd:.3f}")

# interpretación de los resultados
alpha = 0.05  # nivel de significancia
if p_value_ckd > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("A igualdad de edad, sexo y BMI, no hay una diferencia significativa de ácido úrico según la CKD.")
else:
  print("Se rechaza la hipótesis nula.")
  print("A igualdad de edad, sexo y BMI, el ácido úrico difiere significativamente según la CKD.")

In [ ]:
# supuesto de normalidad de los residuos: QQ plot (los tests de Omnibus y Jarque-Bera están en el resumen del modelo)
stats.probplot(modelo_acido_urico.resid, dist="norm", plot=plt)
plt.title("QQ Plot de los residuos del modelo")
plt.show()

In [ ]:
# supuesto de homocedasticidad: valores ajustados contra residuos
plt.scatter(modelo_acido_urico.fittedvalues, modelo_acido_urico.resid, alpha=0.3)
plt.xlabel("Valores ajustados (mg/dL)")
plt.ylabel("Residuos (mg/dL)")
plt.title("Valores ajustados vs. residuos")
plt.axhline(0, color="red", linestyle="--")
plt.show()

# desvío de los residuos en cada grupo de la CKD
for valor, nombre in [(1, "con CKD"), (0, "sin CKD")]:
  print(f"Desvío de los residuos {nombre}: {modelo_acido_urico.resid[datos_acido_urico['ckd_clinical'] == valor].describe()['std']:.2f} mg/dL")

In [ ]:
# supuesto de linealidad: residuos contra cada predictor cuantitativo
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plt.subplots_adjust(wspace=0.3)
for eje, variable, etiqueta, titulo in [(axes[0], "age", "Edad (años)", "Residuos según la edad"),
                                        (axes[1], "bmi", "BMI (kg/m²)", "Residuos según el BMI")]:
  eje.scatter(datos_acido_urico[variable], modelo_acido_urico.resid, alpha=0.3)
  eje.axhline(0, color="red", linestyle="--")
  eje.set_xlabel(etiqueta)
  eje.set_ylabel("Residuos (mg/dL)")
  eje.set_title(titulo)
plt.show()

In [ ]:
# supuesto de ausencia de multicolinealidad: factor de inflación de la varianza (VIF) de cada columna del modelo, con la constante incluida
valores_vif_acido_urico = []
for i in range(X_acido_urico_scaled_df_.shape[1]):
  valores_vif_acido_urico.append(variance_inflation_factor(X_acido_urico_scaled_df_.values, i))
tabla_vif_acido_urico = pd.DataFrame({"variable": X_acido_urico_scaled_df_.columns, "VIF": valores_vif_acido_urico})
tabla_vif_acido_urico.round(3)

In [ ]:
# control de robustez: Kruskal-Wallis entre adultos con y sin CKD dentro de cada sexo y grupo de edad
for sexo, nombre_sexo in [("Female", "Mujeres"), ("Male", "Hombres")]:
  for edad_minima, edad_maxima in [(18, 40), (40, 60), (60, 81)]:
    grupo = datos_acido_urico[(datos_acido_urico["gender"] == sexo) & (datos_acido_urico["age"] >= edad_minima) & (datos_acido_urico["age"] < edad_maxima)]
    acido_urico_grupo_ckd = grupo[grupo["ckd_clinical"] == 1]["uric_acid"]
    acido_urico_grupo_sin_ckd = grupo[grupo["ckd_clinical"] == 0]["uric_acid"]
    stat, p = stats.kruskal(acido_urico_grupo_ckd, acido_urico_grupo_sin_ckd)
    print(f"{nombre_sexo} de {edad_minima} a {edad_maxima - 1} años: mediana con CKD {acido_urico_grupo_ckd.describe()['50%']:.2f} (n = {acido_urico_grupo_ckd.shape[0]}) - sin CKD {acido_urico_grupo_sin_ckd.describe()['50%']:.2f} (n = {acido_urico_grupo_sin_ckd.shape[0]}) - Kruskal-Wallis: Estadístico={stat:.3f}, p-valor={p:.3f}")

In [ ]:
# proporción de los casos de CKD con filtrado disminuido (eGFR < 60) en cada grupo de edad
for edad_minima, edad_maxima in [(18, 40), (40, 60), (60, 81)]:
  casos_grupo = datos_acido_urico[(datos_acido_urico["ckd_clinical"] == 1) & (datos_acido_urico["age"] >= edad_minima) & (datos_acido_urico["age"] < edad_maxima)]
  print(f"De {edad_minima} a {edad_maxima - 1} años: {casos_grupo.shape[0]} casos - con eGFR < 60: {(casos_grupo['egfr'] < 60).sum() / casos_grupo.shape[0] * 100:.1f} %")

**Resultados**

- **Muestra:** 5.514 adultos, 949 con CKD; 61 de los 5.575 clasificados no tienen BMI.
- **Correlaciones:** el ácido úrico se correlaciona más con el sexo (0,41) y el BMI (0,27) que con la CKD (0,16). Entre los predictores, la correlación más alta es la de la CKD con la edad (0,30); las demás son menores a 0,1 en valor absoluto, así que no hay multicolinealidad.
- **Ajuste del modelo:** el test F es significativo (F = 528,6, p < 0,001) y el R² es 0,277: el modelo explica el 27,7 % de la variabilidad del ácido úrico.
- **Coeficientes:**
  - CKD: 0,46 mg/dL más con CKD (IC 95 %: 0,38 a 0,55), p < 0,001;
  - sexo masculino: 1,21 mg/dL más que las mujeres (IC 95 %: 1,15 a 1,28), p < 0,001;
  - BMI: 0,411 por desvío de 7,24 kg/m² (IC 95 %: 0,379 a 0,442), es decir, 0,057 mg/dL por cada kg/m², p < 0,001;
  - edad: 0,029 por desvío de 17,97 años (IC 95 %: −0,004 a 0,062, incluye el 0), es decir, 0,002 mg/dL por año, p = 0,083, no significativo.
- **Diferencia por la CKD:** sin ajustar es de 0,60 mg/dL; ajustada por edad, sexo y BMI es de **0,46 mg/dL**, con un intervalo de confianza del 95 % de 0,38 a 0,55 (t = 10,32, p < 0,001).
- **Supuestos:**
  - normalidad: Omnibus y Jarque-Bera rechazan (p < 0,001). El QQ plot sigue la recta salvo en la cola derecha, que reúne a los adultos con el ácido úrico más alto (asimetría de 0,55 y curtosis de 4,6);
  - homocedasticidad: los residuos forman una banda alrededor de 0, sin forma de embudo, pero su desvío es mayor en el grupo con CKD (1,53 contra 1,12 mg/dL): es la misma diferencia de dispersión que detectó el test de Levene en la sección 3.2;
  - independencia: Durbin-Watson de 1,96;
  - linealidad: los residuos contra la edad se distribuyen en una banda alrededor de 0, sin patrones curvos; contra el BMI también, salvo en los valores más altos (por encima de 50 kg/m²), donde hay pocos adultos y los residuos tienden a ser negativos;
  - multicolinealidad: los VIF de los predictores van de 1,005 a 1,103, prácticamente 1 (el de la constante no se interpreta): cada predictor casi no se explica por los demás.
- **Control por sexo y edad (Kruskal-Wallis):**

| Grupo | Mediana con CKD (mg/dL) | Mediana sin CKD (mg/dL) | p-valor |
|---|---|---|---|
| Mujeres de 18 a 39 años | 4,30 (n = 55) | 4,20 (n = 808) | 0,630 |
| Mujeres de 40 a 59 años | 5,00 (n = 113) | 4,30 (n = 746) | < 0,001 |
| Mujeres de 60 años o más | 5,40 (n = 328) | 4,50 (n = 964) | < 0,001 |
| Hombres de 18 a 39 años | 5,85 (n = 32) | 5,80 (n = 654) | 0,664 |
| Hombres de 40 a 59 años | 5,40 (n = 67) | 5,70 (n = 617) | 0,277 |
| Hombres de 60 años o más | 6,00 (n = 354) | 5,50 (n = 776) | < 0,001 |

- **Filtrado de los casos según la edad:** tienen eGFR menor a 60 el 1,1 % de los casos de 18 a 39 años (87 casos), el 29,4 % de los de 40 a 59 (180) y el 59,7 % de los de 60 o más (682).

**Conclusión: se rechaza H₀. A igualdad de edad, sexo y BMI, los adultos con enfermedad renal crónica tienen en promedio 0,46 mg/dL más de ácido úrico (IC 95 %: 0,38 a 0,55), frente a 0,60 mg/dL sin ajustar. La diferencia se mantiene, por lo que la hipótesis se confirma. Los supuestos de normalidad y homocedasticidad se cumplen solo en parte, y el control no paramétrico muestra que la diferencia se concentra en los adultos de 60 años o más y en las mujeres de 40 a 59.**

**Discusión**

- **Qué explica la reducción.** La diferencia baja de 0,60 a 0,46 mg/dL: una parte de lo observado en la sección 3.2 se debía a las otras variables. El sexo y el BMI tienen coeficientes grandes y significativos, mientras que la edad, a igualdad de las demás, no se vincula significativamente con el ácido úrico. Aun así, la CKD conserva un vínculo significativo y la mayor parte de la diferencia.
- **Dónde aparece la diferencia.** En los adultos de 18 a 39 años no hay diferencia, y en ese grupo casi ningún caso tiene el filtrado disminuido (1,1 %): los casos jóvenes se definen sobre todo por la albuminuria (ACR ≥ 30). En los adultos de 60 años o más, donde la mayoría de los casos (59,7 %) tiene eGFR menor a 60, la diferencia es clara. El patrón es coherente con que el ácido úrico se elimine en parte por los riñones [5] y se acumule cuando disminuye el filtrado, más que cuando solo hay pérdida de albúmina. Es una interpretación: el modelo no separa los dos ejes de la enfermedad. Además, los grupos jóvenes tienen pocos casos (55 y 32), así que la falta de significancia no prueba que no haya diferencia.
- **Supuestos.** Con 5.514 observaciones, los tests de normalidad rechazan ante desvíos pequeños; el QQ plot muestra que el apartamiento se limita a la cola derecha. El control de Kruskal-Wallis no depende de la normalidad y coincide con el modelo en los grupos con más casos, así que la conclusión no descansa solo en ese supuesto. La mayor dispersión de los residuos en el grupo con CKD hace que el error estándar del coeficiente sea aproximado: el intervalo de confianza se interpreta como una referencia.
- **Magnitud.** 0,46 mg/dL es una diferencia moderada frente a la dispersión de los residuos (de 1,1 a 1,5 mg/dL) y menor que la diferencia entre sexos (1,21 mg/dL). El R² de 0,277 indica que la mayor parte de la variabilidad del ácido úrico depende de factores que el archivo no registra.
- **Asociación.** Con una sola medición por persona no se puede establecer si la enfermedad renal eleva el ácido úrico o al revés: el resultado es una asociación.

### 3.4 Hipótesis (a): «La mayoría de los casos de CKF identificados corresponde a blancos no-hispánicos»

#### 3.4.1 Definición

**Motivación.** En el análisis exploratorio, los blancos no hispanos son el 52,1 % de la muestra (sección 2.6.2) y el grupo con mayor edad mediana (sección 2.13.3). Como la enfermedad renal es más frecuente a mayor edad, es esperable que concentren buena parte de los casos.

**Enunciado.** La mayoría de los casos de enfermedad renal crónica identificados corresponde a blancos no hispanos (CKF se interpreta como CKD; criterios de la sección 3).

**Dos lecturas.** La hipótesis admite dos preguntas distintas, que se responden por separado:

1. **Conteo (lectura literal):** ¿más de la mitad de los casos son blancos no hispanos? Depende de cuántos casos hay en cada grupo y, por lo tanto, también de cuántas personas de cada grupo tiene la muestra.
2. **Riesgo:** ¿la proporción de personas con CKD es distinta según el grupo étnico? Compara la frecuencia de la enfermedad dentro de cada grupo, sin depender de su tamaño.

Un grupo puede concentrar la mayoría de los casos solo por ser el más numeroso, sin que la enfermedad sea más frecuente en él. Por eso las dos lecturas se analizan juntas.

**Utilidad.** Distingue el peso de un grupo en el total de casos (útil, por ejemplo, para dimensionar la atención que requiere) de la frecuencia de la enfermedad dentro del grupo (útil para identificar grupos más afectados).

#### 3.4.2 Estrategia

- **Población y variables:** `poblacion_ckd`; los casos son los adultos con `ckd_clinical` = 1. Variables: `ethnicity`, cualitativa nominal (6 grupos), y `ckd_clinical`, binaria.
- **Lectura 1 — Z para una proporción**, el mismo test de la sección 3.1, aplicado a los casos. Sea p la proporción de blancos no hispanos entre los adultos con CKD.
  - H₀: p = 0,5
  - H₁: p > 0,5 (unilateral: la hipótesis fija la dirección, porque "la mayoría" significa más de la mitad; la dirección surge del enunciado, antes de ver los datos)
  - Nivel de significancia: α = 0,05
  - Supuesto: `n · p₀` y `n · (1 − p₀)` son al menos 10, con n la cantidad de casos. El p-valor es la probabilidad, si H₀ fuera cierta, de obtener un z al menos tan grande como el observado (solo hacia la derecha).
- **Lectura 2 — Chi-cuadrado de independencia** entre el grupo étnico y la enfermedad renal.
  - H₀: el grupo étnico y la presencia de CKD son independientes (la proporción con CKD es la misma en todos los grupos)
  - H₁: el grupo étnico y la presencia de CKD no son independientes
  - Se arma la tabla de contingencia con los conteos de cada combinación de grupo étnico y CKD. Para cada celda, la frecuencia esperada si H₀ fuera cierta es `(total de la fila × total de la columna) / n`, y el estadístico suma, en todas las celdas, `(observada − esperada)² / esperada`. Los grados de libertad son `(filas − 1) × (columnas − 1) = (6 − 1) × (2 − 1) = 5`.
  - Es una prueba unilateral derecha: cuanto más se alejan los conteos observados de los esperados, mayor es el estadístico.
  - Condición: las frecuencias esperadas tienen que ser de al menos 5 en todas las celdas.
  - Rechazar H₀ indica que la proporción con CKD no es la misma en todos los grupos, pero no indica cuáles difieren: las diferencias se describen con las proporciones de cada grupo.
- **Interpretación:** la proporción con CKD de los blancos no hispanos y del resto se describe por grupo de edad, por el papel de la edad en la enfermedad (sección 3.1).

#### 3.4.3 Resultados y discusión

In [ ]:
# para cada grupo étnico: adultos, casos, porcentaje del total de casos y proporción con CKD dentro del grupo
casos_totales = (poblacion_ckd["ckd_clinical"] == 1).sum()
filas_etnia_ckd = []
for etnia in poblacion_ckd["ethnicity"].value_counts().index:
  grupo = poblacion_ckd[poblacion_ckd["ethnicity"] == etnia]
  casos = (grupo["ckd_clinical"] == 1).sum()
  filas_etnia_ckd.append({"grupo étnico": etnia,
                          "adultos": grupo.shape[0],
                          "% de los adultos": round(grupo.shape[0] / poblacion_ckd.shape[0] * 100, 1),
                          "casos": casos,
                          "% de los casos": round(casos / casos_totales * 100, 1),
                          "% con CKD dentro del grupo": round(casos / grupo.shape[0] * 100, 1),
                          "edad mediana": grupo["age"].describe()["50%"]})

tabla_etnia_ckd = pd.DataFrame(filas_etnia_ckd)
tabla_etnia_ckd

In [ ]:
# lectura 1: porcentaje de los casos que corresponde a cada grupo étnico
tabla_casos_ordenada = tabla_etnia_ckd.sort_values("% de los casos", ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(tabla_casos_ordenada["grupo étnico"], tabla_casos_ordenada["% de los casos"], color="steelblue")
plt.axhline(y=50, color="red", linestyle="--", label="mitad de los casos")
plt.xticks(rotation=45, ha="right")
plt.ylabel("% del total de casos de CKD")
plt.title("Distribución de los casos de CKD según el grupo étnico")
plt.legend()
plt.show()

# lectura 2: proporción con CKD dentro de cada grupo étnico
tabla_prevalencia_ordenada = tabla_etnia_ckd.sort_values("% con CKD dentro del grupo", ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(tabla_prevalencia_ordenada["grupo étnico"], tabla_prevalencia_ordenada["% con CKD dentro del grupo"], color="steelblue")
plt.axhline(y=casos_totales / poblacion_ckd.shape[0] * 100, color="red", linestyle="--", label="proporción en todos los adultos")
plt.xticks(rotation=45, ha="right")
plt.ylabel("% de adultos con CKD")
plt.title("Proporción con CKD dentro de cada grupo étnico")
plt.legend()
plt.show()

In [ ]:
# lectura 1: test Z para una proporción, unilateral (H1: más de la mitad de los casos son blancos no hispanos)
casos_blancos = (poblacion_ckd[poblacion_ckd["ckd_clinical"] == 1]["ethnicity"] == "Non-Hispanic White").sum()
p0_mayoria = 0.5
print(f"n · p0 = {casos_totales * p0_mayoria:.1f} - n · (1 − p0) = {casos_totales * (1 - p0_mayoria):.1f}")

p_blancos = casos_blancos / casos_totales
error_estandar_mayoria = np.sqrt(p0_mayoria * (1 - p0_mayoria) / casos_totales)
z_mayoria = (p_blancos - p0_mayoria) / error_estandar_mayoria
p_value_mayoria = 1 - stats.norm.cdf(z_mayoria)
print(f"Casos de blancos no hispanos: {casos_blancos} de {casos_totales} - proporción = {p_blancos:.4f}")
print(f"Test Z para una proporción (unilateral): Estadístico={z_mayoria:.3f}, p-valor={p_value_mayoria:.3f}")

# intervalo de confianza del 95 % para la proporción
error_estandar_blancos = np.sqrt(p_blancos * (1 - p_blancos) / casos_totales)
print(f"IC 95 %: [{(p_blancos - 1.96 * error_estandar_blancos) * 100:.1f} %; {(p_blancos + 1.96 * error_estandar_blancos) * 100:.1f} %]")

# interpretación de los resultados
alpha = 0.05  # nivel de significancia
if p_value_mayoria > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("No puede afirmarse que más de la mitad de los casos de CKD correspondan a blancos no hispanos.")
else:
  print("Se rechaza la hipótesis nula.")
  print("Más de la mitad de los casos de CKD corresponden a blancos no hispanos.")

In [ ]:
# lectura 2: tabla de contingencia con los conteos de cada grupo étnico sin CKD y con CKD
filas_contingencia = []
for etnia in poblacion_ckd["ethnicity"].value_counts().index:
  grupo = poblacion_ckd[poblacion_ckd["ethnicity"] == etnia]
  filas_contingencia.append({"grupo étnico": etnia,
                             "sin CKD": (grupo["ckd_clinical"] == 0).sum(),
                             "con CKD": (grupo["ckd_clinical"] == 1).sum()})

tabla_contingencia = pd.DataFrame(filas_contingencia)
tabla_contingencia

In [ ]:
# test de chi-cuadrado de independencia sobre los conteos de la tabla
chi2_statistic, p_value_chi2, grados_libertad, frecuencias_esperadas = stats.chi2_contingency(tabla_contingencia[["sin CKD", "con CKD"]])
print(f"Test de chi-cuadrado de independencia: Estadístico={chi2_statistic:.3f}, p-valor={p_value_chi2:.3f}, grados de libertad={grados_libertad}")
print(f"Menor frecuencia esperada: {frecuencias_esperadas.min():.1f}")

# interpretación de los resultados
if p_value_chi2 > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("No hay evidencia de asociación entre el grupo étnico y la presencia de CKD.")
else:
  print("Se rechaza la hipótesis nula.")
  print("El grupo étnico y la presencia de CKD no son independientes: la proporción con CKD no es la misma en todos los grupos.")

In [ ]:
# proporción con CKD de los blancos no hispanos y del resto de los adultos, por grupo de edad
for edad_minima, edad_maxima in [(18, 40), (40, 60), (60, 81)]:
  grupo_edad = poblacion_ckd[(poblacion_ckd["age"] >= edad_minima) & (poblacion_ckd["age"] < edad_maxima)]
  blancos = grupo_edad[grupo_edad["ethnicity"] == "Non-Hispanic White"]
  resto = grupo_edad[grupo_edad["ethnicity"] != "Non-Hispanic White"]
  porcentaje_blancos = (blancos["ckd_clinical"] == 1).sum() / blancos.shape[0] * 100
  porcentaje_resto = (resto["ckd_clinical"] == 1).sum() / resto.shape[0] * 100
  print(f"De {edad_minima} a {edad_maxima - 1} años: blancos no hispanos {porcentaje_blancos:.1f} % (n = {blancos.shape[0]}) - resto {porcentaje_resto:.1f} % (n = {resto.shape[0]})")

**Resultados**

- **Lectura 1 (conteo).** De los 972 casos, 572 son blancos no hispanos: **58,8 %**, con un intervalo de confianza del 95 % de 55,8 % a 61,9 %. Los supuestos se cumplen (`n · p₀` = `n · (1 − p₀)` = 486). El test da z = 5,52 y un p-valor menor a 0,001.
- **Lectura 2 (riesgo).** El chi-cuadrado da 35,58 con 5 grados de libertad y un p-valor menor a 0,001; la menor frecuencia esperada es 53,2, así que la condición se cumple. La proporción con CKD no es la misma en todos los grupos: va de 12,7 % en los mexicano-americanos a 24,6 % en los negros no hispanos. En los blancos no hispanos es 17,3 %, prácticamente igual a la de todos los adultos (17,4 %).
- **Por edad.** Los blancos no hispanos tienen una proporción con CKD menor que la del resto en los tres grupos de edad: 4,3 % contra 6,9 % de 18 a 39 años, 9,1 % contra 15,3 % de 40 a 59, y 27,4 % contra 30,9 % de 60 o más.

**Conclusión: la hipótesis se confirma en su lectura literal: la mayoría de los casos de enfermedad renal crónica (58,8 %) corresponde a blancos no hispanos (z = 5,52, p < 0,001). Esa mayoría no refleja una mayor frecuencia de la enfermedad en el grupo: su proporción con CKD (17,3 %) es igual a la del total, menor que la de los negros no hispanos (24,6 %) y menor que la del resto de los adultos en cada grupo de edad.**

**Discusión**

- **Composición de la muestra.** Los blancos no hispanos son el 59,2 % de los adultos analizados, y su participación en los casos (58,8 %) es casi la misma. La mayoría de los casos se explica por cuántos son en la muestra, no por una mayor frecuencia de la enfermedad.
- **Edad.** Es el grupo con mayor edad mediana (60 años), y la enfermedad renal aumenta con la edad (sección 3.1). Eso contribuye a su cantidad de casos; a igual edad, su proporción con CKD es menor que la del resto.
- **Alcance del chi-cuadrado.** El test indica que el grupo étnico y la enfermedad renal están asociados, pero no qué grupos difieren entre sí. Las diferencias se describen con las proporciones de cada grupo, sin afirmar que cada par de grupos difiera significativamente.
- **Limitaciones.**
  - La falta de eGFR varía según el grupo étnico (28,8 % en blancos no hispanos contra 37,2 % en negros no hispanos, sección 2.11.4): si los excluidos tuvieran más enfermedad, los casos de los grupos con más faltantes estarían más subestimados.
  - Sin pesos muestrales **[P25]**, las proporciones describen a la muestra y no a la población de EE.UU. El diseño muestral complejo hace, además, que las observaciones no sean completamente independientes (sección 3.1).

### 3.5 Hipótesis (b): «La presión arterial es un factor de riesgo del CKF»

#### 3.5.1 Definición

**Motivación.** La metadata señala a la hipertensión como un factor estrechamente relacionado con el deterioro renal y menciona que los riñones participan en el control de la presión arterial [5]. En el análisis exploratorio, los adultos con eGFR menor a 60 tienen una presión sistólica mediana mayor que el resto (126 contra 119 mmHg, sección 2.13.2).

**Enunciado.** La presión arterial es un factor de riesgo de la enfermedad renal crónica (CKF se interpreta como CKD; criterios de la sección 3).

**Qué se puede evaluar con estos datos.** Un factor de riesgo es una característica que precede a la enfermedad y aumenta la probabilidad de desarrollarla, y demostrarlo requiere seguir a las personas en el tiempo. Con una sola medición por persona (sección 1.2) se puede evaluar si la presión arterial está **asociada** con la enfermedad renal, que es una condición necesaria pero no suficiente para que sea un factor de riesgo. La hipótesis se evalúa, entonces, como una diferencia en la presión entre los adultos con y sin CKD.

**Variable de presión.** El análisis se centra en la presión sistólica, que en el análisis exploratorio mostró diferencias según la función renal (sección 2.13.2). Como indica la metadata, la diastólica se interpreta junto con la sistólica: se describe como complemento.

**Utilidad.** La presión arterial es una medición simple y modificable; establecer su asociación con la enfermedad renal en la muestra es el primer paso para considerarla en el análisis conjunto de factores (hipótesis c).

#### 3.5.2 Estrategia

- **Variables:** `bp_systolic`, cuantitativa continua (mmHg, registrada en valores enteros), y `ckd_clinical`, binaria, en los adultos de `poblacion_ckd` con presión registrada. Se comparan dos grupos independientes.
- **Hipótesis:** sean μ₁ la media de la presión sistólica de los adultos con CKD y μ₀ la de los adultos sin CKD.
  - H₀: μ₁ = μ₀
  - H₁: μ₁ ≠ μ₀ (bilateral)
  - Nivel de significancia: α = 0,05
- **Procedimiento:** la comparación de dos grupos descrita al inicio de la sección 3.
- **Complemento descriptivo:** la presión diastólica de cada grupo.
- **Edad:** no se ajusta en esta hipótesis; su posible efecto se discute con los resultados de la sección 2.13.2, y el análisis conjunto con otros factores corresponde a la hipótesis (c).

#### 3.5.3 Resultados y discusión

In [ ]:
# presión sistólica de cada grupo, sin los nulos
sistolica_ckd = poblacion_ckd[poblacion_ckd["ckd_clinical"] == 1]["bp_systolic"].dropna()
sistolica_sin_ckd = poblacion_ckd[poblacion_ckd["ckd_clinical"] == 0]["bp_systolic"].dropna()
print(f"Adultos clasificados sin presión registrada: {poblacion_ckd['bp_systolic'].isna().sum()}")
for nombre, grupo in [("Con CKD", sistolica_ckd), ("Sin CKD", sistolica_sin_ckd)]:
  estadisticas = grupo.describe()
  print(f"{nombre}: n = {estadisticas['count']:.0f} - media = {estadisticas['mean']:.1f} - mediana = {estadisticas['50%']:.0f} - desvío = {estadisticas['std']:.1f} mmHg")

In [ ]:
# boxplot de la presión sistólica según la enfermedad renal
sns.boxplot(x="ckd_clinical", y="bp_systolic", data=poblacion_ckd, notch=True)
plt.title("Presión sistólica según la enfermedad renal crónica")
plt.xlabel("Enfermedad renal crónica")
plt.xticks(ticks=[0, 1], labels=["Sin CKD", "Con CKD"])
plt.ylabel("Presión sistólica (mmHg)")
plt.show()

In [ ]:
# test de Shapiro-Wilk en cada grupo
stat, p = stats.shapiro(sistolica_ckd)
print(f"Test de Shapiro-Wilk para adultos con CKD: Estadístico={stat:.3f}, p-valor={p:.3f}")

stat, p = stats.shapiro(sistolica_sin_ckd)
print(f"Test de Shapiro-Wilk para adultos sin CKD: Estadístico={stat:.3f}, p-valor={p:.3f}")

In [ ]:
# QQ plot para adultos con CKD
stats.probplot(sistolica_ckd, dist="norm", plot=plt)
plt.title("QQ Plot de la presión sistólica en adultos con CKD")
plt.show()

# QQ plot para adultos sin CKD
stats.probplot(sistolica_sin_ckd, dist="norm", plot=plt)
plt.title("QQ Plot de la presión sistólica en adultos sin CKD")
plt.show()

In [ ]:
# test de Levene: H0 = las varianzas de los dos grupos son iguales
stat_levene, p_levene = stats.levene(sistolica_ckd, sistolica_sin_ckd)
print(f"Test de Levene para bp_systolic: Estadístico={stat_levene:.3f}, p-valor={p_levene:.3f}")

alpha = 0.05  # nivel de significancia
if p_levene > alpha:
  varianzas_iguales = True
  print("No hay evidencia para rechazar la igualdad de varianzas: se usa el test t para muestras independientes.")
else:
  varianzas_iguales = False
  print("Se rechaza la igualdad de varianzas: se usa el test t de Welch.")

In [ ]:
# test principal: test t para muestras independientes (con la corrección de Welch si las varianzas difieren)
t_statistic, p_value = stats.ttest_ind(sistolica_ckd, sistolica_sin_ckd, equal_var=varianzas_iguales)
print(f"Test t: Estadístico={t_statistic:.3f}, p-valor={p_value:.3f}")
print(f"Diferencia de medias (con CKD - sin CKD): {sistolica_ckd.describe()['mean'] - sistolica_sin_ckd.describe()['mean']:.2f} mmHg")

# interpretación de los resultados
if p_value > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
  print("No hay una diferencia significativa en la presión sistólica entre adultos con y sin CKD.")
else:
  print("Se rechaza la hipótesis nula.")
  print("Existe una diferencia significativa en la presión sistólica entre adultos con y sin CKD.")

In [ ]:
# control de robustez: test no paramétrico (Mann-Whitney U si las varianzas son iguales, Kruskal-Wallis si no)
if varianzas_iguales:
  stat, p = stats.mannwhitneyu(sistolica_ckd, sistolica_sin_ckd)
  print(f"Test de Mann-Whitney U para bp_systolic: Estadístico={stat:.3f}, p-valor={p:.3f}")
else:
  stat, p = stats.kruskal(sistolica_ckd, sistolica_sin_ckd)
  print(f"Test de Kruskal-Wallis para bp_systolic: Estadístico={stat:.3f}, p-valor={p:.3f}")

if p > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula.")
else:
  print("Se rechaza la hipótesis nula: coincide con el test principal.")

In [ ]:
# complemento descriptivo: presión diastólica y edad de cada grupo (adultos con presión registrada)
con_presion = poblacion_ckd[poblacion_ckd["bp_systolic"].notna()]
for nombre, valor in [("Con CKD", 1), ("Sin CKD", 0)]:
  grupo = con_presion[con_presion["ckd_clinical"] == valor]
  diastolica = grupo["bp_diastolic"].describe()
  print(f"{nombre}: diastólica media = {diastolica['mean']:.1f} - mediana = {diastolica['50%']:.0f} mmHg - edad mediana = {grupo['age'].describe()['50%']:.0f} años")

**Resultados**

- **Descripción:** de los 5.575 adultos clasificados, 147 no tienen la presión registrada. Los 936 adultos con CKD tienen una presión sistólica media de 130,6 mmHg y una mediana de 128; los 4.492 sin CKD, una media de 120,6 y una mediana de 118. En el boxplot, la caja del grupo con CKD está corrida hacia arriba.
- **Normalidad:** Shapiro-Wilk rechaza en los dos grupos (p < 0,001; estadísticos de 0,978 y 0,965); los QQ plots muestran que el apartamiento se concentra en la cola derecha (valores altos). Con grupos de 936 y 4.492 personas, el test t es robusto a esta falta de normalidad.
- **Varianzas:** Levene rechaza la igualdad (estadístico 151,80, p < 0,001): el desvío es de 22,9 mmHg en el grupo con CKD y de 16,6 en el grupo sin CKD. Por eso se usa el test t de Welch, con Kruskal-Wallis como control.
- **Test t de Welch:** t = 12,62, p < 0,001. La diferencia de medias es de 9,95 mmHg, a favor del grupo con CKD.
- **Kruskal-Wallis:** estadístico 171,97, p < 0,001: coincide con el test principal.
- **Presión diastólica (complemento):** no muestra diferencias entre los grupos (mediana de 74 mmHg en ambos; medias de 75,4 y 74,9).

**Conclusión: se rechaza H₀. La presión sistólica es significativamente mayor en los adultos con enfermedad renal crónica (media de 130,6 contra 120,6 mmHg, unos 10 mmHg más). La hipótesis se confirma como asociación: con estos datos no se puede establecer que la presión arterial sea un factor de riesgo, es decir, que preceda a la enfermedad y aumente la probabilidad de desarrollarla.**

**Discusión**

- **Magnitud.** La diferencia es de unos 10 mmHg tanto en la media como en la mediana, y el grupo con CKD tiene además más dispersión (desvío de 22,9 contra 16,6 mmHg), es decir, más valores altos.
- **Edad.** Los adultos con CKD son mayores (mediana de 68 contra 52 años), y la presión sistólica aumenta con la edad: en el análisis exploratorio, su mediana pasa de 112 mmHg entre los 18 y 39 años a 127 mmHg a partir de los 60 (sección 2.13.2). Parte de la diferencia puede deberse a la edad; esta hipótesis no la ajusta, y el análisis conjunto corresponde a la hipótesis (c) (sección 3.6).
- **Dirección de la relación.** Los riñones participan en el control de la presión arterial [5], así que la enfermedad renal también puede elevarla. Con una sola medición por persona no se puede distinguir si la presión alta precede a la enfermedad o es una consecuencia.
- **Medicación.** El archivo no tiene datos de tratamiento antihipertensivo: una persona hipertensa tratada puede tener la presión normal, lo que tiende a reducir la diferencia observada.
- **Medición.** El archivo trae un solo valor de las tres lecturas que toma NHANES, sin indicar cuál **[P20]**.
- **Presión diastólica.** No difiere entre los grupos. Es coherente con la mayor edad de los adultos con CKD: con la edad, la rigidez de las arterias tiende a elevar la presión sistólica más que la diastólica.

### 3.6 Hipótesis (c): «Existe un vínculo directo entre la edad, sexo, BMI, presión arterial, diabetes, tabaquismo y nivel socioeconómico con la probabilidad de presentar CKD»

#### 3.6.1 Definición

**Motivación.** La metadata señala que *"la diabetes y la hipertensión son factores estrechamente relacionados con el deterioro renal"* y que *"también importan la edad, el tabaquismo, el exceso de peso, las condiciones socioeconómicas"* [5]. Las secciones anteriores muestran que esas variables no son independientes entre sí: por ejemplo, la presión sistólica es más alta en los adultos con CKD, pero ese grupo también es mayor (sección 3.5). Para saber qué variables se vinculan con la enfermedad por sí mismas, hay que analizarlas en conjunto.

**Enunciado.** Existe un vínculo directo entre la edad, el sexo, el BMI, la presión arterial, la diabetes, el tabaquismo y el nivel socioeconómico con la probabilidad de presentar CKD.

**Interpretación de los términos.**
- **Vínculo directo:** asociación de cada variable con la probabilidad de CKD manteniendo constantes las otras seis. Una variable puede asociarse con la CKD solo porque acompaña a otra (por ejemplo, a la edad); en ese caso no tiene un vínculo directo.
- **Nivel socioeconómico:** el índice de pobreza (`poverty_income_ratio`), el ingreso del hogar dividido por el umbral de pobreza.
- **Presión arterial:** la presión sistólica, con el mismo criterio que la sección 3.5.
- **Diabetes:** `diabetes_status` (sin diabetes, borderline o diagnosticada). **Tabaquismo:** `smoking_status` (nunca fumó, ex fumador o fumador actual).
- Como en la hipótesis (b), con una sola medición por persona se evalúa asociación y no causalidad.

**Utilidad.** Identifica qué factores conservan un vínculo con la enfermedad renal cuando se los considera en conjunto, y completa el análisis de la presión arterial de la sección 3.5, que no tenía en cuenta la edad.

#### 3.6.2 Estrategia

- **Población:** los adultos de `poblacion_ckd` con datos en las siete variables (casos completos).
- **Técnica: regresión logística múltiple.** La variable respuesta, `ckd_clinical`, es binaria, y la hipótesis se refiere a la probabilidad de presentar CKD. La regresión logística modela esa probabilidad como una función de los predictores que siempre toma valores entre 0 y 1; una regresión lineal sobre una variable 0/1 podría dar valores fuera de ese rango. Los coeficientes se estiman por máxima verosimilitud.
- **Lectura de los coeficientes:** manteniendo constantes los demás predictores, un coeficiente positivo indica que la probabilidad de CKD aumenta cuando el predictor aumenta, y uno negativo, que disminuye. Los predictores cuantitativos están estandarizados, así que sus coeficientes expresan el efecto de un desvío y se pueden comparar entre sí; los de las columnas 0/1 expresan el efecto de pertenecer a esa categoría en lugar de la de referencia.
- **Variables y codificación:**

| Variable del enunciado | Columna | Tipo | Codificación en el modelo |
|---|---|---|---|
| Probabilidad de CKD (respuesta) | `ckd_clinical` | binaria | 0/1 (1 = con CKD) |
| Edad | `age` | cuantitativa continua (años) | estandarizada |
| Sexo | `gender` | binaria | `hombre`: 1 hombre, 0 mujer (referencia) |
| BMI | `bmi` | cuantitativa continua (kg/m²) | estandarizada |
| Presión arterial | `bp_systolic` | cuantitativa continua (mmHg) | estandarizada |
| Diabetes | `diabetes_status` | cualitativa ordinal, 3 categorías | `diabetes_borderline` y `diabetes_diagnosticada` (0/1); referencia: sin diabetes |
| Tabaquismo | `smoking_status` | cualitativa, 3 categorías | `ex_fumador` y `fumador_actual` (0/1); referencia: nunca fumó |
| Nivel socioeconómico | `poverty_income_ratio` | cuantitativa continua (índice) | estandarizada |

- **Columnas 0/1:** el coeficiente de cada una compara su categoría con la de referencia. Ingresar la diabetes o el tabaquismo como un único número (0, 1, 2) supondría que el paso de una categoría a la siguiente tiene siempre el mismo efecto. La categoría de referencia no lleva columna: las tres columnas sumarían siempre 1, igual que la constante del modelo, y los coeficientes no podrían estimarse (multicolinealidad perfecta).
- **Hipótesis para cada predictor:**
  - H₀: βⱼ = 0 (a igualdad de las demás variables, el predictor no se vincula con la probabilidad de CKD)
  - H₁: βⱼ ≠ 0 (bilateral)
  - Nivel de significancia: α = 0,05
  - Se usa el test z de cada coeficiente. La significancia global se evalúa con el test de razón de verosimilitud, que compara el modelo con uno sin predictores. Como medida descriptiva del ajuste se informa el pseudo R² de McFadden, que no se interpreta como el R² de la regresión lineal.
- **Criterio de decisión:** una variable tiene vínculo directo si su coeficiente es significativo; en la diabetes y el tabaquismo, si lo es al menos una de sus categorías. La hipótesis se confirma por completo si las siete variables tienen vínculo directo, y en parte si solo algunas lo tienen.
- **Supuestos:**
  - observaciones independientes (una fila por persona; ver el diseño muestral en la sección 3.1);
  - ausencia de multicolinealidad fuerte: correlaciones entre los predictores y VIF;
  - cantidad suficiente de casos en relación con el número de predictores;
  - la regresión logística no requiere que los residuos sigan una distribución normal ni que las varianzas sean iguales.

#### 3.6.3 Resultados y discusión

In [ ]:
# variables del enunciado; el nivel socioeconómico se mide con el índice de pobreza
variables_hipotesis_c = ["age", "gender", "bmi", "bp_systolic", "diabetes_status", "smoking_status", "poverty_income_ratio"]

# adultos clasificados sin dato en cada variable
for variable in variables_hipotesis_c:
  print(f"{variable}: {poblacion_ckd[variable].isna().sum()} adultos sin dato")

# casos completos de las siete variables
datos_hipotesis_c = poblacion_ckd.dropna(subset=variables_hipotesis_c).copy()
print(f"Adultos con las siete variables: {datos_hipotesis_c.shape[0]} de {poblacion_ckd.shape[0]} (con CKD: {(datos_hipotesis_c['ckd_clinical'] == 1).sum()})")

In [ ]:
# sexo como variable binaria: 1 si es hombre, 0 si es mujer
datos_hipotesis_c["hombre"] = datos_hipotesis_c["gender"].apply(lambda x: 1 if x == "Male" else 0)
# diabetes: una columna 0/1 por categoría; "sin diabetes" (0) queda como referencia
datos_hipotesis_c["diabetes_borderline"] = datos_hipotesis_c["diabetes_status"].apply(lambda x: 1 if x == 1 else 0)
datos_hipotesis_c["diabetes_diagnosticada"] = datos_hipotesis_c["diabetes_status"].apply(lambda x: 1 if x == 2 else 0)
# tabaquismo: una columna 0/1 por categoría; "nunca fumó" (0) queda como referencia
datos_hipotesis_c["ex_fumador"] = datos_hipotesis_c["smoking_status"].apply(lambda x: 1 if x == 1 else 0)
datos_hipotesis_c["fumador_actual"] = datos_hipotesis_c["smoking_status"].apply(lambda x: 1 if x == 2 else 0)

# verificamos las columnas nuevas contra las variables originales
print(f"hombre = 1: {datos_hipotesis_c['hombre'].sum()} (Male en gender: {(datos_hipotesis_c['gender'] == 'Male').sum()})")
print(f"diabetes_borderline = 1: {datos_hipotesis_c['diabetes_borderline'].sum()} - diabetes_diagnosticada = 1: {datos_hipotesis_c['diabetes_diagnosticada'].sum()}")
print(f"ex_fumador = 1: {datos_hipotesis_c['ex_fumador'].sum()} - fumador_actual = 1: {datos_hipotesis_c['fumador_actual'].sum()}")
print(datos_hipotesis_c["diabetes_status"].value_counts().sort_index())
print(datos_hipotesis_c["smoking_status"].value_counts().sort_index())

In [ ]:
# mediana de cada variable cuantitativa según la CKD
for variable in ["age", "bmi", "bp_systolic", "poverty_income_ratio"]:
  mediana_ckd = datos_hipotesis_c[datos_hipotesis_c["ckd_clinical"] == 1][variable].describe()["50%"]
  mediana_sin_ckd = datos_hipotesis_c[datos_hipotesis_c["ckd_clinical"] == 0][variable].describe()["50%"]
  print(f"{variable}: mediana con CKD {mediana_ckd:.1f} - sin CKD {mediana_sin_ckd:.1f}")

# proporción con CKD en cada categoría de las variables cualitativas
for variable, categorias in [("gender", {"Female": "mujeres", "Male": "hombres"}),
                             ("diabetes_status", {0: "sin diabetes", 1: "diabetes borderline", 2: "diabetes diagnosticada"}),
                             ("smoking_status", {0: "nunca fumó", 1: "ex fumador", 2: "fumador actual"})]:
  for valor, nombre in categorias.items():
    grupo = datos_hipotesis_c[datos_hipotesis_c[variable] == valor]
    print(f"{nombre}: {grupo.shape[0]} adultos - con CKD {(grupo['ckd_clinical'] == 1).sum() / grupo.shape[0] * 100:.1f} %")

In [ ]:
# correlación entre los predictores, para detectar variables muy relacionadas entre sí
predictores_hipotesis_c = ["age", "hombre", "bmi", "bp_systolic", "diabetes_borderline", "diabetes_diagnosticada",
                           "ex_fumador", "fumador_actual", "poverty_income_ratio"]
matriz_correlacion_hipotesis_c = datos_hipotesis_c[predictores_hipotesis_c].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_correlacion_hipotesis_c, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre los predictores de la hipótesis (c)")
plt.show()

In [ ]:
# variable respuesta y predictores
y_hipotesis_c = datos_hipotesis_c["ckd_clinical"]
X_hipotesis_c = datos_hipotesis_c[predictores_hipotesis_c]

# estandarizamos los predictores cuantitativos (media 0 y desvío 1); las variables 0/1 quedan como están
columnas_cuantitativas_hipotesis_c = ["age", "bmi", "bp_systolic", "poverty_income_ratio"]
scaler_hipotesis_c = StandardScaler()
X_hipotesis_c_scaled_df = X_hipotesis_c.copy()
X_hipotesis_c_scaled_df[columnas_cuantitativas_hipotesis_c] = scaler_hipotesis_c.fit_transform(X_hipotesis_c[columnas_cuantitativas_hipotesis_c])

# verificamos la estandarización: medias cercanas a 0 y desvíos cercanos a 1
print(X_hipotesis_c_scaled_df[columnas_cuantitativas_hipotesis_c].describe().loc[["mean", "std"]].round(3))

In [ ]:
# agregamos una constante como característica y ajustamos la regresión logística (estimación por máxima verosimilitud)
X_hipotesis_c_scaled_df_ = sm.add_constant(X_hipotesis_c_scaled_df)
modelo_hipotesis_c = sm.Logit(y_hipotesis_c, X_hipotesis_c_scaled_df_).fit()
print(modelo_hipotesis_c.summary())

In [ ]:
# supuesto de ausencia de multicolinealidad: factor de inflación de la varianza (VIF) de cada columna del modelo, con la constante incluida
valores_vif_hipotesis_c = []
for i in range(X_hipotesis_c_scaled_df_.shape[1]):
  valores_vif_hipotesis_c.append(variance_inflation_factor(X_hipotesis_c_scaled_df_.values, i))
tabla_vif_hipotesis_c = pd.DataFrame({"variable": X_hipotesis_c_scaled_df_.columns, "VIF": valores_vif_hipotesis_c})
tabla_vif_hipotesis_c.round(3)

In [ ]:
# desvío de cada predictor estandarizado, para volver a las unidades originales
desvios_hipotesis_c = pd.Series(np.sqrt(scaler_hipotesis_c.var_), index=columnas_cuantitativas_hipotesis_c)
print(desvios_hipotesis_c.round(2))

# coeficientes con su intervalo de confianza del 95 %; en unidades originales, los de los predictores estandarizados
# se dividen por su desvío y los de las columnas 0/1 ya comparan la categoría con la de referencia
divisores_hipotesis_c = pd.Series(1.0, index=X_hipotesis_c.columns)
divisores_hipotesis_c[columnas_cuantitativas_hipotesis_c] = desvios_hipotesis_c
coeficientes_hipotesis_c = modelo_hipotesis_c.params[X_hipotesis_c.columns]
intervalos_hipotesis_c = modelo_hipotesis_c.conf_int().loc[X_hipotesis_c.columns]
tabla_coeficientes_hipotesis_c = pd.DataFrame({"coeficiente del modelo": coeficientes_hipotesis_c,
                                               "IC 95 % inferior": intervalos_hipotesis_c[0],
                                               "IC 95 % superior": intervalos_hipotesis_c[1],
                                               "coeficiente en unidades originales": coeficientes_hipotesis_c / divisores_hipotesis_c,
                                               "p-valor": modelo_hipotesis_c.pvalues[X_hipotesis_c.columns]})

# test z de cada coeficiente: H0 = a igualdad de las demás variables, el predictor no se vincula con la probabilidad de CKD
alpha = 0.05  # nivel de significancia
tabla_coeficientes_hipotesis_c["vínculo significativo"] = tabla_coeficientes_hipotesis_c["p-valor"].apply(lambda x: "sí" if x < alpha else "no")
tabla_coeficientes_hipotesis_c.round(4)

In [ ]:
# significancia global del modelo: test de razón de verosimilitud contra un modelo sin predictores
print(f"Test de razón de verosimilitud: Estadístico={modelo_hipotesis_c.llr:.3f}, p-valor={modelo_hipotesis_c.llr_pvalue:.3f}")
print(f"Pseudo R² de McFadden: {modelo_hipotesis_c.prsquared:.3f}")

# interpretación de los resultados
if modelo_hipotesis_c.llr_pvalue > alpha:
  print("No hay suficiente evidencia para rechazar la hipótesis nula: el modelo en conjunto no se vincula con la probabilidad de CKD.")
else:
  print("Se rechaza la hipótesis nula: el modelo en conjunto se vincula significativamente con la probabilidad de CKD.")

In [ ]:
# coeficientes del modelo: el signo indica si la variable aumenta o disminuye la probabilidad de CKD
colores = ["steelblue" if p < alpha else "lightgray" for p in tabla_coeficientes_hipotesis_c["p-valor"]]
plt.figure(figsize=(10, 5))
plt.bar(tabla_coeficientes_hipotesis_c.index, tabla_coeficientes_hipotesis_c["coeficiente del modelo"], color=colores)
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Coeficiente del modelo")
plt.title("Coeficientes de la regresión logística (azul: p < 0,05; gris: no significativo)")
plt.show()

**Resultados**

- **Muestra:** 4.698 de los 5.575 adultos clasificados tienen las siete variables, 778 con CKD (16,6 %). Se excluyen 877 adultos (15,7 %), sobre todo por el índice de pobreza (713 sin dato); faltan además 147 valores de presión, 61 de BMI, 9 de tabaquismo y 1 de diabetes. Las columnas 0/1 coinciden con los conteos de las variables originales.
- **Descripción:**
  - medianas con y sin CKD: edad de 68 contra 53 años, BMI de 29,7 contra 28,0, presión sistólica de 128 contra 119 mmHg e índice de pobreza de 2,4 contra 3,0;
  - proporción con CKD: 15,9 % en las mujeres y 17,3 % en los hombres; 12,6 % sin diabetes, 21,6 % con diabetes borderline y 40,4 % con diabetes diagnosticada; 13,7 % en quienes nunca fumaron, 22,3 % en los ex fumadores y 18,0 % en los fumadores actuales.
- **Supuestos:** la correlación más alta entre los predictores es la de la edad con la presión sistólica (0,38), y ninguna supera 0,4 en valor absoluto. Los VIF de los predictores van de 1,024 a 1,303 (el más alto, el de la edad): no hay multicolinealidad fuerte. Hay 778 casos para 9 predictores, unos 86 casos por predictor.
- **Ajuste global:** el test de razón de verosimilitud es significativo (estadístico de 675,7, p < 0,001); el pseudo R² es 0,160.
- **Coeficientes:**

| Variable | Coeficiente del modelo | IC 95 % | Qué compara | p-valor | Vínculo directo |
|---|---|---|---|---|---|
| Edad | 0,809 | 0,694 a 0,924 | un desvío (17,8 años); 0,045 por año | < 0,001 | Sí |
| Presión sistólica | 0,276 | 0,195 a 0,357 | un desvío (18,1 mmHg); 0,015 por mmHg | < 0,001 | Sí |
| Índice de pobreza | −0,280 | −0,370 a −0,191 | un desvío (1,65); −0,170 por unidad | < 0,001 | Sí |
| BMI | 0,168 | 0,082 a 0,254 | un desvío (7,1 kg/m²); 0,024 por kg/m² | < 0,001 | Sí |
| Diabetes diagnosticada | 1,030 | 0,827 a 1,234 | frente a sin diabetes | < 0,001 | Sí |
| Diabetes borderline | 0,139 | −0,270 a 0,549 | frente a sin diabetes | 0,505 | No |
| Ex fumador | 0,160 | −0,031 a 0,350 | frente a nunca fumó | 0,100 | No |
| Fumador actual | 0,115 | −0,135 a 0,364 | frente a nunca fumó | 0,369 | No |
| Hombre | 0,050 | −0,120 a 0,221 | frente a mujer | 0,565 | No |

Los intervalos de los cinco coeficientes significativos no incluyen el 0; los de los cuatro no significativos sí lo incluyen.

**Conclusión: la hipótesis se confirma en parte. A igualdad de las demás variables, la edad, el BMI, la presión arterial sistólica, la diabetes diagnosticada y el nivel socioeconómico tienen un vínculo directo y significativo con la probabilidad de presentar enfermedad renal crónica: la probabilidad aumenta con la edad, el BMI y la presión, es mayor con diabetes diagnosticada y disminuye cuando aumenta el ingreso del hogar. El sexo y el tabaquismo no tienen un vínculo directo significativo.**

**Discusión**

- **Fuerza de los vínculos.** Entre los predictores cuantitativos, un desvío de edad (0,81) pesa casi el triple que un desvío de presión sistólica (0,28) o de índice de pobreza (−0,28), y unas cinco veces más que un desvío de BMI (0,17). La diabetes diagnosticada tiene el coeficiente más alto del modelo (1,03), aunque esa comparación es aproximada: contrasta el paso de una categoría a otra con el aumento de un desvío.
- **Edad.** Parte de su vínculo se explica por la propia definición: el eGFR, uno de los dos criterios de la CKD, se calcula con una ecuación que incluye la edad [4], así que, con la misma creatinina, un adulto mayor obtiene un eGFR más bajo. Además, la edad está censurada en 80 años **[P8]**.
- **Presión arterial.** Conserva un vínculo significativo al considerar la edad y las demás variables: la diferencia de la sección 3.5 no se explica solo por la edad. Sigue siendo una asociación; valen las salvedades de esa sección sobre la dirección de la relación y la falta de datos de medicación.
- **Diabetes.** La diabetes diagnosticada tiene un vínculo claro. La diabetes borderline no: su mayor proporción de CKD (21,6 % contra 12,6 %) se explica por las otras variables, y con 162 adultos su estimación es poco precisa.
- **Tabaquismo.** Sin ajustar, los ex fumadores tienen más CKD que quienes nunca fumaron (22,3 % contra 13,7 %), pero el vínculo no es significativo al considerar las demás variables. Los ex fumadores son, en promedio, mayores (correlación de 0,23 con la edad), así que la diferencia sin ajustar refleja sobre todo otras características del grupo: es un ejemplo de asociación sin vínculo directo. El tabaquismo, además, es autodeclarado.
- **Sexo.** La proporción con CKD es parecida en hombres y mujeres (17,3 % contra 15,9 %), y el sexo no tiene un vínculo directo.
- **Nivel socioeconómico.** A igualdad de las demás variables, un menor ingreso del hogar se vincula con una mayor probabilidad de CKD. El índice falta en 713 adultos y su falta varía según el grupo étnico **[P11]**, y está censurado en 5 **[P10]**: si los adultos excluidos difirieran en su salud renal, la estimación podría estar sesgada.
- **Ajuste.** El pseudo R² de 0,16 indica que las siete variables explican una parte limitada de la presencia de CKD. El modelo se usa para evaluar vínculos, no para predecir casos individuales.
- **Limitaciones.**
  - Se analizan los casos completos: los 877 adultos excluidos pueden diferir de los incluidos.
  - El modelo supone que el efecto de cada predictor cuantitativo es el mismo en todo su rango (por ejemplo, entre los 30 y los 40 años que entre los 70 y los 80), y eso no se verificó.
  - Con una sola medición por persona y sin pesos muestrales **[P25]**, los resultados son asociaciones que describen a la muestra.

---
## 4. Conclusiones

La tabla resume las seis hipótesis. La técnica, los supuestos y la discusión de cada una están en su sección.

| Sección | Hipótesis | Técnica | Resultado principal | ¿Se confirma? |
|---|---|---|---|---|
| 3.1 | La proporción de adultos con CKD difiere del 14 % estimado por el CDC | Test Z para una proporción (bilateral) e intervalo de confianza del 95 % | 17,4 % (IC 95 %: 16,4 % a 18,4 %); z = 7,39, p < 0,001 | Sí: la proporción de la muestra es mayor |
| 3.2 | El ácido úrico difiere entre los adultos con y sin CKD | Test t de Welch, con Kruskal-Wallis como control | Media de 5,65 contra 5,05 mg/dL; t = 10,81, p < 0,001 | Sí |
| 3.3 | La diferencia de ácido úrico se mantiene a igualdad de edad, sexo y BMI | Regresión lineal múltiple, con Kruskal-Wallis por sexo y edad como control | 0,46 mg/dL (IC 95 %: 0,38 a 0,55) frente a 0,60 sin ajustar; t = 10,32, p < 0,001 | Sí; supuestos de normalidad y homocedasticidad cumplidos en parte |
| 3.4 (a) | La mayoría de los casos de CKD corresponde a blancos no hispanos | Test Z para una proporción (unilateral) y chi-cuadrado de independencia | 58,8 % de los casos (z = 5,52, p < 0,001); proporción con CKD en el grupo de 17,3 %, prácticamente igual a la del total (17,4 %) | Sí en su lectura literal; no refleja un mayor riesgo |
| 3.5 (b) | La presión arterial es un factor de riesgo de la CKD | Test t de Welch, con Kruskal-Wallis como control | Presión sistólica media de 130,6 contra 120,6 mmHg; t = 12,62, p < 0,001 | Como asociación; no puede establecerse como factor de riesgo |
| 3.6 (c) | Vínculo directo de la edad, el sexo, el BMI, la presión arterial, la diabetes, el tabaquismo y el nivel socioeconómico con la probabilidad de CKD | Regresión logística múltiple | Vínculo directo de la edad, el BMI, la presión sistólica, la diabetes diagnosticada y el índice de pobreza; sin vínculo del sexo ni del tabaquismo | En parte: cinco de las siete variables |

**Limitaciones generales**

- **Muestra sin pesos muestrales [P25].** Los resultados describen a la muestra y no a la población de EE.UU.; además, el ciclo sobremuestrea por grupo de edad (sección 3.1).
- **Una sola medición por persona.** No se puede confirmar la persistencia de tres meses que exige la definición de CKD ni el orden temporal entre las variables: todos los resultados son asociaciones.
- **Adultos sin clasificar.** Quedan fuera de los análisis renales 2.495 adultos sin eGFR y 83 sin ACR (sección 2.15.5), y la falta de eGFR varía según el grupo étnico.
- **Faltantes que no son al azar.** El índice de pobreza falta en 713 de los adultos clasificados y su falta varía según el grupo étnico **[P11]**; la hipótesis (c) usa solo los casos completos.
- **Mediciones.** La presión es una sola lectura sin indicar cuál **[P20]** y no hay datos de medicación; la edad y el índice de pobreza están censurados en 80 y 5 **[P8] [P10]**; el eGFR incluye la edad y el sexo en su ecuación [4].
- **Diseño muestral complejo.** Las observaciones no son completamente independientes (sección 3.1).
- **Tamaño de la muestra.** Con miles de observaciones, diferencias pequeñas resultan significativas: por eso cada hipótesis informa la magnitud además del p-valor.

---
## Referencias

1. National Center for Health Statistics. *NHANES August 2021–August 2023*. https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023
2. *CKD-NHANES 2021-2023: Staged Kidney Disease* [dataset]. Kaggle. https://www.kaggle.com/datasets/alitaqishah/ckd-nhanes-2021-2023-staged-kidney-disease
3. National Center for Health Statistics. *Demographic Variables and Sample Weights (DEMO_L)*. https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm
4. Inker LA, Eneanya ND, Coresh J, et al. *New Creatinine- and Cystatin C–Based Equations to Estimate GFR without Race.* N Engl J Med 2021;385:1737-1749. https://pubmed.ncbi.nlm.nih.gov/34554658
5. `metadata.docx`: diccionario de variables que acompaña al conjunto de datos `CKD_NHANES_2021_2023.csv`.
6. National Center for Health Statistics. *NHANES August 2021–August 2023 Response Rates*. https://wwwn.cdc.gov/nchs/data/ResponseRates/NHANES-August-2021-August-2023-Response-Rates.pdf
7. KDIGO. *CKD Evaluation and Management Guideline*. https://kdigo.org/guidelines/ckd-evaluation-and-management/
8. National Center for Health Statistics. *Blood Pressure – Oscillometric Measurement (BPXO_L)*. https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.htm
9. National Center for Health Statistics. *Albumin & Creatinine – Urine (ALB_CR_L)*. https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/ALB_CR_L.htm
10. Centers for Disease Control and Prevention. *Chronic Kidney Disease in the United States* (2026). https://www.cdc.gov/kidney-disease/media/pdfs/CKD-Factsheet-H.pdf